In [1]:
import os
import warnings
import random
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
import optuna
import joblib
from sklearn.inspection import permutation_importance
from catboost import CatBoostRegressor
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import json
import joblib

/Users/dhanujiamanda/Documents/Projects/Agentic AI /Pipeline/Agentic-AI-for-Pharma-Stockout-Problem/ENV/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ALLOW_FUTURE_VALIDATION = True  

warnings.filterwarnings("ignore")

# PreProcess

### DATA LOADING

In [3]:
Data = pd.read_excel("/Users/dhanujiamanda/Documents/Projects/Agentic AI /Pipeline/Agentic-AI-for-Pharma-Stockout-Problem/data/Company Data.xlsx")
Data.to_csv("/Users/dhanujiamanda/Documents/Projects/Agentic AI /Pipeline/Agentic-AI-for-Pharma-Stockout-Problem/data/Company Data.csv", index=False)

In [4]:
# To delete incomplete month
Data = Data[~((Data["Year"] == 2026) & (Data["Month_Number"] == 2))].copy()

Data = Data.sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)


In [5]:
import pandas as pd

# Load focus item codes
focus_file = "/Users/dhanujiamanda/Documents/Projects/Agentic AI /Pipeline/Agentic-AI-for-Pharma-Stockout-Problem/data/FocusItemCodes.xlsx"   

focus_df = pd.read_excel(focus_file)

# clean and convert code column
focus_df["Code"] = pd.to_numeric(focus_df["Code"], errors="coerce")
focus_df = focus_df.dropna(subset=["Code"])
focus_df["Code"] = focus_df["Code"].astype(int)

PHARMA_SKUS = focus_df["Code"].unique().tolist()

print("Focus SKU count from file:", len(PHARMA_SKUS))
print("First 10 SKUs:", PHARMA_SKUS[:10])


# Filter main dataset
Data["ItemCode"] = pd.to_numeric(Data["ItemCode"], errors="coerce")
Data = Data[Data["ItemCode"].isin(PHARMA_SKUS)].copy()

print("Filtered pharma rows:", len(Data))
print("Filtered pharma SKUs:", Data["ItemCode"].nunique())


# Missing SKU check
input_skus = set(PHARMA_SKUS)
found_skus = set(Data["ItemCode"].dropna().astype(int).unique())
missing_skus = sorted(list(input_skus - found_skus))

print("Requested pharma SKU count:", len(input_skus))
print("Found in dataset:", len(found_skus))
print("Missing from dataset:", len(missing_skus))

if missing_skus:
    print("First 50 missing SKUs:", missing_skus[:50])

Focus SKU count from file: 728
First 10 SKUs: [600308, 600310, 600311, 600315, 600319, 600457, 600458, 600459, 600460, 600461]
Filtered pharma rows: 35708
Filtered pharma SKUs: 727
Requested pharma SKU count: 728
Found in dataset: 727
Missing from dataset: 1
First 50 missing SKUs: [612103]


In [6]:
# Clean raw negatives
for c in [
    "Secondary_Sales_Qty",
    "Primary_Sales_Qty",
    "Free_Qty",
    "Available_Primary_Inventory_Qty",
    "Distributor_Inventory_Qty",
    "Blocked_Stock_Qty",
    "Inspection_Stock_Qty",
    "Total_Primary_Inventory_Qty"
]:
    if c in Data.columns:
        Data[c] = Data[c].clip(lower=0)

# Base observed movement
Data["Observed_Demand"] = Data["Secondary_Sales_Qty"].clip(lower=0)

In [7]:
print(Data.info())
print(Data.head(5))
print(Data.count())

<class 'pandas.core.frame.DataFrame'>
Index: 35708 entries, 462 to 146352
Data columns (total 17 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Month                            35708 non-null  object 
 1   Year                             35708 non-null  int64  
 2   Month_Number                     35708 non-null  int64  
 3   ItemCode                         35708 non-null  int64  
 4   Secondary_Sales_Qty              35708 non-null  float64
 5   Free_Qty                         35708 non-null  float64
 6   Primary_Sales_Qty                35708 non-null  float64
 7   Available_Primary_Inventory_Qty  35708 non-null  float64
 8   Blocked_Stock_Qty                35708 non-null  float64
 9   Inspection_Stock_Qty             35708 non-null  float64
 10  Total_Primary_Inventory_Qty      35708 non-null  float64
 11  Distributor_Inventory_Qty        35708 non-null  int64  
 12  Bonus_Flag          

### DATA QUALITY CHECKS

In [8]:
# Check Duplicates
dup_count = Data.duplicated().sum()
print(dup_count)

# Check Nulls
null_count = Data.isnull().sum()
print("Nulls:\n", null_count)

0
Nulls:
 Month                              0
Year                               0
Month_Number                       0
ItemCode                           0
Secondary_Sales_Qty                0
Free_Qty                           0
Primary_Sales_Qty                  0
Available_Primary_Inventory_Qty    0
Blocked_Stock_Qty                  0
Inspection_Stock_Qty               0
Total_Primary_Inventory_Qty        0
Distributor_Inventory_Qty          0
Bonus_Flag                         0
Supply_Constraint_Flag             0
Distributor_Buffer_Flag            0
Time_Index                         0
Observed_Demand                    0
dtype: int64


### DEMAND SIGNAL CONSTRUCTION

In [9]:
# =========================
# PAST-ONLY HELPER FEATURES
# =========================
grp = Data.groupby("ItemCode")

# Lag demand
Data["Lag1_Obs"] = grp["Observed_Demand"].shift(1)
Data["Lag2_Obs"] = grp["Observed_Demand"].shift(2)
Data["Lag3_Obs"] = grp["Observed_Demand"].shift(3)
Data["Lag6_Obs"] = grp["Observed_Demand"].shift(6)
Data["Lag12_Obs"] = grp["Observed_Demand"].shift(12)

# Rolling stats from observed demand
Data["Rolling3M_Obs_Mean"] = grp["Observed_Demand"].transform(lambda x: x.rolling(3, min_periods=1).mean().shift(1))
Data["Rolling6M_Obs_Mean"] = grp["Observed_Demand"].transform(lambda x: x.rolling(6, min_periods=1).mean().shift(1))
Data["Rolling3M_Obs_Std"] = grp["Observed_Demand"].transform(lambda x: x.rolling(3, min_periods=1).std().shift(1)).fillna(0)

# Safe baseline
Data["Baseline_Demand"] = Data["Rolling3M_Obs_Mean"].fillna(Data["Lag1_Obs"]).fillna(0)

# Uplift ratio
# safer denominator
safe_baseline = np.maximum(Data["Baseline_Demand"], 1)

Data["Uplift_vs_Baseline"] = Data["Observed_Demand"] / safe_baseline

# clip extreme values (VERY IMPORTANT)
Data["Uplift_vs_Baseline"] = Data["Uplift_vs_Baseline"].clip(0, 5)

# Z-score past-only
Data["Z_Score_Obs"] = (
    (Data["Observed_Demand"] - Data["Rolling3M_Obs_Mean"]) /
    (Data["Rolling3M_Obs_Std"] + 1)
).fillna(0)

In [10]:
# =========================
# RECURRING BONUS SKU DETECTION
# =========================
def detect_recurring_bonus_skus(df,
                                min_bonus_months=3,
                                gap_tolerance=1,
                                uplift_threshold=1.4):
    """
    Detect SKUs where bonus months happen in a stable repeated interval
    and those months consistently create strong uplift.
    """
    out = []

    for item, g in df.groupby("ItemCode"):
        g = g.sort_values(["Year", "Month_Number"]).copy()
        g["Time_Index"] = np.arange(len(g))

        bonus_rows = g[g["Bonus_Flag"] == 1].copy()

        recurring_flag = 0
        cycle_len = 0
        avg_gap = np.nan
        bonus_freq_12m = 0.0
        avg_bonus_uplift = 1.0

        if len(g) > 0:
            bonus_freq_12m = bonus_rows.shape[0] / len(g)

        if len(bonus_rows) >= min_bonus_months:
            gaps = bonus_rows["Time_Index"].diff().dropna()

            if len(gaps) > 0:
                avg_gap = gaps.mean()

                rounded_gap = int(round(avg_gap))
                stable_gap = ((gaps - rounded_gap).abs() <= gap_tolerance).mean()

                avg_bonus_uplift = bonus_rows["Uplift_vs_Baseline"].replace([np.inf, -np.inf], np.nan).clip(0,5).fillna(1.0).median()

                # recurring if gaps are stable and uplift is meaningful
                if stable_gap >= 0.6 and avg_bonus_uplift >= uplift_threshold:
                    recurring_flag = 1
                    cycle_len = rounded_gap

        out.append({
            "ItemCode": item,
            "Recurring_Bonus_SKU": recurring_flag,
            "Bonus_Cycle_Length": cycle_len,
            "Avg_Bonus_Gap": avg_gap if pd.notna(avg_gap) else 0,
            "Bonus_Frequency_All": bonus_freq_12m,
            "Avg_Bonus_Uplift": avg_bonus_uplift
        })

    return pd.DataFrame(out)

In [11]:
# =========================
# BONUS CYCLE FEATURES
# =========================
def add_bonus_cycle_features(df):
    df = df.sort_values(["ItemCode", "Year", "Month_Number"]).copy()

    pieces = []

    for item_code, g in df.groupby("ItemCode", sort=False):
        g = g.sort_values(["Year", "Month_Number"]).copy()

        # make absolutely sure ItemCode exists as a column
        g["ItemCode"] = item_code

        bonus_positions = np.where(g["Bonus_Flag"].values == 1)[0]

        months_since_last_bonus = []
        expected_bonus_this_month = []

        for i in range(len(g)):
            past_bonus = bonus_positions[bonus_positions < i]

            if len(past_bonus) == 0:
                months_since_last_bonus.append(999)
            else:
                months_since_last_bonus.append(i - past_bonus[-1])

            cyc = g["Bonus_Cycle_Length"].iloc[i]
            recurring = g["Recurring_Bonus_SKU"].iloc[i]

            if recurring == 1 and cyc > 0 and len(past_bonus) > 0:
                expected_bonus_this_month.append(
                    1 if abs((i - past_bonus[-1]) - cyc) <= 1 else 0
                )
            else:
                expected_bonus_this_month.append(0)

        g["Months_Since_Last_Bonus"] = months_since_last_bonus
        g["Expected_Bonus_Month"] = expected_bonus_this_month

        g["Bonus_Flag_Lag1"] = g["Bonus_Flag"].shift(1).fillna(0)
        g["Bonus_Flag_Lag2"] = g["Bonus_Flag"].shift(2).fillna(0)
        g["Bonus_Flag_Lag3"] = g["Bonus_Flag"].shift(3).fillna(0)

        g["Bonus_Frequency_12M"] = (
            g["Bonus_Flag"]
            .rolling(12, min_periods=1)
            .mean()
            .shift(1)
            .fillna(0)
        )

        pieces.append(g)

    out = pd.concat(pieces, axis=0, ignore_index=True)
    return out

In [12]:
# Build recurring bonus pattern features on full historical data
bonus_pattern_df = detect_recurring_bonus_skus(Data)[[
    "ItemCode",
    "Recurring_Bonus_SKU",
    "Bonus_Cycle_Length",
    "Avg_Bonus_Gap",
    "Bonus_Frequency_All",
    "Avg_Bonus_Uplift"
]].copy()

Data = Data.drop(columns=[
    "Recurring_Bonus_SKU",
    "Bonus_Cycle_Length",
    "Avg_Bonus_Gap",
    "Bonus_Frequency_All",
    "Avg_Bonus_Uplift"
], errors="ignore")

Data = Data.merge(bonus_pattern_df, on="ItemCode", how="left")

for c in ["Recurring_Bonus_SKU", "Bonus_Cycle_Length", "Avg_Bonus_Gap", "Bonus_Frequency_All"]:
    Data[c] = Data[c].fillna(0)

Data["Avg_Bonus_Uplift"] = Data["Avg_Bonus_Uplift"].fillna(1.0)

for c in [
    "Months_Since_Last_Bonus",
    "Expected_Bonus_Month",
    "Expected_Bonus_NextMonth",
    "Post_Bonus_NextMonth_Flag",
    "Bonus_Flag_Lag1",
    "Bonus_Flag_Lag2",
    "Bonus_Flag_Lag3",
    "Bonus_Frequency_12M"
]:
    if c not in Data.columns:
        Data[c] = 0

In [13]:
# =========================
# CHANNEL / STOCK FLOW FEATURES
# =========================
Data["Net_Available_Stock"] = (
    Data["Total_Primary_Inventory_Qty"]
    - Data["Blocked_Stock_Qty"]
    - Data["Inspection_Stock_Qty"]
).clip(lower=0)

Data["Primary_Stock_Cover"] = np.where(
    Data["Baseline_Demand"] <= 0,
    0,
    Data["Net_Available_Stock"] / (Data["Baseline_Demand"] + 1)
)

Data["Distributor_Stock_Cover"] = np.where(
    Data["Baseline_Demand"] <= 0,
    0,
    Data["Distributor_Inventory_Qty"] / (Data["Baseline_Demand"] + 1)
)

Data["Primary_to_Distributor_Ratio"] = np.where(
    Data["Distributor_Inventory_Qty"] <= 0,
    0,
    Data["Net_Available_Stock"] / (Data["Distributor_Inventory_Qty"] + 1)
)

Data["Blocked_Stock_Ratio"] = np.where(
    Data["Total_Primary_Inventory_Qty"] <= 0,
    0,
    Data["Blocked_Stock_Qty"] / (Data["Total_Primary_Inventory_Qty"] + 1)
)

Data["Inspection_Stock_Ratio"] = np.where(
    Data["Total_Primary_Inventory_Qty"] <= 0,
    0,
    Data["Inspection_Stock_Qty"] / (Data["Total_Primary_Inventory_Qty"] + 1)
)

Data["Primary_Inv_Change"] = Data.groupby("ItemCode")["Net_Available_Stock"].diff().fillna(0)
Data["Distributor_Inv_Change"] = Data.groupby("ItemCode")["Distributor_Inventory_Qty"].diff().fillna(0)

Data["Primary_to_Distributor_Ratio"] = Data["Primary_to_Distributor_Ratio"].clip(upper=Data["Primary_to_Distributor_Ratio"].quantile(0.99))

Data["Distributor_Inv_Change"] = Data["Distributor_Inv_Change"].clip(upper=Data["Distributor_Inv_Change"].quantile(0.99))
Data["Primary_Inv_Change"] = Data["Primary_Inv_Change"].clip(upper=Data["Primary_Inv_Change"].quantile(0.99))


### BUSINESS RULE ADJUSTMENTS

In [14]:
Data["Effective_Demand"] = Data["Observed_Demand"].copy()

In [15]:
# ─── RULE: Supply-constraint correction ────────────────────────────────────────────────── 

Data["Supply_Baseline"] = Data["Rolling3M_Obs_Mean"].fillna(Data["Lag1_Obs"]).fillna(Data["Observed_Demand"])

supply_constrained = (Data["Supply_Constraint_Flag"] == 1)

Data["Effective_Demand"] = np.where(
    supply_constrained,
    np.maximum(Data["Observed_Demand"], 0.85 * Data["Supply_Baseline"]),
    Data["Effective_Demand"]
)

'''
If supply was constrained (stock not available), observed sales may be artificially low.
So we cap demand to a safer value: last 3-month average secondary sales (shifted to avoid leakage).

If Supply_Constraint_Flag == 1:
   Effective_Demand = min(current sales, rolling average)
Else:
   keep current demand

if constrained, observed sales may be lower than true pull.
Instead of min(current, rolling), use max(current, a safe baseline fraction)
'''

'\nIf supply was constrained (stock not available), observed sales may be artificially low.\nSo we cap demand to a safer value: last 3-month average secondary sales (shifted to avoid leakage).\n\nIf Supply_Constraint_Flag == 1:\n   Effective_Demand = min(current sales, rolling average)\nElse:\n   keep current demand\n\nif constrained, observed sales may be lower than true pull.\nInstead of min(current, rolling), use max(current, a safe baseline fraction)\n'

In [16]:
# ─── RULE: Irregular bonus spike detection via Z-score ────────────────────────────────────────────────── 

irregular_bonus_spike = (
    (Data["Bonus_Flag"] == 1) &
    (Data["Recurring_Bonus_SKU"] == 0) &
    (Data["Z_Score_Obs"] > 2.0) &
    (Data["Uplift_vs_Baseline"] > 1.6)
)
'''
High Z-score means current demand is unusually higher than its recent baseline.
+1 in denominator prevents division exploding for stable/low-variance SKUs.
'''

'\nHigh Z-score means current demand is unusually higher than its recent baseline.\n+1 in denominator prevents division exploding for stable/low-variance SKUs.\n'

In [17]:
# ─── RULE: Stockout-like demand suppression (past-only) ────────────────────────────────────────────────── 

grp = Data.groupby("ItemCode")

prev_obs = grp["Observed_Demand"].shift(1)
prev_primary_cover = grp["Primary_Stock_Cover"].shift(1)
prev_dist_cover = grp["Distributor_Stock_Cover"].shift(1)

stockout_drop_condition = (
    (Data["Observed_Demand"] < 0.65 * prev_obs.fillna(Data["Observed_Demand"])) &
    (Data["Supply_Constraint_Flag"] == 1) &
    (
        (prev_primary_cover.fillna(99) < 1.0) |
        (prev_dist_cover.fillna(99) < 1.0)
    )
)

In [18]:
# Start clean demand
Data["Clean_Demand"] = Data["Effective_Demand"].copy()

# For irregular bonus spikes: smooth partially
Data.loc[irregular_bonus_spike, "Clean_Demand"] = (
    0.60 * Data.loc[irregular_bonus_spike, "Observed_Demand"] +
    0.40 * Data.loc[irregular_bonus_spike, "Baseline_Demand"]
)

# For stockout-like drop: normalize upward toward baseline
Data.loc[stockout_drop_condition, "Clean_Demand"] = np.maximum(
    Data.loc[stockout_drop_condition, "Observed_Demand"],
    0.90 * Data.loc[stockout_drop_condition, "Baseline_Demand"]
)

Data["Clean_Demand"] = Data["Clean_Demand"].clip(lower=0)

# Flags for model
Data["Bonus_Shock"] = irregular_bonus_spike.astype(int)
# Data["Recurring_Bonus_Month"] = 0
Data["Supply_Shock"] = stockout_drop_condition.astype(int)

# PROMO INTENSITY FEATURES
Data["Free_Ratio"] = np.where(
    Data["Primary_Sales_Qty"] <= 0,
    0,
    Data["Free_Qty"] / (Data["Primary_Sales_Qty"] + 1)
)

grp = Data.groupby("ItemCode")

### SAVE BASE DATASET

In [19]:
Cleaned_Base_Data = Data.copy()
Cleaned_Base_Data.to_csv("base_cleaned_data.csv", index=False)

# Model 

In [20]:
# =========================
# SKU HISTORY SEGMENTATION
# =========================
def add_history_length(df):
    df = df.copy()
    hist_len = (
        df.groupby("ItemCode")
        .size()
        .reset_index(name="History_Length")
    )
    df = df.merge(hist_len, on="ItemCode", how="left")

    df["History_Segment"] = np.select(
        [
            df["History_Length"] >= 18,
            (df["History_Length"] >= 6) & (df["History_Length"] < 18),
            df["History_Length"] < 6
        ],
        [
            "LONG",
            "MEDIUM",
            "SHORT"
        ],
        default="SHORT"
    )

    return df

Data = add_history_length(Data)

print(Data[["ItemCode", "History_Length", "History_Segment"]].drop_duplicates()["History_Segment"].value_counts())

History_Segment
LONG      602
MEDIUM    124
SHORT       1
Name: count, dtype: int64


## MODEL FEATURE ENGINEERING

#### Core Config

In [21]:
ACTUAL_TARGET_COL = "Target"
MODEL_TARGET_COL = "Residual_Target"
BASELINE_COL = "Residual_Baseline"

#### Helper Functions

##### Other

In [22]:
def sanitize(df):
    df = df.replace([np.inf, -np.inf], np.nan)
    return df.fillna(0)

def recency_weights(df, yearly_boost=0.25, base=1.0):
    y0 = df["Year"].min()
    return base + (df["Year"] - y0) * yearly_boost

def wmape(y_true, y_pred):
    denominator = np.sum(y_true)
    if denominator == 0:
        return 0
    return np.sum(np.abs(y_true - y_pred)) / denominator * 100

def forecast_bias(y_true, y_pred):
    denominator = np.sum(y_true)
    if denominator == 0:
        return 0
    return np.sum(y_pred - y_true) / denominator * 100

def recompute_target(df):
    df = df.copy()
    df[ACTUAL_TARGET_COL] = df.groupby("ItemCode")["Clean_Demand"].shift(-1)
    return df

def add_residual_target(df):
    df = df.copy()

    df[BASELINE_COL] = df["Rolling3M_Mean"].fillna(df["Lag1"]).fillna(0)
    df[MODEL_TARGET_COL] = df[ACTUAL_TARGET_COL] - df[BASELINE_COL]

    return df

def compute_clip_caps(train_df, cols, q=0.99):
    """Compute clipping caps using TRAIN only."""
    caps = {}
    for c in cols:
        if c in train_df.columns:
            caps[c] = float(train_df[c].replace([np.inf, -np.inf], np.nan).dropna().quantile(q))
    return caps

def apply_clip_caps(df, caps):
    """Apply previously computed caps to any df."""
    df = df.copy()
    for c, cap in caps.items():
        if c in df.columns:
            df[c] = df[c].clip(upper=cap)
    return df

def assert_features_exist(df, feature_cols, where=""):
    missing = [c for c in feature_cols if c not in df.columns]
    if missing:
        raise KeyError(f"[{where}] Missing required features: {missing}")

def underforecast_rate(y_true, y_pred):
    diff = y_true - y_pred
    under = np.where(diff > 0, diff, 0)
    denom = np.sum(y_true)
    if denom == 0:
        return 0
    return np.sum(under) / denom * 100

def evaluate_all_metrics(y_true, y_pred):
    return {
        "WMAPE": wmape(y_true, y_pred),
        "Bias": forecast_bias(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "Underforecast_Rate": underforecast_rate(y_true, y_pred)
    }

In [23]:
# Rebuild Time Features

def rebuild_time_features(df):
    df = df.sort_values(["ItemCode", "Year", "Month_Number"]).copy()
    grp = df.groupby("ItemCode")

    for lag in [1, 2, 3, 6, 12]:
        df[f"Lag{lag}"] = grp["Clean_Demand"].shift(lag)

    df["Rolling3M_Mean"] = grp["Clean_Demand"].transform(lambda x: x.rolling(3, min_periods=1).mean().shift(1))
    df["Rolling6M_Mean"] = grp["Clean_Demand"].transform(lambda x: x.rolling(6, min_periods=1).mean().shift(1))
    df["Rolling3M_Std"] = grp["Clean_Demand"].transform(lambda x: x.rolling(3, min_periods=1).std().shift(1)).fillna(0)
    df["Rolling3M_Std"] = df["Rolling3M_Std"].clip(upper=df["Rolling3M_Std"].quantile(0.99))

    df["Month_Sin"] = np.sin(2 * np.pi * df["Month_Number"] / 12)
    df["Month_Cos"] = np.cos(2 * np.pi * df["Month_Number"] / 12)

    df["Quarter_Sin"] = np.sin(2 * np.pi * df["Month_Number"] / 3)
    df["Quarter_Cos"] = np.cos(2 * np.pi * df["Month_Number"] / 3)

    df["Momentum"] = df["Lag1"] - df["Lag3"]

    df["Is_Zero"] = (df["Clean_Demand"] == 0).astype(int)
    df["ZeroRate_6M"] = grp["Is_Zero"].transform(
        lambda x: x.rolling(6, min_periods=1).mean().shift(1)
    ).fillna(0)

    df["Inventory_Pressure"] = np.where(
        df["Lag1"].fillna(0) == 0,
        0,
        df["Available_Primary_Inventory_Qty"] / (df["Lag1"] + 1)
    )
    df["Inventory_Pressure"] = df["Inventory_Pressure"].clip(
        upper=df["Inventory_Pressure"].quantile(0.99)
    )

    df["Net_Available_Stock"] = (
        df["Total_Primary_Inventory_Qty"]
        - df["Blocked_Stock_Qty"]
        - df["Inspection_Stock_Qty"]
    ).clip(lower=0)

    df["Stock_Cover_Months"] = np.where(
        df["Rolling3M_Mean"].fillna(0) == 0,
        0,
        df["Net_Available_Stock"] / (df["Rolling3M_Mean"] + 1)
    )
    df["Stock_Cover_Months"] = pd.Series(df["Stock_Cover_Months"], index=df.index).clip(
        upper=pd.Series(df["Stock_Cover_Months"], index=df.index).quantile(0.99)
    )

    df["Primary_Stock_Cover"] = np.where(
        df["Rolling3M_Mean"].fillna(0) == 0,
        0,
        df["Net_Available_Stock"] / (df["Rolling3M_Mean"] + 1)
    )

    df["Distributor_Stock_Cover"] = np.where(
        df["Rolling3M_Mean"].fillna(0) == 0,
        0,
        df["Distributor_Inventory_Qty"] / (df["Rolling3M_Mean"] + 1)
    )

    df["Supply_Constraint_Lag1"] = grp["Supply_Constraint_Flag"].shift(1).fillna(0)
    df["Supply_Constraint_Lag2"] = grp["Supply_Constraint_Flag"].shift(2).fillna(0)

    df["Primary_Stock_Cover_Lag1"] = grp["Primary_Stock_Cover"].shift(1).fillna(0)
    df["Distributor_Stock_Cover_Lag1"] = grp["Distributor_Stock_Cover"].shift(1).fillna(0)

    df["Free_Qty_Lag1"] = grp["Free_Qty"].shift(1).fillna(0)
    df["Free_Ratio_Lag1"] = grp["Free_Ratio"].shift(1).fillna(0)

    df["Free_Qty_Rolling3"] = grp["Free_Qty"].transform(
        lambda x: x.rolling(3, min_periods=1).mean().shift(1)
    ).fillna(0)

    df["Promo_Intensity_History"] = np.where(
        df["Rolling3M_Mean"].fillna(0) <= 0,
        0,
        df["Free_Qty_Rolling3"] / (df["Rolling3M_Mean"] + 1)
    )

    df["Primary_to_Distributor_Ratio"] = np.where(
        df["Distributor_Inventory_Qty"] <= 0,
        0,
        df["Net_Available_Stock"] / (df["Distributor_Inventory_Qty"] + 1)
    )

    df["Blocked_Stock_Ratio"] = np.where(
        df["Total_Primary_Inventory_Qty"] <= 0,
        0,
        df["Blocked_Stock_Qty"] / (df["Total_Primary_Inventory_Qty"] + 1)
    )

    df["Inspection_Stock_Ratio"] = np.where(
        df["Total_Primary_Inventory_Qty"] <= 0,
        0,
        df["Inspection_Stock_Qty"] / (df["Total_Primary_Inventory_Qty"] + 1)
    )

    df["Primary_Inv_Change"] = df.groupby("ItemCode")["Net_Available_Stock"].diff().fillna(0)
    df["Distributor_Inv_Change"] = df.groupby("ItemCode")["Distributor_Inventory_Qty"].diff().fillna(0)

    df["Primary_to_Distributor_Ratio"] = df["Primary_to_Distributor_Ratio"].clip(upper=df["Primary_to_Distributor_Ratio"].quantile(0.99))
    df["Distributor_Inv_Change"] = df["Distributor_Inv_Change"].clip(upper=df["Distributor_Inv_Change"].quantile(0.99))
    df["Primary_Inv_Change"] = df["Primary_Inv_Change"].clip(upper=df["Primary_Inv_Change"].quantile(0.99))

    # bonus history
    df["Bonus_Flag_Lag1"] = grp["Bonus_Flag"].shift(1).fillna(0)
    df["Bonus_Flag_Lag2"] = grp["Bonus_Flag"].shift(2).fillna(0)
    df["Bonus_Flag_Lag3"] = grp["Bonus_Flag"].shift(3).fillna(0)
    df["Bonus_Frequency_12M"] = grp["Bonus_Flag"].transform(
        lambda x: x.rolling(12, min_periods=1).mean().shift(1)
    ).fillna(0)

    # months since last bonus
    def _months_since_last_bonus(g):
        pos = np.where(g["Bonus_Flag"].values == 1)[0]
        out = []
        for i in range(len(g)):
            past = pos[pos < i]
            out.append(999 if len(past) == 0 else i - past[-1])
        return pd.Series(out, index=g.index)

    months_since = []
    for _, g in df.groupby("ItemCode", sort=False):
        pos = np.where(g["Bonus_Flag"].values == 1)[0]
        out = []
        for i in range(len(g)):
            past = pos[pos < i]
            out.append(999 if len(past) == 0 else i - past[-1])
        months_since.extend(out)
    df["Months_Since_Last_Bonus"] = months_since

    df["Expected_Bonus_Month"] = np.where(
        (df["Recurring_Bonus_SKU"] == 1) &
        (df["Bonus_Cycle_Length"] > 0) &
        (np.abs(df["Months_Since_Last_Bonus"] - df["Bonus_Cycle_Length"]) <= 1),
        1, 0
    )

    df["Expected_Bonus_NextMonth"] = np.where(
        (df["Recurring_Bonus_SKU"] == 1) &
        (df["Bonus_Cycle_Length"] > 0) &
        (np.abs((df["Months_Since_Last_Bonus"] + 1) - df["Bonus_Cycle_Length"]) <= 1),
        1, 0
    )

    df["Post_Bonus_NextMonth_Flag"] = ((df["Bonus_Flag"] == 1) | (df["Expected_Bonus_Month"] == 1)).astype(int)

    # ─── Recurring Bonus Month (pattern-aware promo timing) ───
    df["Recurring_Bonus_Month"] = np.where(
        (df["Recurring_Bonus_SKU"] == 1) &
        (
            (df["Bonus_Flag"] == 1) |
            (
                (df["Bonus_Cycle_Length"] > 0) &
                (np.abs(df["Months_Since_Last_Bonus"] - df["Bonus_Cycle_Length"]) <= 1)
            )
        ),
        1, 0
    )

    # Prom Block
    safe_mean = np.maximum(df["Rolling3M_Mean"].fillna(0), 1.0)
    df["Realized_Uplift"] = df["Clean_Demand"] / safe_mean
    df["Realized_Uplift"] = df["Realized_Uplift"].clip(0, 6)

    df["Bonus_Demand_Only"] = np.where(
    df["Bonus_Flag"] == 1,df["Clean_Demand"],np.nan)

    grp2 = df.groupby("ItemCode")
    df["Promo_Uplift_Lag1"] = grp2["Realized_Uplift"].shift(1).fillna(1.0)
    df["Promo_Uplift_Lag2"] = grp2["Realized_Uplift"].shift(2).fillna(1.0)
    df["Promo_Uplift_6M"] = grp2["Realized_Uplift"].transform(lambda x: x.rolling(6, min_periods=1).mean().shift(1)).fillna(1.0)
    df["Last_Bonus_Demand"] = (grp2["Bonus_Demand_Only"].transform(lambda x: x.shift(1).ffill()).fillna(0))

    df["Promo_Uplift_Lag1"] = df["Promo_Uplift_Lag1"].clip(0.5, 3.0)
    df["Promo_Uplift_Lag2"] = df["Promo_Uplift_Lag2"].clip(0.5, 3.0)
    df["Promo_Uplift_6M"] = df["Promo_Uplift_6M"].clip(0.5, 2.5)

    df["Bonus_Sin"] = df["Bonus_Flag"] * np.sin(2 * np.pi * df["Month_Number"] / 12)
    df["Bonus_Cos"] = df["Bonus_Flag"] * np.cos(2 * np.pi * df["Month_Number"] / 12)

    sku_stats = df.groupby("ItemCode").agg(
        SKU_Mean_Demand=("Clean_Demand","mean"),
        SKU_Std_Demand=("Clean_Demand","std"),
        SKU_Max_Demand=("Clean_Demand","max"),
        SKU_ZeroRate=("Is_Zero","mean"),
        SKU_BonusRate=("Bonus_Flag","mean"),
        SKU_SupplyConstraintRate=("Supply_Constraint_Flag","mean")
    ).reset_index()

    sku_stats["SKU_CV"] = np.where(
        sku_stats["SKU_Mean_Demand"] <= 0,
        0,
        sku_stats["SKU_Std_Demand"].fillna(0) /
        (sku_stats["SKU_Mean_Demand"] + 1)
    )

    df = df.drop(columns=[
        "SKU_Mean_Demand",
        "SKU_Std_Demand",
        "SKU_Max_Demand",
        "SKU_ZeroRate",
        "SKU_BonusRate",
        "SKU_SupplyConstraintRate",
        "SKU_CV"
    ], errors="ignore")

    df = df.merge(
        sku_stats[
            ["ItemCode","SKU_Mean_Demand","SKU_ZeroRate",
             "SKU_BonusRate","SKU_SupplyConstraintRate","SKU_CV"]
        ],
        on="ItemCode",
        how="left"
    )

    df["Last_Bonus_Demand"] = np.where(
        df["SKU_Mean_Demand"].fillna(0) > 0,
        np.minimum(df["Last_Bonus_Demand"], df["SKU_Mean_Demand"] * 3),
        df["Last_Bonus_Demand"]
    )

    # Demand to stock pressure
    if "Net_Available_Stock" in df.columns:
        df["Demand_to_Stock_Ratio"] = np.where(
            df["Net_Available_Stock"] <= 0,
            0,
            df["Rolling3M_Mean"] / (df["Net_Available_Stock"] + 1)
        )
    else:
        df["Demand_to_Stock_Ratio"] = 0

    df = df.drop(columns=["Bonus_Demand_Only"], errors="ignore")

    return df


##### Fold Adjustments

In [24]:
# ItemCode Encoding
def encode_itemcode(train_df, valid_df):
    train_df = train_df.copy()
    valid_df = valid_df.copy()

    categories = pd.Index(train_df["ItemCode"].astype(str).unique())
    cat_to_code = {k: i for i, k in enumerate(categories)}
    unk_code = len(cat_to_code)

    train_df["ItemCode"] = train_df["ItemCode"].astype(str).map(cat_to_code).fillna(unk_code).astype(int)
    valid_df["ItemCode"] = valid_df["ItemCode"].astype(str).map(cat_to_code).fillna(unk_code).astype(int)

    return train_df, valid_df, categories


# SKU Cap Function
def apply_sku_cap(train_df, valid_df, quantile=0.999):
    train_df = train_df.copy()
    valid_df = valid_df.copy()

    sku_cap = train_df.groupby("ItemCode")["Clean_Demand"].quantile(quantile)

    # clip TRAIN only
    train_df["Clean_Demand"] = np.minimum(
        train_df["Clean_Demand"],
        train_df["ItemCode"].map(sku_cap)
    )

    # do NOT modify valid/test target
    return train_df, valid_df


# ABC Classification Function
def apply_abc_classification(train_df, valid_df):

    train_df = train_df.copy()
    valid_df = valid_df.copy()

    sku_total = (
        train_df.groupby("ItemCode")["Clean_Demand"]
        .sum()
        .sort_values(ascending=False)
    )

    cum_pct = sku_total.cumsum() / sku_total.sum()

    abc_series = pd.cut(
        cum_pct,
        bins=[0, 0.7, 0.9, 1.0],
        labels=[0, 1, 2]
    )

    abc_map = abc_series.to_dict()

    train_df["ABC_Class"] = train_df["ItemCode"].map(abc_map).fillna(2)
    valid_df["ABC_Class"] = valid_df["ItemCode"].map(abc_map).fillna(2)

    return train_df, valid_df, abc_map


# Combined Wrapper
def apply_fold_adjustments(train_df, valid_df):
    train_df, valid_df = apply_sku_cap(train_df, valid_df)
    train_df, valid_df, abc_map = apply_abc_classification(train_df, valid_df)

    return train_df, valid_df, abc_map


# Segmentation Leakage 
def add_history_length_from_subset(df_subset, df_target=None):
    """
    Compute history length from df_subset only.
    Apply the segment labels onto df_target if provided.
    Otherwise return labels on df_subset itself.
    """
    hist_len = (
        df_subset.groupby("ItemCode")
        .size()
        .reset_index(name="History_Length")
    )

    hist_len["History_Segment"] = np.select(
        [
            hist_len["History_Length"] >= 18,
            (hist_len["History_Length"] >= 6) & (hist_len["History_Length"] < 18),
            hist_len["History_Length"] < 6
        ],
        ["LONG", "MEDIUM", "SHORT"],
        default="SHORT"
    )

    if df_target is None:
        out = df_subset.copy()
        out = out.drop(columns=["History_Length", "History_Segment"], errors="ignore")
        out = out.merge(hist_len, on="ItemCode", how="left")
        out["History_Length"] = out["History_Length"].fillna(0)
        out["History_Segment"] = out["History_Segment"].fillna("SHORT")
        return out

    out = df_target.copy()
    out = out.drop(columns=["History_Length", "History_Segment"], errors="ignore")
    out = out.merge(hist_len, on="ItemCode", how="left")
    out["History_Length"] = out["History_Length"].fillna(0)
    out["History_Segment"] = out["History_Segment"].fillna("SHORT")
    return out

#### Bonus 

Full-data bonus features below are used only for base demand cleaning / signal engineering.
Fold-safe bonus features for model training are recomputed later inside train/valid functions.

In [25]:
# Bonus Detection
def apply_recurring_bonus_features(train_df, valid_df):
    train_df = train_df.copy()
    valid_df = valid_df.copy()

    keep_cols = [
        "ItemCode",
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ]

    bonus_pattern_df = detect_recurring_bonus_skus(train_df)[keep_cols].copy()

    train_df = train_df.drop(columns=keep_cols[1:], errors="ignore")
    valid_df = valid_df.drop(columns=keep_cols[1:], errors="ignore")

    train_df = train_df.merge(bonus_pattern_df, on="ItemCode", how="left")
    valid_df = valid_df.merge(bonus_pattern_df, on="ItemCode", how="left")

    for c in [
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All"
    ]:
        train_df[c] = train_df[c].fillna(0)
        valid_df[c] = valid_df[c].fillna(0)

    train_df["Avg_Bonus_Uplift"] = train_df["Avg_Bonus_Uplift"].fillna(1.0)
    valid_df["Avg_Bonus_Uplift"] = valid_df["Avg_Bonus_Uplift"].fillna(1.0)

    train_df["_is_train"] = 1
    valid_df["_is_train"] = 0

    combined = pd.concat([train_df, valid_df], ignore_index=True)
    combined = combined.sort_values(["ItemCode", "Year", "Month_Number"]).copy()

    combined = add_bonus_cycle_features(combined)
    combined = combined.reset_index(drop=True)

    train_df = combined[combined["_is_train"] == 1].drop(columns=["_is_train"]).copy()
    valid_df = combined[combined["_is_train"] == 0].drop(columns=["_is_train"]).copy()

    train_df = train_df.reset_index(drop=True)
    valid_df = valid_df.reset_index(drop=True)

    if "ItemCode" not in train_df.columns:
        raise KeyError(f"ItemCode missing in train_df after apply_recurring_bonus_features. Columns: {train_df.columns.tolist()}")

    if "ItemCode" not in valid_df.columns:
        raise KeyError(f"ItemCode missing in valid_df after apply_recurring_bonus_features. Columns: {valid_df.columns.tolist()}")

    train_df = train_df.reset_index(drop=True)
    valid_df = valid_df.reset_index(drop=True)

    return train_df, valid_df, bonus_pattern_df


def build_promo_profile(df, min_bonus_months=3, corr_threshold_promo=0.45, corr_threshold_pure=0.70):
    df = df.copy().sort_values(["ItemCode", "Year", "Month_Number"])

    out = []

    for item, g in df.groupby("ItemCode"):
        g = g.copy()

        if "Clean_Demand" not in g.columns:
            continue

        bonus_months = g[g["Bonus_Flag"] == 1]
        non_bonus_months = g[g["Bonus_Flag"] == 0]

        bonus_count = len(bonus_months)
        total_count = len(g)
        bonus_freq = bonus_count / total_count if total_count > 0 else 0.0

        bonus_avg = float(bonus_months["Clean_Demand"].mean()) if bonus_count > 0 else 0.0
        non_bonus_avg = float(non_bonus_months["Clean_Demand"].mean()) if len(non_bonus_months) > 0 else 0.0

        bonus_std = float(bonus_months["Clean_Demand"].std()) if bonus_count > 1 else 0.0
        non_bonus_std = float(non_bonus_months["Clean_Demand"].std()) if len(non_bonus_months) > 1 else 0.0

        if non_bonus_avg < 5:
            uplift_ratio = 1.0
        else:
            uplift_ratio = bonus_avg / max(non_bonus_avg, 1.0)

        uplift_ratio = np.clip(uplift_ratio, 0, 5)

        # safer correlation
        if g["Bonus_Flag"].nunique() > 1 and g["Clean_Demand"].nunique() > 1:
            corr = g["Bonus_Flag"].corr(g["Clean_Demand"])
            corr = 0.0 if pd.isna(corr) else float(corr)
        else:
            corr = 0.0

        # share of demand happening in bonus months
        total_demand = float(g["Clean_Demand"].sum())
        bonus_demand_share = float(bonus_months["Clean_Demand"].sum() / total_demand) if total_demand > 0 else 0.0

        # profile rules
        if (
            bonus_count >= min_bonus_months
            and corr >= corr_threshold_pure
            and uplift_ratio >= 2.0
            and bonus_demand_share >= 0.65
        ):
            promo_profile = "PURE_PROMO"
        elif (
            bonus_count >= min_bonus_months
            and corr >= corr_threshold_promo
            and uplift_ratio >= 1.25
        ):
            promo_profile = "PROMO_INFLUENCED"
        else:
            promo_profile = "NORMAL"

        out.append({
            "ItemCode": item,
            "Promo_Profile": promo_profile,
            "Bonus_Corr": corr,
            "Bonus_Frequency_Profile": bonus_freq,
            "Bonus_Avg_Demand": bonus_avg,
            "NonBonus_Avg_Demand": non_bonus_avg,
            "Bonus_Uplift_Ratio_Profile": uplift_ratio,
            "Bonus_Std_Demand": bonus_std,
            "NonBonus_Std_Demand": non_bonus_std,
            "Bonus_Demand_Share": bonus_demand_share,
            "Bonus_Month_Count": bonus_count
        })

    return pd.DataFrame(out)


def merge_promo_profile(df, promo_profile_df):
    df = df.copy()

    keep_cols = [
        "ItemCode",
        "Promo_Profile",
        "Bonus_Corr",
        "Bonus_Frequency_Profile",
        "Bonus_Avg_Demand",
        "NonBonus_Avg_Demand",
        "Bonus_Uplift_Ratio_Profile",
        "Bonus_Std_Demand",
        "NonBonus_Std_Demand",
        "Bonus_Demand_Share",
        "Bonus_Month_Count"
    ]

    df = df.drop(columns=[c for c in keep_cols if c != "ItemCode"], errors="ignore")
    df = df.merge(promo_profile_df[keep_cols], on="ItemCode", how="left")

    df["Promo_Profile"] = df["Promo_Profile"].fillna("NORMAL")
    for c in [
        "Bonus_Corr",
        "Bonus_Frequency_Profile",
        "Bonus_Avg_Demand",
        "NonBonus_Avg_Demand",
        "Bonus_Uplift_Ratio_Profile",
        "Bonus_Std_Demand",
        "NonBonus_Std_Demand",
        "Bonus_Demand_Share",
        "Bonus_Month_Count"
    ]:
        df[c] = df[c].fillna(0)

    return df


## MODEL PIPELINE 

In [26]:
SINGLE_FEATURE_COLS = [
    "ItemCode", "ABC_Class",

    "Lag1", "Lag2", "Lag3", "Lag6", "Lag12",
    "Rolling3M_Mean", "Rolling6M_Mean", "Rolling3M_Std",
    "Momentum",
    "Month_Sin", "Month_Cos",
    "Quarter_Sin", "Quarter_Cos",

    "Bonus_Flag",
    "Free_Qty",
    "Free_Ratio",
    "Bonus_Flag_Lag1",
    "Free_Qty_Lag1",
    "Free_Ratio_Lag1",
    "Free_Qty_Rolling3", "Promo_Intensity_History",
    "Bonus_Frequency_12M",
    "Expected_Bonus_Month",
    "Expected_Bonus_NextMonth",
    "Post_Bonus_NextMonth_Flag",

    "Supply_Constraint_Flag",
    "Supply_Constraint_Lag1",
    "Supply_Constraint_Lag2",

    "Available_Primary_Inventory_Qty",
    "Distributor_Inventory_Qty",
    "Net_Available_Stock",
    "Stock_Cover_Months",
    "Demand_to_Stock_Ratio",
    "Primary_Stock_Cover",
    "Distributor_Stock_Cover",
    "Distributor_Stock_Cover_Lag1",

    "Inventory_Pressure",
    "Supply_Shock",
    "Recurring_Bonus_SKU",
    "Recurring_Bonus_Month",
    "Bonus_Cycle_Length",
    "Months_Since_Last_Bonus",
    "Avg_Bonus_Uplift",

    "Promo_Uplift_Lag1",
    "Promo_Uplift_Lag2",
    "Promo_Uplift_6M",
    "Last_Bonus_Demand",
    "Bonus_Sin",
    "Bonus_Cos",

    "Bonus_Corr",
    "Bonus_Frequency_Profile",
    "Bonus_Uplift_Ratio_Profile",
    "Bonus_Demand_Share",
    "Bonus_Month_Count",

    "ZeroRate_6M",
    "SKU_Mean_Demand",
    "SKU_ZeroRate",
    "SKU_CV"
]

In [27]:
# =========================
# FEATURE SETS BY SEGMENT
# =========================

LONG_FEATURE_COLS = SINGLE_FEATURE_COLS.copy()

MEDIUM_FEATURE_COLS = [
    f for f in SINGLE_FEATURE_COLS
    if f not in [
        "Lag12",
        "Rolling6M_Mean",
        "Quarter_Sin",
        "Quarter_Cos",
    ]
]

SHORT_FEATURE_COLS = [
    "ItemCode", "ABC_Class",
    "Lag1", "Lag2",
    "Rolling3M_Mean",
    "Bonus_Flag",
    "Free_Qty",
    "Free_Ratio",
    "Bonus_Flag_Lag1",
    "Promo_Intensity_History",
    "Bonus_Frequency_12M",
    "Expected_Bonus_Month",
    "Expected_Bonus_NextMonth",
    "Recurring_Bonus_SKU",
    "Bonus_Cycle_Length",
    "Months_Since_Last_Bonus",
    "Avg_Bonus_Uplift",
    "Bonus_Uplift_Ratio_Profile",
    "Bonus_Demand_Share",
    "Available_Primary_Inventory_Qty",
    "Distributor_Inventory_Qty",
    "Net_Available_Stock",
    "Distributor_Stock_Cover",
    "Supply_Constraint_Flag",
    "Supply_Constraint_Lag1",
    "ZeroRate_6M",
    "SKU_Mean_Demand",
    "SKU_ZeroRate",
    "SKU_CV",
    "History_Length"
]

# =========================
# SEGMENTED DATASETS
# =========================
Data = add_history_length_from_subset(Data)

Data_long = Data[Data["History_Segment"] == "LONG"].copy()
Data_medium = Data[Data["History_Segment"] == "MEDIUM"].copy()
Data_short = Data[Data["History_Segment"] == "SHORT"].copy()


print("LONG SKUs:", Data_long["ItemCode"].nunique())
print("MEDIUM SKUs:", Data_medium["ItemCode"].nunique())
print("SHORT SKUs:", Data_short["ItemCode"].nunique())

LONG SKUs: 602
MEDIUM SKUs: 124
SHORT SKUs: 1


### Long History Segment

In [28]:
# COMMON LONG PREP HELPERS
LONG_TUNE_YEARS = [2023, 2024]

def permutation_rank(model, test_df, feature_cols, target_col, n_repeats=5):
    
    """Permutation importance on the test set (2025). Higher = more important."""
    
    X = sanitize(test_df[feature_cols])
    y = test_df[target_col].values

    r = permutation_importance(
        model, X, y,
        scoring="neg_mean_absolute_error",
        n_repeats=n_repeats,
        random_state=42
    )

    imp = pd.DataFrame({
        "feature": feature_cols,
        "perm_importance": r.importances_mean
    }).sort_values("perm_importance", ascending=False)

    return imp

def prepare_long_frame_foldsafe(train_df, valid_df):
    train_df = train_df.copy().sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)
    valid_df = valid_df.copy().sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)

    train_df = add_history_length_from_subset(train_df, train_df)
    valid_df = add_history_length_from_subset(train_df, valid_df)

    train_df = train_df[train_df["History_Segment"] == "LONG"].copy()
    valid_df = valid_df[valid_df["History_Segment"] == "LONG"].copy()

    if train_df.empty or valid_df.empty:
        return None, None, None

    train_df, valid_df, _ = apply_recurring_bonus_features(train_df, valid_df)
    train_df, valid_df, abc_map = apply_fold_adjustments(train_df, valid_df)

    promo_profile_df = build_promo_profile(train_df)
    train_df = merge_promo_profile(train_df, promo_profile_df)
    valid_df = merge_promo_profile(valid_df, promo_profile_df)

    combined = pd.concat(
        [train_df.assign(_is_train=1), valid_df.assign(_is_train=0)],
        ignore_index=True
    ).sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)

    combined = rebuild_time_features(combined)

    train_out = combined[combined["_is_train"] == 1].drop(columns=["_is_train"]).copy()
    valid_out = combined[combined["_is_train"] == 0].drop(columns=["_is_train"]).copy()

    caps = compute_clip_caps(train_out, cols=["Inventory_Pressure", "Stock_Cover_Months"], q=0.99)
    train_out = apply_clip_caps(train_out, caps)
    valid_out = apply_clip_caps(valid_out, caps)

    train_out = recompute_target(train_out)
    valid_out = recompute_target(valid_out)

    train_out = add_residual_target(train_out)
    valid_out = add_residual_target(valid_out)

    train_out = train_out.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()
    valid_out = valid_out.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()

    if train_out.empty or valid_out.empty:
        return None, None, None

    return train_out, valid_out, {
        "abc_map": abc_map,
        "promo_profile_df": promo_profile_df,
        "clip_caps": caps
    }

def prepare_long_train_test_2025(full_data):
    train_df = full_data[full_data["Year"] < 2025].copy()
    test_df = full_data[full_data["Year"] == 2025].copy()

    if train_df.empty or test_df.empty:
        raise ValueError("Need both train (<2025) and test (2025) data.")

    train_df, test_df, prep_artifacts = prepare_long_frame_foldsafe(train_df, test_df)

    if train_df is None or test_df is None:
        raise ValueError("No usable LONG rows after fold-safe preparation.")

    train_df["ItemCode_Original"] = train_df["ItemCode"]
    test_df["ItemCode_Original"] = test_df["ItemCode"]

    train_df, test_df, itemcode_categories = encode_itemcode(train_df, test_df)
    prep_artifacts["itemcode_categories"] = itemcode_categories

    return train_df, test_df, prep_artifacts

def prepare_long_deploy_frame(full_data):
    deploy_df = full_data.copy().sort_values(["ItemCode", "Year", "Month_Number"])
    deploy_df = add_history_length_from_subset(deploy_df, deploy_df)
    deploy_df = deploy_df[deploy_df["History_Segment"] == "LONG"].copy()

    if deploy_df.empty:
        raise ValueError("No LONG rows available for deployment.")

    bonus_pattern_df = detect_recurring_bonus_skus(deploy_df)[[
        "ItemCode",
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ]].copy()

    deploy_df = deploy_df.drop(columns=[
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ], errors="ignore")

    deploy_df = deploy_df.merge(bonus_pattern_df, on="ItemCode", how="left")

    deploy_df, _ = apply_sku_cap(deploy_df.copy(), deploy_df.copy())

    for c in ["Recurring_Bonus_SKU", "Bonus_Cycle_Length", "Avg_Bonus_Gap", "Bonus_Frequency_All"]:
        deploy_df[c] = deploy_df[c].fillna(0)

    deploy_df["Avg_Bonus_Uplift"] = deploy_df["Avg_Bonus_Uplift"].fillna(1.0)

    sku_total = deploy_df.groupby("ItemCode")["Clean_Demand"].sum().sort_values(ascending=False)
    cum_pct = sku_total.cumsum() / sku_total.sum()
    abc_series = pd.cut(cum_pct, bins=[0, 0.7, 0.9, 1.0], labels=[0, 1, 2])
    abc_map = abc_series.to_dict()
    deploy_df["ABC_Class"] = deploy_df["ItemCode"].map(abc_map).fillna(2)

    promo_profile_df = build_promo_profile(deploy_df)
    deploy_df = merge_promo_profile(deploy_df, promo_profile_df)

    deploy_df = add_bonus_cycle_features(deploy_df)
    deploy_df = rebuild_time_features(deploy_df)

    caps = compute_clip_caps(deploy_df, cols=["Inventory_Pressure", "Stock_Cover_Months"], q=0.99)
    deploy_df = apply_clip_caps(deploy_df, caps)

    deploy_df = recompute_target(deploy_df)
    deploy_df = add_residual_target(deploy_df)
    deploy_df = deploy_df.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()

    deploy_df["ItemCode_Original"] = deploy_df["ItemCode"]
    deploy_df, _, itemcode_categories = encode_itemcode(deploy_df, deploy_df)

    return deploy_df, {
        "abc_map": abc_map,
        "promo_profile_df": promo_profile_df,
        "clip_caps": caps,
        "itemcode_categories": itemcode_categories
    }

def build_train_weights(df, yearly_boost=0.25):
    w = recency_weights(df, yearly_boost=yearly_boost).astype(float)
    w *= np.where(df["ABC_Class"] == 0, 2.5,
         np.where(df["ABC_Class"] == 1, 1.2, 1.0))
    return w

def standardize_long_model_output(
    df,
    actual_col,
    pred_col,
    item_col="ItemCode",
    year_col="Year",
    month_col="Month_Number",
    model_name="UNKNOWN",
    segment="LONG"
):
    out = df.copy()

    if item_col not in out.columns:
        raise KeyError(f"Missing item column: {item_col}")
    if actual_col not in out.columns:
        raise KeyError(f"Missing actual column: {actual_col}")
    if pred_col not in out.columns:
        raise KeyError(f"Missing pred column: {pred_col}")

    out["ItemCode_Original"] = out[item_col].astype(str)
    out["ItemCode"] = out["ItemCode_Original"]
    out["Actual"] = pd.to_numeric(out[actual_col], errors="coerce")
    out["Pred"] = pd.to_numeric(out[pred_col], errors="coerce").clip(lower=0)

    if year_col in out.columns:
        out["Year"] = out[year_col]
    else:
        out["Year"] = np.nan

    if month_col in out.columns:
        out["Month_Number"] = out[month_col]
    else:
        out["Month_Number"] = np.nan

    out["Error"] = out["Actual"] - out["Pred"]
    out["Abs_Error"] = np.abs(out["Error"])
    out["Segment"] = segment
    out["Model_Name"] = model_name

    keep_cols = [
        "ItemCode",
        "ItemCode_Original",
        "Year",
        "Month_Number",
        "Actual",
        "Pred",
        "Error",
        "Abs_Error",
        "Segment",
        "Model_Name"
    ]

    extra_cols = [c for c in out.columns if c not in keep_cols]
    return out[keep_cols + extra_cols].copy()


#### XGB_LONG

##### Tune

In [29]:
# TUNING FUNCTION BY SEGMENT
def tune_residual_xgb_long(full_data, feature_cols, n_trials=40, study_name=None):
    full_data = full_data.copy()

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 700, 1400),
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.06),
            "max_depth": trial.suggest_int("max_depth", 4, 7),
            "max_leaves": 64,
            "grow_policy": "lossguide",
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 6),
            "subsample": trial.suggest_float("subsample", 0.75, 0.9),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.75, 0.9),
            "gamma": trial.suggest_float("gamma", 0, 0.3),
            "reg_lambda": trial.suggest_float("reg_lambda", 2, 12),
            "reg_alpha": trial.suggest_float("reg_alpha", 0, 3),
        }

        scores = []

        for year in LONG_TUNE_YEARS:
            train_df = full_data[full_data["Year"] < year].copy()
            valid_df = full_data[full_data["Year"] == year].copy()

            train_df, valid_df, _ = prepare_long_frame_foldsafe(train_df, valid_df)
            if train_df is None or valid_df is None:
                continue

            train_df, valid_df, _ = encode_itemcode(train_df, valid_df)

            assert_features_exist(train_df, feature_cols, where="XGB_LONG_TUNE_TRAIN")
            assert_features_exist(valid_df, feature_cols, where="XGB_LONG_TUNE_VALID")

            model = xgb.XGBRegressor(
                objective="reg:squarederror",
                eval_metric="rmse",
                random_state=42,
                tree_method="hist",
                n_jobs=-1,
                **params
            )

            Xtr = sanitize(train_df[feature_cols])
            ytr = train_df[MODEL_TARGET_COL]
            Xva = sanitize(valid_df[feature_cols])

            model.fit(
                Xtr,
                ytr,
                sample_weight=build_train_weights(train_df),
                verbose=False
            )

            pred_residual = model.predict(Xva)
            pred_final = np.clip(valid_df[BASELINE_COL].values + pred_residual, 0, None)

            scores.append(wmape(valid_df[ACTUAL_TARGET_COL].values, pred_final))

        return 999999.0 if len(scores) == 0 else np.mean(scores)

    study = optuna.create_study(direction="minimize", study_name=study_name)
    study.optimize(objective, n_trials=n_trials)

    return study.best_params, study


In [30]:
long_best_params, long_study = tune_residual_xgb_long(
    full_data=Data,
    feature_cols=LONG_FEATURE_COLS,
    n_trials=40,
    study_name="residual_xgb_long"
)
print("LONG best params:", long_best_params)

[I 2026-04-01 16:07:20,382] A new study created in memory with name: residual_xgb_long
[I 2026-04-01 16:07:31,388] Trial 0 finished with value: 26.925327442296727 and parameters: {'n_estimators': 1165, 'learning_rate': 0.029877382751815305, 'max_depth': 6, 'min_child_weight': 5, 'subsample': 0.7585611152968509, 'colsample_bytree': 0.7505712075781216, 'gamma': 0.22136929745003242, 'reg_lambda': 2.1347812711096186, 'reg_alpha': 1.095488683410045}. Best is trial 0 with value: 26.925327442296727.
[I 2026-04-01 16:07:38,543] Trial 1 finished with value: 26.887024123459206 and parameters: {'n_estimators': 1131, 'learning_rate': 0.032722838819626246, 'max_depth': 4, 'min_child_weight': 4, 'subsample': 0.8511820543796175, 'colsample_bytree': 0.8650393049057962, 'gamma': 0.17133142776955199, 'reg_lambda': 2.4266239435948305, 'reg_alpha': 1.8186507497765847}. Best is trial 1 with value: 26.887024123459206.
[I 2026-04-01 16:07:49,581] Trial 2 finished with value: 26.555231902346215 and parameters

LONG best params: {'n_estimators': 741, 'learning_rate': 0.02266420097072063, 'max_depth': 5, 'min_child_weight': 1, 'subsample': 0.8058158184917442, 'colsample_bytree': 0.8820932275673908, 'gamma': 0.15653069377216178, 'reg_lambda': 6.761774462566195, 'reg_alpha': 1.348531724533847}


##### Feature Pruning

In [31]:
def train_eval_fixed_split_xgb_long(full_data, feature_cols, best_params, yearly_boost=0.25):
    train_df, test_df, prep_artifacts = prepare_long_train_test_2025(full_data)

    assert_features_exist(train_df, feature_cols, where="XGB_LONG_TRAIN")
    assert_features_exist(test_df, feature_cols, where="XGB_LONG_TEST")

    model = xgb.XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        random_state=42,
        tree_method="hist",
        n_jobs=-1,
        **best_params
    )

    model.fit(
        sanitize(train_df[feature_cols]),
        train_df[MODEL_TARGET_COL],
        sample_weight=build_train_weights(train_df, yearly_boost=yearly_boost),
        verbose=False
    )

    test_df["Pred_Residual"] = model.predict(sanitize(test_df[feature_cols]))
    test_df["Pred"] = np.clip(test_df[BASELINE_COL] + test_df["Pred_Residual"], 0, None)

    metrics = evaluate_all_metrics(test_df[ACTUAL_TARGET_COL].values, test_df["Pred"].values)

    return model, test_df, metrics, prep_artifacts

def iterative_feature_prune_xgb_long(
    full_data,
    start_features,
    best_params,
    drop_k=1,
    min_features=18,
    max_rounds=12,
    tolerance=0.10,
    n_repeats=5
):
    """
    Fold-safe feature pruning for one segment.
    Segment membership is recomputed from TRAIN history only inside evaluation.
    """

    history = []
    features = start_features.copy()

    model, test_df, m, _ = train_eval_fixed_split_xgb_long(
        full_data=full_data,
        feature_cols=features,
        best_params=best_params
    )

    best_wmape = m["WMAPE"]
    last_accepted_state = (features.copy(), model, test_df.copy(), m.copy())

    for r in range(1, max_rounds + 1):
        if len(features) <= min_features:
            break

        imp = permutation_rank(
            model=model,
            test_df=test_df,
            feature_cols=features,
            target_col=MODEL_TARGET_COL,
            n_repeats=n_repeats
        )

        protected = {
            "ItemCode",
            "ABC_Class",
            "Lag1",
            "Rolling3M_Mean",
            "Month_Sin",
            "Month_Cos",
            "Recurring_Bonus_SKU",
            "Expected_Bonus_Month"
        }

        drop_candidates = [
            f for f in imp.sort_values("perm_importance").feature.tolist()
            if f not in protected
        ]
        to_drop = drop_candidates[:drop_k]

        if not to_drop:
            break

        new_features = [f for f in features if f not in to_drop]

        new_model, new_test_df, new_m, _ = train_eval_fixed_split_xgb_long(
            full_data=full_data,
            feature_cols=new_features,
            best_params=best_params,
        )

        history.append({
            "round": r,
            "model": "XGBoost",
            "segment": "LONG",
            "dropped": to_drop,
            "n_features": len(new_features),
            **new_m
        })

        if new_m["WMAPE"] <= best_wmape + tolerance:
            features = new_features
            model, test_df = new_model, new_test_df
            best_wmape = min(best_wmape, new_m["WMAPE"])
            last_accepted_state = (features.copy(), model, test_df.copy(), new_m.copy())
        else:
            break

    results_df = pd.DataFrame(history)

    return (
        last_accepted_state[0],
        last_accepted_state[1],
        last_accepted_state[2],
        last_accepted_state[3],
        results_df
    )


In [32]:
# LONG
long_best_feats, _, _, long_best_metrics, long_prune_log = iterative_feature_prune_xgb_long(
    full_data=Data,
    start_features=LONG_FEATURE_COLS,
    best_params=long_best_params,
    drop_k=1,
    min_features=18,
    max_rounds=12,
    tolerance=0.10,
    n_repeats=5
)

print("LONG best metrics:", long_best_metrics)
print("LONG best features:", long_best_feats)
print(long_prune_log)

LONG best metrics: {'WMAPE': 24.04263877216143, 'Bias': -2.2271218245643203, 'MAE': 1399.8966910537704, 'RMSE': 3676.044181458648, 'Underforecast_Rate': 13.134880298362875}
LONG best features: ['ItemCode', 'ABC_Class', 'Lag1', 'Lag2', 'Lag3', 'Lag6', 'Lag12', 'Rolling3M_Mean', 'Rolling6M_Mean', 'Rolling3M_Std', 'Momentum', 'Month_Sin', 'Month_Cos', 'Quarter_Sin', 'Quarter_Cos', 'Bonus_Flag', 'Free_Qty', 'Free_Ratio', 'Bonus_Flag_Lag1', 'Free_Qty_Lag1', 'Free_Ratio_Lag1', 'Free_Qty_Rolling3', 'Promo_Intensity_History', 'Bonus_Frequency_12M', 'Expected_Bonus_Month', 'Expected_Bonus_NextMonth', 'Post_Bonus_NextMonth_Flag', 'Supply_Constraint_Flag', 'Supply_Constraint_Lag1', 'Supply_Constraint_Lag2', 'Available_Primary_Inventory_Qty', 'Distributor_Inventory_Qty', 'Net_Available_Stock', 'Stock_Cover_Months', 'Demand_to_Stock_Ratio', 'Primary_Stock_Cover', 'Distributor_Stock_Cover', 'Distributor_Stock_Cover_Lag1', 'Inventory_Pressure', 'Supply_Shock', 'Recurring_Bonus_SKU', 'Recurring_Bonus_

##### Evaluation Model

In [33]:
def train_single_evaluation_2025_xgb_long(full_data, feature_cols, best_params):
    print("\n========== XGBOOST LONG EVALUATION MODEL → TEST ON 2025 ==========")

    model, test_df, metrics, prep_artifacts = train_eval_fixed_split_xgb_long(
        full_data=full_data,
        feature_cols=feature_cols,
        best_params=best_params
    )

    artifacts = {
        "model": model,
        "feature_cols": feature_cols,
        "best_params": best_params,
        "itemcode_categories": prep_artifacts["itemcode_categories"],
        "abc_map": prep_artifacts["abc_map"],
        "clip_caps": prep_artifacts["clip_caps"],
        "promo_profile_df": prep_artifacts["promo_profile_df"],
        "target_mode": "residual",
        "baseline_col": BASELINE_COL,
        "actual_target_col": ACTUAL_TARGET_COL,
        "model_target_col": MODEL_TARGET_COL,
        "segment": "LONG",
        "model_name": "XGBOOST"
    }

    return artifacts, test_df, metrics

##### Deployement Model

In [34]:
def train_single_deployment_model_xgb_long(full_data, feature_cols, best_params):
    print("\n========== XGBOOST LONG DEPLOYMENT MODEL → TRAIN ON ALL COMPLETE DATA ==========")

    deploy_df, prep_artifacts = prepare_long_deploy_frame(full_data)

    assert_features_exist(deploy_df, feature_cols, where="XGB_LONG_DEPLOY_TRAIN")

    model = xgb.XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        random_state=42,
        tree_method="hist",
        n_jobs=-1,
        **best_params
    )

    model.fit(
        sanitize(deploy_df[feature_cols]),
        deploy_df[MODEL_TARGET_COL],
        sample_weight=build_train_weights(deploy_df),
        verbose=False
    )

    artifacts = {
        "model": model,
        "segment": "LONG",
        "feature_cols": feature_cols,
        "best_params": best_params,
        "itemcode_categories": prep_artifacts["itemcode_categories"],
        "abc_map": prep_artifacts["abc_map"],
        "clip_caps": prep_artifacts["clip_caps"],
        "promo_profile_df": prep_artifacts["promo_profile_df"],
        "target_mode": "residual",
        "baseline_col": BASELINE_COL,
        "actual_target_col": ACTUAL_TARGET_COL,
        "model_target_col": MODEL_TARGET_COL,
        "model_name": "XGBOOST"
    }

    return artifacts, deploy_df


##### Model Save

In [35]:
# =========================
# EVALUATION MODELS
# =========================

long_eval_artifacts, long_test_2025, long_eval_metrics = train_single_evaluation_2025_xgb_long(
    full_data=Data,
    feature_cols=long_best_feats,
    best_params=long_best_params
)
print("\n===== LONG RESIDUAL XGB EVALUATION METRICS =====")
print(long_eval_metrics)


========== XGBOOST LONG EVALUATION MODEL → TEST ON 2025 ==========

===== LONG RESIDUAL XGB EVALUATION METRICS =====
{'WMAPE': 24.04263877216143, 'Bias': -2.2271218245643203, 'MAE': 1399.8966910537704, 'RMSE': 3676.044181458648, 'Underforecast_Rate': 13.134880298362875}


In [36]:
# =========================
# DEPLOYMENT MODELS
# =========================

long_deploy_artifacts, long_deploy_train_df = train_single_deployment_model_xgb_long(
    full_data=Data_long,
    feature_cols=long_best_feats,
    best_params=long_best_params
)


========== XGBOOST LONG DEPLOYMENT MODEL → TRAIN ON ALL COMPLETE DATA ==========


In [37]:
# =========================
# SAVE ARTIFACTS
# =========================

joblib.dump(long_eval_artifacts, "xgb_long_eval_artifacts_residual.pkl")
joblib.dump(long_deploy_artifacts, "xgb_long_deploy_artifacts_residual.pkl")

['xgb_long_deploy_artifacts_residual.pkl']

#### CAT_LONG

In [38]:
# ============================================================
# CATBOOST LONG
# ============================================================

# 1) TUNING
def tune_residual_catboost_long(full_data, feature_cols, n_trials=40, study_name=None):
    full_data = full_data.copy()

    def objective(trial):
        params = {
            "loss_function": "RMSE",
            "eval_metric": "RMSE",
            "random_seed": 42,
            "verbose": 0,

            "iterations": trial.suggest_int("iterations", 500, 1400),
            "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.06),
            "depth": trial.suggest_int("depth", 4, 8),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 2.0, 15.0),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 5, 40),
            "subsample": trial.suggest_float("subsample", 0.70, 0.95),
            "rsm": trial.suggest_float("rsm", 0.70, 1.00),
            "random_strength": trial.suggest_float("random_strength", 0.0, 3.0),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 3.0)
        }

        scores = []

        for year in LONG_TUNE_YEARS:
            train_df = full_data[full_data["Year"] < year].copy()
            valid_df = full_data[full_data["Year"] == year].copy()

            train_df, valid_df, _ = prepare_long_frame_foldsafe(train_df, valid_df)
            if train_df is None or valid_df is None:
                continue

            train_df, valid_df, _ = encode_itemcode(train_df, valid_df)

            assert_features_exist(train_df, feature_cols, where="CATBOOST_LONG_TUNE_TRAIN")
            assert_features_exist(valid_df, feature_cols, where="CATBOOST_LONG_TUNE_VALID")

            Xtr = sanitize(train_df[feature_cols])
            ytr = train_df[MODEL_TARGET_COL]
            Xva = sanitize(valid_df[feature_cols])

            model = CatBoostRegressor(**params)

            model.fit(
                Xtr,
                ytr,
                sample_weight=build_train_weights(train_df),
                verbose=False
            )

            pred_residual = model.predict(Xva)
            pred_final = np.clip(valid_df[BASELINE_COL].values + pred_residual, 0, None)

            scores.append(wmape(valid_df[ACTUAL_TARGET_COL].values, pred_final))

        return 999999.0 if len(scores) == 0 else np.mean(scores)

    study = optuna.create_study(direction="minimize", study_name=study_name)
    study.optimize(objective, n_trials=n_trials)

    return study.best_params, study

In [39]:
# 2) FIXED-SPLIT EVALUATION
def train_eval_fixed_split_catboost_long(full_data, feature_cols, best_params, yearly_boost=0.25):
    train_df, test_df, prep_artifacts = prepare_long_train_test_2025(full_data)

    assert_features_exist(train_df, feature_cols, where="CATBOOST_LONG_TRAIN")
    assert_features_exist(test_df, feature_cols, where="CATBOOST_LONG_TEST")

    model = CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=42,
        verbose=0,
        **best_params
    )

    model.fit(
        sanitize(train_df[feature_cols]),
        train_df[MODEL_TARGET_COL],
        sample_weight=build_train_weights(train_df, yearly_boost=yearly_boost),
        verbose=False
    )

    test_df["Pred_Residual"] = model.predict(sanitize(test_df[feature_cols]))
    test_df["Pred"] = np.clip(test_df[BASELINE_COL] + test_df["Pred_Residual"], 0, None)

    metrics = evaluate_all_metrics(test_df[ACTUAL_TARGET_COL].values, test_df["Pred"].values)

    return model, test_df, metrics, prep_artifacts

In [40]:
# 3) FEATURE PRUNING
def iterative_feature_prune_catboost_long(
    full_data,
    start_features,
    best_params,
    drop_k=1,
    min_features=18,
    max_rounds=12,
    tolerance=0.10,
    n_repeats=5
):
    history = []
    features = start_features.copy()

    model, test_df, m, _ = train_eval_fixed_split_catboost_long(
        full_data=full_data,
        feature_cols=features,
        best_params=best_params
    )

    best_wmape = m["WMAPE"]
    last_accepted_state = (features.copy(), model, test_df.copy(), m.copy())

    for r in range(1, max_rounds + 1):
        if len(features) <= min_features:
            break

        imp = permutation_rank(
            model=model,
            test_df=test_df,
            feature_cols=features,
            target_col=MODEL_TARGET_COL,
            n_repeats=n_repeats
        )

        protected = {
            "ItemCode",
            "ABC_Class",
            "Lag1",
            "Rolling3M_Mean",
            "Month_Sin",
            "Month_Cos",
            "Recurring_Bonus_SKU",
            "Expected_Bonus_Month"
        }

        drop_candidates = [
            f for f in imp.sort_values("perm_importance").feature.tolist()
            if f not in protected
        ]
        to_drop = drop_candidates[:drop_k]

        if not to_drop:
            break

        new_features = [f for f in features if f not in to_drop]

        new_model, new_test_df, new_m, _ = train_eval_fixed_split_catboost_long(
            full_data=full_data,
            feature_cols=new_features,
            best_params=best_params
        )

        history.append({
            "round": r,
            "model": "CATBOOST",
            "segment": "LONG",
            "dropped": to_drop,
            "n_features": len(new_features),
            **new_m
        })

        if new_m["WMAPE"] <= best_wmape + tolerance:
            features = new_features
            model, test_df = new_model, new_test_df
            best_wmape = min(best_wmape, new_m["WMAPE"])
            last_accepted_state = (features.copy(), model, test_df.copy(), new_m.copy())
        else:
            break

    results_df = pd.DataFrame(history)

    return (
        last_accepted_state[0],
        last_accepted_state[1],
        last_accepted_state[2],
        last_accepted_state[3],
        results_df
    )

In [41]:
# 4) EVALUATION MODEL
def train_single_evaluation_2025_catboost_long(full_data, feature_cols, best_params):
    print("\n========== CATBOOST LONG EVALUATION MODEL → TEST ON 2025 ==========")

    model, test_df, metrics, prep_artifacts = train_eval_fixed_split_catboost_long(
        full_data=full_data,
        feature_cols=feature_cols,
        best_params=best_params
    )

    artifacts = {
        "model": model,
        "feature_cols": feature_cols,
        "best_params": best_params,
        "itemcode_categories": prep_artifacts["itemcode_categories"],
        "abc_map": prep_artifacts["abc_map"],
        "clip_caps": prep_artifacts["clip_caps"],
        "promo_profile_df": prep_artifacts["promo_profile_df"],
        "target_mode": "residual",
        "baseline_col": BASELINE_COL,
        "actual_target_col": ACTUAL_TARGET_COL,
        "model_target_col": MODEL_TARGET_COL,
        "segment": "LONG",
        "model_name": "CATBOOST"
    }

    return artifacts, test_df, metrics

In [42]:
# 5) DEPLOYMENT MODEL
def train_single_deployment_model_catboost_long(full_data, feature_cols, best_params):
    print("\n========== CATBOOST LONG DEPLOYMENT MODEL → TRAIN ON ALL COMPLETE DATA ==========")

    deploy_df, prep_artifacts = prepare_long_deploy_frame(full_data)

    assert_features_exist(deploy_df, feature_cols, where="CATBOOST_LONG_DEPLOY_TRAIN")

    model = CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=42,
        verbose=0,
        **best_params
    )

    model.fit(
        sanitize(deploy_df[feature_cols]),
        deploy_df[MODEL_TARGET_COL],
        sample_weight=build_train_weights(deploy_df),
        verbose=False
    )

    artifacts = {
        "model": model,
        "segment": "LONG",
        "feature_cols": feature_cols,
        "best_params": best_params,
        "itemcode_categories": prep_artifacts["itemcode_categories"],
        "abc_map": prep_artifacts["abc_map"],
        "clip_caps": prep_artifacts["clip_caps"],
        "promo_profile_df": prep_artifacts["promo_profile_df"],
        "target_mode": "residual",
        "baseline_col": BASELINE_COL,
        "actual_target_col": ACTUAL_TARGET_COL,
        "model_target_col": MODEL_TARGET_COL,
        "model_name": "CATBOOST"
    }

    return artifacts, deploy_df

In [43]:
# ============================================================
# CATBOOST LONG RUN
# ============================================================

catboost_long_best_params, catboost_long_study = tune_residual_catboost_long(
    full_data=Data,
    feature_cols=LONG_FEATURE_COLS,
    n_trials=40,
    study_name="residual_catboost_long"
)
print("CATBOOST LONG best params:", catboost_long_best_params)

catboost_long_best_feats, _, _, catboost_long_best_metrics, catboost_long_prune_log = iterative_feature_prune_catboost_long(
    full_data=Data,
    start_features=LONG_FEATURE_COLS,
    best_params=catboost_long_best_params,
    drop_k=1,
    min_features=18,
    max_rounds=12,
    tolerance=0.10,
    n_repeats=5
)

print("CATBOOST LONG best metrics:", catboost_long_best_metrics)
print("CATBOOST LONG best features:", catboost_long_best_feats)
print(catboost_long_prune_log)

catboost_long_eval_artifacts, catboost_long_test_2025, catboost_long_eval_metrics = train_single_evaluation_2025_catboost_long(
    full_data=Data,
    feature_cols=catboost_long_best_feats,
    best_params=catboost_long_best_params
)
print("\n===== CATBOOST LONG EVALUATION METRICS =====")
print(catboost_long_eval_metrics)

catboost_long_deploy_artifacts, catboost_long_deploy_train_df = train_single_deployment_model_catboost_long(
    full_data=Data,
    feature_cols=catboost_long_best_feats,
    best_params=catboost_long_best_params
)

joblib.dump(catboost_long_eval_artifacts, "catboost_long_eval_artifacts_residual.pkl")
joblib.dump(catboost_long_deploy_artifacts, "catboost_long_deploy_artifacts_residual.pkl")

[I 2026-04-01 16:13:02,116] A new study created in memory with name: residual_catboost_long
[I 2026-04-01 16:13:10,922] Trial 0 finished with value: 26.603675067715017 and parameters: {'iterations': 1127, 'learning_rate': 0.05202179735920813, 'depth': 5, 'l2_leaf_reg': 8.799298385816545, 'min_data_in_leaf': 12, 'subsample': 0.7468341243644961, 'rsm': 0.9157765542313474, 'random_strength': 0.4744253367044625, 'bagging_temperature': 2.6359256742716863}. Best is trial 0 with value: 26.603675067715017.
[I 2026-04-01 16:13:24,598] Trial 1 finished with value: 26.294191766820052 and parameters: {'iterations': 1143, 'learning_rate': 0.04358362652990118, 'depth': 7, 'l2_leaf_reg': 3.0695214038830296, 'min_data_in_leaf': 34, 'subsample': 0.8961522558171279, 'rsm': 0.7919691329519006, 'random_strength': 0.840463692729003, 'bagging_temperature': 2.3154588239027145}. Best is trial 1 with value: 26.294191766820052.
[I 2026-04-01 16:13:37,843] Trial 2 finished with value: 26.608522510910262 and para

CATBOOST LONG best params: {'iterations': 1057, 'learning_rate': 0.0351302876823464, 'depth': 8, 'l2_leaf_reg': 2.024196002690868, 'min_data_in_leaf': 6, 'subsample': 0.7189384881907559, 'rsm': 0.7070710465902161, 'random_strength': 1.0303810090450427, 'bagging_temperature': 2.531591459557647}
CATBOOST LONG best metrics: {'WMAPE': 24.19355720616992, 'Bias': -2.1897619587428223, 'MAE': 1408.68400505826, 'RMSE': 3757.9801242036865, 'Underforecast_Rate': 13.191659582456372}
CATBOOST LONG best features: ['ItemCode', 'ABC_Class', 'Lag1', 'Lag2', 'Lag3', 'Lag6', 'Lag12', 'Rolling3M_Mean', 'Rolling6M_Mean', 'Rolling3M_Std', 'Momentum', 'Month_Sin', 'Month_Cos', 'Quarter_Cos', 'Bonus_Flag', 'Free_Qty', 'Free_Ratio', 'Bonus_Flag_Lag1', 'Free_Qty_Lag1', 'Free_Ratio_Lag1', 'Free_Qty_Rolling3', 'Promo_Intensity_History', 'Bonus_Frequency_12M', 'Expected_Bonus_Month', 'Expected_Bonus_NextMonth', 'Post_Bonus_NextMonth_Flag', 'Supply_Constraint_Flag', 'Supply_Constraint_Lag1', 'Supply_Constraint_Lag2

['catboost_long_deploy_artifacts_residual.pkl']

#### LGBM_LONG

In [44]:
# ============================================================
# LIGHTGBM LONG
# ============================================================

# 1) TUNING
def tune_residual_lgbm_long(full_data, feature_cols, n_trials=40, study_name=None):
    full_data = full_data.copy()

    def objective(trial):
        params = {
            "objective": "regression",
            "metric": "rmse",
            "random_state": 42,
            "n_jobs": -1,
            "verbosity": -1,

            "n_estimators": trial.suggest_int("n_estimators", 500, 1400),
            "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.06),
            "num_leaves": trial.suggest_int("num_leaves", 31, 127),
            "max_depth": trial.suggest_int("max_depth", 4, 8),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 60),
            "subsample": trial.suggest_float("subsample", 0.70, 0.95),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.70, 0.95),
            "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 8.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 12.0),
            "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 0.3)
        }

        scores = []

        for year in LONG_TUNE_YEARS:
            train_df = full_data[full_data["Year"] < year].copy()
            valid_df = full_data[full_data["Year"] == year].copy()

            train_df, valid_df, _ = prepare_long_frame_foldsafe(train_df, valid_df)
            if train_df is None or valid_df is None:
                continue

            train_df, valid_df, _ = encode_itemcode(train_df, valid_df)

            assert_features_exist(train_df, feature_cols, where="LGBM_LONG_TUNE_TRAIN")
            assert_features_exist(valid_df, feature_cols, where="LGBM_LONG_TUNE_VALID")

            Xtr = sanitize(train_df[feature_cols])
            ytr = train_df[MODEL_TARGET_COL]
            Xva = sanitize(valid_df[feature_cols])

            model = LGBMRegressor(**params)

            model.fit(
                Xtr,
                ytr,
                sample_weight=build_train_weights(train_df)
            )

            pred_residual = model.predict(Xva)
            pred_final = np.clip(valid_df[BASELINE_COL].values + pred_residual, 0, None)

            scores.append(wmape(valid_df[ACTUAL_TARGET_COL].values, pred_final))

        return 999999.0 if len(scores) == 0 else np.mean(scores)

    study = optuna.create_study(direction="minimize", study_name=study_name)
    study.optimize(objective, n_trials=n_trials)

    return study.best_params, study

In [45]:
# 2) FIXED-SPLIT EVALUATION
def train_eval_fixed_split_lgbm_long(full_data, feature_cols, best_params, yearly_boost=0.25):
    train_df, test_df, prep_artifacts = prepare_long_train_test_2025(full_data)

    assert_features_exist(train_df, feature_cols, where="LGBM_LONG_TRAIN")
    assert_features_exist(test_df, feature_cols, where="LGBM_LONG_TEST")

    model = LGBMRegressor(
        objective="regression",
        metric="rmse",
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
        **best_params
    )

    model.fit(
        sanitize(train_df[feature_cols]),
        train_df[MODEL_TARGET_COL],
        sample_weight=build_train_weights(train_df, yearly_boost=yearly_boost)
    )

    test_df["Pred_Residual"] = model.predict(sanitize(test_df[feature_cols]))
    test_df["Pred"] = np.clip(test_df[BASELINE_COL] + test_df["Pred_Residual"], 0, None)

    metrics = evaluate_all_metrics(test_df[ACTUAL_TARGET_COL].values, test_df["Pred"].values)

    return model, test_df, metrics, prep_artifacts

In [46]:
# 3) FEATURE PRUNING
def iterative_feature_prune_lgbm_long(
    full_data,
    start_features,
    best_params,
    drop_k=1,
    min_features=18,
    max_rounds=12,
    tolerance=0.10,
    n_repeats=5
):
    history = []
    features = start_features.copy()

    model, test_df, m, _ = train_eval_fixed_split_lgbm_long(
        full_data=full_data,
        feature_cols=features,
        best_params=best_params
    )

    best_wmape = m["WMAPE"]
    last_accepted_state = (features.copy(), model, test_df.copy(), m.copy())

    for r in range(1, max_rounds + 1):
        if len(features) <= min_features:
            break

        imp = permutation_rank(
            model=model,
            test_df=test_df,
            feature_cols=features,
            target_col=MODEL_TARGET_COL,
            n_repeats=n_repeats
        )

        protected = {
            "ItemCode",
            "ABC_Class",
            "Lag1",
            "Rolling3M_Mean",
            "Month_Sin",
            "Month_Cos",
            "Recurring_Bonus_SKU",
            "Expected_Bonus_Month"
        }

        drop_candidates = [
            f for f in imp.sort_values("perm_importance").feature.tolist()
            if f not in protected
        ]
        to_drop = drop_candidates[:drop_k]

        if not to_drop:
            break

        new_features = [f for f in features if f not in to_drop]

        new_model, new_test_df, new_m, _ = train_eval_fixed_split_lgbm_long(
            full_data=full_data,
            feature_cols=new_features,
            best_params=best_params
        )

        history.append({
            "round": r,
            "model": "LIGHTGBM",
            "segment": "LONG",
            "dropped": to_drop,
            "n_features": len(new_features),
            **new_m
        })

        if new_m["WMAPE"] <= best_wmape + tolerance:
            features = new_features
            model, test_df = new_model, new_test_df
            best_wmape = min(best_wmape, new_m["WMAPE"])
            last_accepted_state = (features.copy(), model, test_df.copy(), new_m.copy())
        else:
            break

    results_df = pd.DataFrame(history)

    return (
        last_accepted_state[0],
        last_accepted_state[1],
        last_accepted_state[2],
        last_accepted_state[3],
        results_df
    )

In [47]:
# 4) EVALUATION MODEL
def train_single_evaluation_2025_lgbm_long(full_data, feature_cols, best_params):
    print("\n========== LIGHTGBM LONG EVALUATION MODEL → TEST ON 2025 ==========")

    model, test_df, metrics, prep_artifacts = train_eval_fixed_split_lgbm_long(
        full_data=full_data,
        feature_cols=feature_cols,
        best_params=best_params
    )

    artifacts = {
        "model": model,
        "feature_cols": feature_cols,
        "best_params": best_params,
        "itemcode_categories": prep_artifacts["itemcode_categories"],
        "abc_map": prep_artifacts["abc_map"],
        "clip_caps": prep_artifacts["clip_caps"],
        "promo_profile_df": prep_artifacts["promo_profile_df"],
        "target_mode": "residual",
        "baseline_col": BASELINE_COL,
        "actual_target_col": ACTUAL_TARGET_COL,
        "model_target_col": MODEL_TARGET_COL,
        "segment": "LONG",
        "model_name": "LIGHTGBM"
    }

    return artifacts, test_df, metrics

In [48]:
# 5) DEPLOYMENT MODEL
def train_single_deployment_model_lgbm_long(full_data, feature_cols, best_params):
    print("\n========== LIGHTGBM LONG DEPLOYMENT MODEL → TRAIN ON ALL COMPLETE DATA ==========")

    deploy_df, prep_artifacts = prepare_long_deploy_frame(full_data)

    assert_features_exist(deploy_df, feature_cols, where="LGBM_LONG_DEPLOY_TRAIN")

    model = LGBMRegressor(
        objective="regression",
        metric="rmse",
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
        **best_params
    )

    model.fit(
        sanitize(deploy_df[feature_cols]),
        deploy_df[MODEL_TARGET_COL],
        sample_weight=build_train_weights(deploy_df)
    )

    artifacts = {
        "model": model,
        "segment": "LONG",
        "feature_cols": feature_cols,
        "best_params": best_params,
        "itemcode_categories": prep_artifacts["itemcode_categories"],
        "abc_map": prep_artifacts["abc_map"],
        "clip_caps": prep_artifacts["clip_caps"],
        "promo_profile_df": prep_artifacts["promo_profile_df"],
        "target_mode": "residual",
        "baseline_col": BASELINE_COL,
        "actual_target_col": ACTUAL_TARGET_COL,
        "model_target_col": MODEL_TARGET_COL,
        "model_name": "LIGHTGBM"
    }

    return artifacts, deploy_df

In [49]:
# ============================================================
# LIGHTGBM LONG RUN
# ============================================================

lgbm_long_best_params, lgbm_long_study = tune_residual_lgbm_long(
    full_data=Data,
    feature_cols=LONG_FEATURE_COLS,
    n_trials=40,
    study_name="residual_lgbm_long"
)
print("LIGHTGBM LONG best params:", lgbm_long_best_params)

lgbm_long_best_feats, _, _, lgbm_long_best_metrics, lgbm_long_prune_log = iterative_feature_prune_lgbm_long(
    full_data=Data,
    start_features=LONG_FEATURE_COLS,
    best_params=lgbm_long_best_params,
    drop_k=1,
    min_features=18,
    max_rounds=12,
    tolerance=0.10,
    n_repeats=5
)

print("LIGHTGBM LONG best metrics:", lgbm_long_best_metrics)
print("LIGHTGBM LONG best features:", lgbm_long_best_feats)
print(lgbm_long_prune_log)

lgbm_long_eval_artifacts, lgbm_long_test_2025, lgbm_long_eval_metrics = train_single_evaluation_2025_lgbm_long(
    full_data=Data,
    feature_cols=lgbm_long_best_feats,
    best_params=lgbm_long_best_params
)
print("\n===== LIGHTGBM LONG EVALUATION METRICS =====")
print(lgbm_long_eval_metrics)

lgbm_long_deploy_artifacts, lgbm_long_deploy_train_df = train_single_deployment_model_lgbm_long(
    full_data=Data,
    feature_cols=lgbm_long_best_feats,
    best_params=lgbm_long_best_params
)

joblib.dump(lgbm_long_eval_artifacts, "lgbm_long_eval_artifacts_residual.pkl")
joblib.dump(lgbm_long_deploy_artifacts, "lgbm_long_deploy_artifacts_residual.pkl")

[I 2026-04-01 16:31:22,467] A new study created in memory with name: residual_lgbm_long
[I 2026-04-01 16:31:33,542] Trial 0 finished with value: 27.62192935906147 and parameters: {'n_estimators': 879, 'learning_rate': 0.05860308737906885, 'num_leaves': 120, 'max_depth': 8, 'min_child_samples': 36, 'subsample': 0.7487616258775469, 'colsample_bytree': 0.8660103362513321, 'reg_alpha': 2.7920943905340705, 'reg_lambda': 3.2986876544101533, 'min_split_gain': 0.2116856811093354}. Best is trial 0 with value: 27.62192935906147.
[I 2026-04-01 16:31:41,267] Trial 1 finished with value: 26.843310293355422 and parameters: {'n_estimators': 556, 'learning_rate': 0.03240527130818591, 'num_leaves': 104, 'max_depth': 7, 'min_child_samples': 43, 'subsample': 0.7702932446902746, 'colsample_bytree': 0.7890345275472058, 'reg_alpha': 7.677562177185701, 'reg_lambda': 2.622408497639439, 'min_split_gain': 0.2948360632513891}. Best is trial 1 with value: 26.843310293355422.
[I 2026-04-01 16:31:49,654] Trial 2 fi

LIGHTGBM LONG best params: {'n_estimators': 774, 'learning_rate': 0.01904594141953638, 'num_leaves': 60, 'max_depth': 7, 'min_child_samples': 18, 'subsample': 0.8772302563568481, 'colsample_bytree': 0.8697487369392399, 'reg_alpha': 3.177890933611388, 'reg_lambda': 10.977527365118435, 'min_split_gain': 0.018937801112077695}
LIGHTGBM LONG best metrics: {'WMAPE': 24.05100278227426, 'Bias': -2.4553199771730587, 'MAE': 1400.383690429831, 'RMSE': 3743.2930476296606, 'Underforecast_Rate': 13.253161379723661}
LIGHTGBM LONG best features: ['ItemCode', 'ABC_Class', 'Lag1', 'Lag2', 'Lag3', 'Lag6', 'Lag12', 'Rolling3M_Mean', 'Rolling6M_Mean', 'Rolling3M_Std', 'Momentum', 'Month_Sin', 'Month_Cos', 'Bonus_Flag', 'Free_Qty', 'Free_Ratio', 'Bonus_Flag_Lag1', 'Free_Qty_Lag1', 'Free_Ratio_Lag1', 'Free_Qty_Rolling3', 'Promo_Intensity_History', 'Bonus_Frequency_12M', 'Expected_Bonus_Month', 'Expected_Bonus_NextMonth', 'Post_Bonus_NextMonth_Flag', 'Supply_Constraint_Flag', 'Supply_Constraint_Lag1', 'Supply

['lgbm_long_deploy_artifacts_residual.pkl']

#### DL_LONG

In [55]:
# 1) CONFIG
GRU_SEED = 42

GRU_SEQ_LEN = 18
GRU_BATCH_SIZE = 128
GRU_EPOCHS = 45
GRU_LR = 8e-4
GRU_WEIGHT_DECAY = 1e-5

GRU_HIDDEN_SIZE = 64
GRU_NUM_LAYERS = 2
GRU_DROPOUT = 0.25
GRU_EMBED_DIM = 32

GRU_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

GRU_TRAIN_END_YEAR = 2023
GRU_VALID_YEAR = 2024
GRU_TEST_YEAR = 2025

GRU_ABC_WEIGHT_MAP = {0: 2.5, 1: 1.2, 2: 1.0}
GRU_PROMO_WEIGHT = 1.35
GRU_SUPPLY_WEIGHT = 1.20
GRU_RECURRING_PROMO_WEIGHT = 1.25
GRU_EXPECTED_PROMO_WEIGHT = 1.20

GRU_UNDER_PENALTY = 1.75

GRU_EVAL_DIR = "gru_long_eval_artifacts"
GRU_DEPLOY_DIR = "gru_long_deploy_artifacts"

In [56]:
# 2) REPRODUCIBILITY
def gru_seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# 3) GRU HELPERS
def gru_signed_log_transform(x):
    x = np.asarray(x, dtype=float)
    return np.sign(x) * np.log1p(np.abs(x))

def gru_signed_log_inverse(x):
    x = np.asarray(x, dtype=float)
    return np.sign(x) * np.expm1(np.abs(x))

def gru_infer_next_year_month(year, month_number):
    year = int(year)
    month_number = int(month_number)
    if month_number == 12:
        return year + 1, 1
    return year, month_number + 1

def make_item_mapping_from_train(train_df):
    item_codes = sorted(train_df["ItemCode"].astype(int).unique().tolist())
    return {item: i for i, item in enumerate(item_codes)}

# 4.1) LONG DATA ADAPTER FOR GRU
def prepare_long_gru_from_prepared_df(prepared_df):
    df = prepared_df.copy().sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)

    if df.empty:
        return df

    df["Residual_Target_Log"] = gru_signed_log_transform(df[MODEL_TARGET_COL].fillna(0))

    return df

def prepare_long_gru_eval_data(full_data):
    """
    Reuse shared LONG prep:
    - train/valid split prepared fold-safely for 2024 validation
    - train/test split prepared fold-safely for 2025 evaluation
    """

    # train/valid for model fitting
    train_raw = full_data[full_data["Year"] <= GRU_TRAIN_END_YEAR].copy()
    valid_raw = full_data[full_data["Year"] == GRU_VALID_YEAR].copy()

    train_prep, valid_prep, valid_artifacts = prepare_long_frame_foldsafe(train_raw, valid_raw)
    if train_prep is None or valid_prep is None:
        raise ValueError("No usable LONG rows for GRU train/valid preparation.")

    train_prep = prepare_long_gru_from_prepared_df(train_prep)
    valid_prep = prepare_long_gru_from_prepared_df(valid_prep)

    # train/test for 2025 evaluation
    train_test_prep, test_prep, test_artifacts = prepare_long_train_test_2025(full_data)
    if train_test_prep is None or test_prep is None:
        raise ValueError("No usable LONG rows for GRU train/test preparation.")

    train_test_prep = prepare_long_gru_from_prepared_df(train_test_prep)
    test_prep = prepare_long_gru_from_prepared_df(test_prep)

    artifacts = {
        "abc_map": test_artifacts["abc_map"],
        "promo_profile_df": test_artifacts["promo_profile_df"],
        "clip_caps": test_artifacts["clip_caps"],
        "itemcode_categories": test_artifacts["itemcode_categories"]
    }

    return train_prep, valid_prep, train_test_prep, test_prep, artifacts

def prepare_long_gru_deploy_data(full_data):
    """
    Reuse shared LONG deployment prep from tree models.
    """
    deploy_prep, deploy_artifacts = prepare_long_deploy_frame(full_data)
    deploy_prep = prepare_long_gru_from_prepared_df(deploy_prep)

    return deploy_prep, deploy_artifacts


# 5) FEATURE SETS FOR GRU
GRU_LONG_SEQ_FEATURES = [
    "Clean_Demand",
    "Secondary_Sales_Qty",
    "Primary_Sales_Qty",
    "Free_Qty",
    "Free_Ratio",
    "Bonus_Flag",
    "Bonus_Flag_Lag1",
    "Bonus_Flag_Lag2",
    "Bonus_Frequency_12M",
    "Recurring_Bonus_SKU",
    "Bonus_Cycle_Length",
    "Months_Since_Last_Bonus",
    "Expected_Bonus_Month",
    "Expected_Bonus_NextMonth",
    "Post_Bonus_NextMonth_Flag",
    "Avg_Bonus_Uplift",
    "Promo_Uplift_Lag1",
    "Promo_Uplift_Lag2",
    "Promo_Uplift_6M",
    "Last_Bonus_Demand",
    "Supply_Constraint_Flag",
    "Supply_Constraint_Lag1",
    "Supply_Constraint_Lag2",
    "Supply_Shock",
    "Available_Primary_Inventory_Qty",
    "Distributor_Inventory_Qty",
    "Net_Available_Stock",
    "Inventory_Pressure",
    "Stock_Cover_Months",
    "Primary_Stock_Cover",
    "Distributor_Stock_Cover",
    "Demand_to_Stock_Ratio",
    "Lag1",
    "Lag2",
    "Lag3",
    "Lag6",
    "Lag12",
    "Rolling3M_Mean",
    "Rolling6M_Mean",
    "Rolling3M_Std",
    "Momentum",
    "Month_Sin",
    "Month_Cos",
    "Quarter_Sin",
    "Quarter_Cos",
    "ZeroRate_6M",
]

GRU_LONG_STATIC_FEATURES = [
    "ABC_Class",
    "SKU_Mean_Demand",
    "SKU_ZeroRate",
    "SKU_CV",
    "Recurring_Bonus_SKU",
    "Bonus_Cycle_Length",
    "Avg_Bonus_Uplift",
    "Bonus_Corr",
    "Bonus_Frequency_Profile",
    "Bonus_Uplift_Ratio_Profile",
    "Bonus_Demand_Share",
    "Bonus_Month_Count"
]

GRU_LONG_TARGET_COL = ACTUAL_TARGET_COL
GRU_LONG_BASELINE_COL = BASELINE_COL
GRU_LONG_RESIDUAL_COL = MODEL_TARGET_COL
GRU_LONG_RESIDUAL_LOG_COL = "Residual_Target_Log"

In [ ]:
# 6) DATASET
class LongGRUSequenceDataset(Dataset):
    def __init__(self, X_seq, X_static, X_item, y_res_log, sample_w):
        self.X_seq = X_seq
        self.X_static = X_static
        self.X_item = X_item
        self.y_res_log = y_res_log
        self.sample_w = sample_w

    def __len__(self):
        return len(self.y_res_log)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.X_seq[idx], dtype=torch.float32),
            torch.tensor(self.X_static[idx], dtype=torch.float32),
            torch.tensor(self.X_item[idx], dtype=torch.long),
            torch.tensor(self.y_res_log[idx], dtype=torch.float32),
            torch.tensor(self.sample_w[idx], dtype=torch.float32),
        )

# 7) MODEL
class LongGRUResidualForecaster(nn.Module):
    def __init__(
        self,
        num_items,
        seq_input_dim,
        static_input_dim,
        embed_dim=32,
        hidden_size=64,
        num_layers=2,
        dropout=0.25,
    ):
        super().__init__()

        self.item_embedding = nn.Embedding(num_items, embed_dim)

        self.gru = nn.GRU(
            input_size=seq_input_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.seq_fc = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.static_fc = nn.Sequential(
            nn.Linear(static_input_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.head = nn.Sequential(
            nn.Linear(64 + 32 + embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x_seq, x_static, x_item):
        out, _ = self.gru(x_seq)
        last_hidden = out[:, -1, :]

        seq_repr = self.seq_fc(last_hidden)
        static_repr = self.static_fc(x_static)
        item_repr = self.item_embedding(x_item)

        x = torch.cat([seq_repr, static_repr, item_repr], dim=1)
        pred_res_log = self.head(x).squeeze(1)
        return pred_res_log

# 8) LOSS
class WeightedAsymmetricMAELoss(nn.Module):
    def __init__(self, under_penalty=1.75):
        super().__init__()
        self.under_penalty = under_penalty

    def forward(self, preds, targets, sample_weights):
        err = preds - targets
        abs_err = torch.abs(err)
        penalty = torch.where(err < 0, self.under_penalty, 1.0)
        loss = abs_err * penalty * sample_weights
        return loss.mean()

# 9) SCALERS
@dataclass
class LongGRUScalerBundle:
    seq_scaler: StandardScaler
    static_scaler: StandardScaler

def fit_long_gru_scalers(train_df):
    seq_scaler = StandardScaler()
    static_scaler = StandardScaler()

    seq_scaler.fit(train_df[GRU_LONG_SEQ_FEATURES].fillna(0))
    static_scaler.fit(train_df[GRU_LONG_STATIC_FEATURES].fillna(0))

    return LongGRUScalerBundle(
        seq_scaler=seq_scaler,
        static_scaler=static_scaler
    )

# ============================================================
# 10) BUILD SEQUENCES
# ============================================================
def build_long_gru_sequences(df, item_to_idx, scalers, seq_len=18):
    X_seq, X_static, X_item = [], [], []
    y_res_log, sample_w = [], []
    meta = []

    for item, g in df.groupby("ItemCode"):
        if int(item) not in item_to_idx:
            continue

        g = g.sort_values(["Year", "Month_Number"]).copy().reset_index(drop=True)
        usable_idx = g.index[g[GRU_LONG_RESIDUAL_LOG_COL].notna()].tolist()

        for idx in usable_idx:
            start = idx - seq_len + 1
            if start < 0:
                continue

            seq_slice = g.iloc[start:idx + 1].copy()
            if len(seq_slice) != seq_len:
                continue

            seq_vals = seq_slice[GRU_LONG_SEQ_FEATURES].fillna(0).values
            seq_vals = scalers.seq_scaler.transform(seq_vals)

            static_vals = seq_slice.iloc[-1][GRU_LONG_STATIC_FEATURES].fillna(0).values.reshape(1, -1)
            static_vals = scalers.static_scaler.transform(static_vals)[0]

            y_val = float(g.iloc[idx][GRU_LONG_RESIDUAL_LOG_COL])

            abc_class = int(seq_slice.iloc[-1]["ABC_Class"])
            bonus_flag = int(seq_slice.iloc[-1]["Bonus_Flag"])
            supply_flag = int(seq_slice.iloc[-1]["Supply_Constraint_Flag"])
            recurring_flag = int(seq_slice.iloc[-1]["Recurring_Bonus_SKU"])
            expected_bonus_flag = int(seq_slice.iloc[-1]["Expected_Bonus_NextMonth"])

            w = GRU_ABC_WEIGHT_MAP.get(abc_class, 1.0)
            if bonus_flag == 1:
                w *= GRU_PROMO_WEIGHT
            if supply_flag == 1:
                w *= GRU_SUPPLY_WEIGHT
            if recurring_flag == 1:
                w *= GRU_RECURRING_PROMO_WEIGHT
            if expected_bonus_flag == 1:
                w *= GRU_EXPECTED_PROMO_WEIGHT

            X_seq.append(seq_vals.astype(np.float32))
            X_static.append(static_vals.astype(np.float32))
            X_item.append(item_to_idx[int(item)])
            y_res_log.append(np.float32(y_val))
            sample_w.append(np.float32(w))

            meta.append({
                "ItemCode": int(item),
                "Year": int(g.iloc[idx]["Year"]),
                "Month_Number": int(g.iloc[idx]["Month_Number"]),
                "ABC_Class": abc_class,
                "Bonus_Flag": bonus_flag,
                "Supply_Constraint_Flag": supply_flag,
                "Recurring_Bonus_SKU": recurring_flag,
                "Expected_Bonus_NextMonth": expected_bonus_flag,
                "Actual": float(g.iloc[idx][GRU_LONG_TARGET_COL]),
                "Residual_Baseline": float(g.iloc[idx][GRU_LONG_BASELINE_COL]),
                "Residual_Target": float(g.iloc[idx][GRU_LONG_RESIDUAL_COL]),
            })

    meta_columns = [
        "ItemCode",
        "Year",
        "Month_Number",
        "ABC_Class",
        "Bonus_Flag",
        "Supply_Constraint_Flag",
        "Recurring_Bonus_SKU",
        "Expected_Bonus_NextMonth",
        "Actual",
        "Residual_Baseline",
        "Residual_Target",
    ]

    meta_df = pd.DataFrame(meta, columns=meta_columns)

    return (
        np.array(X_seq, dtype=np.float32),
        np.array(X_static, dtype=np.float32),
        np.array(X_item, dtype=np.int64),
        np.array(y_res_log, dtype=np.float32),
        np.array(sample_w, dtype=np.float32),
        meta_df,
    )

def filter_sequence_pack_by_year(X_seq, X_static, X_item, y, w, meta_df, target_year):
    if meta_df.empty:
        return X_seq[:0], X_static[:0], X_item[:0], y[:0], w[:0], meta_df.copy()

    mask = meta_df["Year"] == target_year
    idx = np.where(mask.values)[0]

    return (
        X_seq[idx],
        X_static[idx],
        X_item[idx],
        y[idx],
        w[idx],
        meta_df.iloc[idx].reset_index(drop=True)
    )

# ============================================================
# 11) ARRAY SPLIT
# ============================================================
def split_long_gru_sequence_arrays(X_seq, X_static, X_item, y, w, meta_df):
    train_mask = meta_df["Year"] <= GRU_TRAIN_END_YEAR
    valid_mask = meta_df["Year"] == GRU_VALID_YEAR
    test_mask = meta_df["Year"] == GRU_TEST_YEAR

    def take(mask):
        idx = np.where(mask.values)[0]
        return (
            X_seq[idx],
            X_static[idx],
            X_item[idx],
            y[idx],
            w[idx],
            meta_df.iloc[idx].reset_index(drop=True)
        )

    return take(train_mask), take(valid_mask), take(test_mask)


# ============================================================
# 12) TRAIN / PREDICT
# ============================================================
def train_long_gru_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0

    for x_seq, x_static, x_item, y_res_log, sample_w in loader:
        x_seq = x_seq.to(GRU_DEVICE)
        x_static = x_static.to(GRU_DEVICE)
        x_item = x_item.to(GRU_DEVICE)
        y_res_log = y_res_log.to(GRU_DEVICE)
        sample_w = sample_w.to(GRU_DEVICE)

        optimizer.zero_grad()
        preds = model(x_seq, x_static, x_item)
        loss = criterion(preds, y_res_log, sample_w)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * len(y_res_log)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def predict_long_gru_residual_log(model, loader):
    model.eval()

    preds_all, y_all = [], []

    for x_seq, x_static, x_item, y_res_log, sample_w in loader:
        x_seq = x_seq.to(GRU_DEVICE)
        x_static = x_static.to(GRU_DEVICE)
        x_item = x_item.to(GRU_DEVICE)

        preds = model(x_seq, x_static, x_item).cpu().numpy()
        preds_all.append(preds)
        y_all.append(y_res_log.numpy())

    preds_all = np.concatenate(preds_all)
    y_all = np.concatenate(y_all)
    return preds_all, y_all


def evaluate_long_gru_on_loader(model, loader, meta_df):
    pred_res_log, true_res_log = predict_long_gru_residual_log(model, loader)

    pred_residual = gru_signed_log_inverse(pred_res_log)
    true_residual = gru_signed_log_inverse(true_res_log)

    out = meta_df.copy()
    out["Pred_Residual_Log"] = pred_res_log
    out["Pred_Residual"] = pred_residual
    out["True_Residual"] = true_residual

    out["Pred"] = out["Residual_Baseline"] + out["Pred_Residual"]
    out["Pred"] = out["Pred"].clip(lower=0)

    out["Error"] = out["Actual"] - out["Pred"]
    out["Abs_Error"] = np.abs(out["Error"])

    metrics = evaluate_all_metrics(out["Actual"].values, out["Pred"].values)
    return metrics, out


def fit_long_gru_model(train_dataset, valid_dataset, valid_meta, num_items, seq_input_dim, static_input_dim):
    train_loader = DataLoader(train_dataset, batch_size=GRU_BATCH_SIZE, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=GRU_BATCH_SIZE, shuffle=False)

    model = LongGRUResidualForecaster(
        num_items=num_items,
        seq_input_dim=seq_input_dim,
        static_input_dim=static_input_dim,
        embed_dim=GRU_EMBED_DIM,
        hidden_size=GRU_HIDDEN_SIZE,
        num_layers=GRU_NUM_LAYERS,
        dropout=GRU_DROPOUT,
    ).to(GRU_DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=GRU_LR, weight_decay=GRU_WEIGHT_DECAY)
    criterion = WeightedAsymmetricMAELoss(under_penalty=GRU_UNDER_PENALTY)

    best_valid_wmape = float("inf")
    best_state = None
    patience = 8
    wait = 0

    for epoch in range(1, GRU_EPOCHS + 1):
        train_loss = train_long_gru_one_epoch(model, train_loader, optimizer, criterion)
        valid_metrics, _ = evaluate_long_gru_on_loader(model, valid_loader, valid_meta)

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.5f} | "
            f"Valid WMAPE: {valid_metrics['WMAPE']:.4f} | "
            f"Valid Bias: {valid_metrics['Bias']:.4f}"
        )

        if valid_metrics["WMAPE"] < best_valid_wmape:
            best_valid_wmape = valid_metrics["WMAPE"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print("Early stopping triggered.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model


# 13) STANDARDIZE OUTPUT FOR MODEL COMPARISON
def convert_gru_long_output_for_comparison(test_result_df):
    out = test_result_df.copy()

    out["ItemCode_Original"] = out["ItemCode"].astype(str)
    out["Segment"] = "LONG"
    out["Model_Name"] = "GRU"

    keep_cols = [
        "ItemCode",
        "ItemCode_Original",
        "Year",
        "Month_Number",
        "Actual",
        "Pred",
        "Error",
        "Abs_Error",
        "Segment",
        "Model_Name"
    ]

    extra_cols = [c for c in out.columns if c not in keep_cols]
    return out[keep_cols + extra_cols].copy()


# 14) LONG GRU EVALUATION PIPELINE
def run_long_gru_eval_pipeline(full_data):
    print("\n========== LONG GRU EVALUATION PIPELINE ==========")

    train_prep, valid_prep, train_test_prep, test_prep, artifacts_meta = prepare_long_gru_eval_data(full_data)

    if train_prep.empty or valid_prep.empty or train_test_prep.empty or test_prep.empty:
        raise ValueError("Prepared LONG GRU data is empty.")

    # final evaluation should align with tree models:
    # train on <2025, test on 2025
    scalers = fit_long_gru_scalers(train_test_prep)
    item_to_idx = make_item_mapping_from_train(train_test_prep)

    train_prep = train_prep[train_prep["ItemCode"].isin(item_to_idx.keys())].copy()
    valid_prep = valid_prep[valid_prep["ItemCode"].isin(item_to_idx.keys())].copy()
    train_test_prep = train_test_prep[train_test_prep["ItemCode"].isin(item_to_idx.keys())].copy()
    test_prep = test_prep[test_prep["ItemCode"].isin(item_to_idx.keys())].copy()

    # optional monitoring sets
    train_seq_source = train_prep.copy()
    valid_seq_source = pd.concat([train_prep, valid_prep], ignore_index=True).sort_values(
        ["ItemCode", "Year", "Month_Number"]
    )

    # final train/test sets
    final_train_seq_source = train_test_prep.copy()
    test_seq_source = pd.concat([train_test_prep, test_prep], ignore_index=True).sort_values(
        ["ItemCode", "Year", "Month_Number"]
    )

    print("train_seq_source rows:", len(train_seq_source), "skus:", train_seq_source["ItemCode"].nunique())
    print("valid_seq_source rows:", len(valid_seq_source), "skus:", valid_seq_source["ItemCode"].nunique())
    print("final_train_seq_source rows:", len(final_train_seq_source), "skus:", final_train_seq_source["ItemCode"].nunique())
    print("test_seq_source rows:", len(test_seq_source), "skus:", test_seq_source["ItemCode"].nunique())

    # build train-only sequences
    train_all = build_long_gru_sequences(train_seq_source, item_to_idx, scalers, seq_len=GRU_SEQ_LEN)
    valid_all = build_long_gru_sequences(valid_seq_source, item_to_idx, scalers, seq_len=GRU_SEQ_LEN)

    final_train_all = build_long_gru_sequences(final_train_seq_source, item_to_idx, scalers, seq_len=GRU_SEQ_LEN)
    test_all = build_long_gru_sequences(test_seq_source, item_to_idx, scalers, seq_len=GRU_SEQ_LEN)

    print("train_all meta rows:", len(train_all[5]))
    print("valid_all meta rows:", len(valid_all[5]))
    print("final_train_all meta rows:", len(final_train_all[5]))
    print("test_all meta rows:", len(test_all[5]))

    if not test_all[5].empty:
        print("test_all years:")
        print(test_all[5]["Year"].value_counts().sort_index())
    else:
        print("test_all meta is empty")

    # monitoring split
    X_seq_train, X_static_train, X_item_train, y_train, w_train, meta_train = train_all
    X_seq_valid, X_static_valid, X_item_valid, y_valid, w_valid, meta_valid = filter_sequence_pack_by_year(
        *valid_all, target_year=GRU_VALID_YEAR
    )

    # final train/test split
    X_seq_final_train, X_static_final_train, X_item_final_train, y_final_train, w_final_train, meta_final_train = final_train_all
    X_seq_test, X_static_test, X_item_test, y_test, w_test, meta_test = filter_sequence_pack_by_year(
        *test_all, target_year=GRU_TEST_YEAR
    )

    if len(y_final_train) == 0:
        raise ValueError("No LONG GRU final-train sequences were created.")
    if len(y_test) == 0:
        raise ValueError("No LONG GRU test sequences were created.")

    print("LONG GRU monitor-train sequences:", len(y_train))
    print("LONG GRU monitor-valid sequences:", len(y_valid))
    print("LONG GRU final-train sequences:", len(y_final_train))
    print("LONG GRU test sequences:", len(y_test))

    # use final <2025 train for actual evaluation model
    final_train_dataset = LongGRUSequenceDataset(
        X_seq_final_train, X_static_final_train, X_item_final_train, y_final_train, w_final_train
    )
    test_dataset = LongGRUSequenceDataset(
        X_seq_test, X_static_test, X_item_test, y_test, w_test
    )

    # optional valid dataset only for monitoring if available
    valid_dataset = None
    if len(y_valid) > 0:
        valid_dataset = LongGRUSequenceDataset(
            X_seq_valid, X_static_valid, X_item_valid, y_valid, w_valid
        )

    model = fit_long_gru_model(
        train_dataset=final_train_dataset,
        valid_dataset=valid_dataset if valid_dataset is not None else final_train_dataset,
        valid_meta=meta_valid if len(y_valid) > 0 else meta_final_train.head(min(len(meta_final_train), len(y_final_train))),
        num_items=len(item_to_idx),
        seq_input_dim=X_seq_final_train.shape[2],
        static_input_dim=X_static_final_train.shape[1],
    )

    test_loader = DataLoader(test_dataset, batch_size=GRU_BATCH_SIZE, shuffle=False)
    test_metrics, test_result_df = evaluate_long_gru_on_loader(model, test_loader, meta_test)

    valid_result_df = pd.DataFrame()
    valid_metrics = {}

    if len(y_valid) > 0:
        valid_loader = DataLoader(valid_dataset, batch_size=GRU_BATCH_SIZE, shuffle=False)
        valid_metrics, valid_result_df = evaluate_long_gru_on_loader(model, valid_loader, meta_valid)

        print("\n===== LONG GRU VALID METRICS (2024) =====")
        print(valid_metrics)

    print("\n===== LONG GRU TEST METRICS (2025) =====")
    print(test_metrics)

    test_std_df = convert_gru_long_output_for_comparison(test_result_df)

    artifacts = {
        "model": model,
        "model_type": "GRU",
        "model_name": "GRU",
        "segment": "LONG",
        "seq_features": GRU_LONG_SEQ_FEATURES,
        "static_features": GRU_LONG_STATIC_FEATURES,
        "seq_len": GRU_SEQ_LEN,
        "embed_dim": GRU_EMBED_DIM,
        "hidden_size": GRU_HIDDEN_SIZE,
        "num_layers": GRU_NUM_LAYERS,
        "dropout": GRU_DROPOUT,
        "item_to_idx": item_to_idx,
        "abc_map": artifacts_meta["abc_map"],
        "promo_profile_df": artifacts_meta["promo_profile_df"],
        "clip_caps": artifacts_meta["clip_caps"],
        "valid_metrics": valid_metrics,
        "test_metrics": test_metrics
    }

    return artifacts, valid_result_df, test_result_df, test_std_df, test_metrics, scalers


# 15) SAVE LONG GRU EVAL ARTIFACTS
def save_long_gru_eval_artifacts(artifacts, scalers, valid_result_df, test_result_df, test_std_df):
    os.makedirs(GRU_EVAL_DIR, exist_ok=True)

    torch.save(
        {
            "model_state_dict": artifacts["model"].state_dict(),
            "model_type": artifacts["model_type"],
            "segment": artifacts["segment"],
            "seq_features": artifacts["seq_features"],
            "static_features": artifacts["static_features"],
            "seq_len": artifacts["seq_len"],
            "embed_dim": artifacts["embed_dim"],
            "hidden_size": artifacts["hidden_size"],
            "num_layers": artifacts["num_layers"],
            "dropout": artifacts["dropout"],
            "item_to_idx": artifacts["item_to_idx"],
            "abc_map": artifacts["abc_map"],
            "clip_caps": artifacts["clip_caps"],
            "test_metrics": artifacts["test_metrics"]
        },
        os.path.join(GRU_EVAL_DIR, "gru_long_eval_model.pt")
    )

    joblib.dump(scalers.seq_scaler, os.path.join(GRU_EVAL_DIR, "gru_long_seq_scaler.pkl"))
    joblib.dump(scalers.static_scaler, os.path.join(GRU_EVAL_DIR, "gru_long_static_scaler.pkl"))
    joblib.dump(artifacts["promo_profile_df"], os.path.join(GRU_EVAL_DIR, "gru_long_promo_profile_df.pkl"))

    valid_result_df.to_csv(os.path.join(GRU_EVAL_DIR, "gru_long_valid_predictions_2024.csv"), index=False)
    test_result_df.to_csv(os.path.join(GRU_EVAL_DIR, "gru_long_test_predictions_2025.csv"), index=False)
    test_std_df.to_csv(os.path.join(GRU_EVAL_DIR, "gru_long_test_standardized_2025.csv"), index=False)

    meta = {
        "model_type": artifacts["model_type"],
        "segment": artifacts["segment"],
        "seq_features": artifacts["seq_features"],
        "static_features": artifacts["static_features"],
        "seq_len": artifacts["seq_len"],
        "embed_dim": artifacts["embed_dim"],
        "hidden_size": artifacts["hidden_size"],
        "num_layers": artifacts["num_layers"],
        "dropout": artifacts["dropout"],
        "test_metrics": artifacts["test_metrics"]
    }

    with open(os.path.join(GRU_EVAL_DIR, "gru_long_eval_meta.json"), "w") as f:
        json.dump(meta, f, indent=2)

    print("\nSaved LONG GRU eval artifacts:")
    print(f" - {GRU_EVAL_DIR}/gru_long_eval_model.pt")
    print(f" - {GRU_EVAL_DIR}/gru_long_seq_scaler.pkl")
    print(f" - {GRU_EVAL_DIR}/gru_long_static_scaler.pkl")
    print(f" - {GRU_EVAL_DIR}/gru_long_promo_profile_df.pkl")
    print(f" - {GRU_EVAL_DIR}/gru_long_valid_predictions_2024.csv")
    print(f" - {GRU_EVAL_DIR}/gru_long_test_predictions_2025.csv")
    print(f" - {GRU_EVAL_DIR}/gru_long_test_standardized_2025.csv")
    print(f" - {GRU_EVAL_DIR}/gru_long_eval_meta.json")


# 16) LONG GRU DEPLOYMENT TRAINING
def train_long_gru_deployment_model(full_data):
    print("\n========== LONG GRU DEPLOYMENT TRAINING ==========")

    deploy_df, deploy_meta = prepare_long_gru_deploy_data(full_data)

    if deploy_df.empty:
        raise ValueError("No LONG rows available for GRU deployment.")

    deploy_df = deploy_df.dropna(subset=[GRU_LONG_TARGET_COL, GRU_LONG_RESIDUAL_COL]).copy()

    item_to_idx = make_item_mapping_from_train(deploy_df)
    scalers = fit_long_gru_scalers(deploy_df)

    X_seq, X_static, X_item, y, w, meta = build_long_gru_sequences(
        deploy_df,
        item_to_idx,
        scalers,
        seq_len=GRU_SEQ_LEN
    )

    if len(y) == 0:
        raise ValueError("No LONG GRU deployment sequences created.")

    dataset = LongGRUSequenceDataset(X_seq, X_static, X_item, y, w)
    loader = DataLoader(dataset, batch_size=GRU_BATCH_SIZE, shuffle=True)

    model = LongGRUResidualForecaster(
        num_items=len(item_to_idx),
        seq_input_dim=len(GRU_LONG_SEQ_FEATURES),
        static_input_dim=len(GRU_LONG_STATIC_FEATURES),
        embed_dim=GRU_EMBED_DIM,
        hidden_size=GRU_HIDDEN_SIZE,
        num_layers=GRU_NUM_LAYERS,
        dropout=GRU_DROPOUT,
    ).to(GRU_DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=GRU_LR, weight_decay=GRU_WEIGHT_DECAY)
    criterion = WeightedAsymmetricMAELoss(under_penalty=GRU_UNDER_PENALTY)

    best_loss = float("inf")
    best_state = None

    for epoch in range(1, GRU_EPOCHS + 1):
        model.train()
        total_loss = 0.0

        for x_seq, x_static, x_item, y_batch, w_batch in loader:
            x_seq = x_seq.to(GRU_DEVICE)
            x_static = x_static.to(GRU_DEVICE)
            x_item = x_item.to(GRU_DEVICE)
            y_batch = y_batch.to(GRU_DEVICE)
            w_batch = w_batch.to(GRU_DEVICE)

            optimizer.zero_grad()
            preds = model(x_seq, x_static, x_item)
            loss = criterion(preds, y_batch, w_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item() * len(y_batch)

        epoch_loss = total_loss / len(loader.dataset)
        print(f"Epoch {epoch:02d} | Train Loss: {epoch_loss:.5f}")

        if epoch_loss < best_loss:
            best_loss = epoch_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    artifacts = {
        "model": model,
        "model_type": "GRU",
        "model_name": "GRU",
        "segment": "LONG",
        "seq_features": GRU_LONG_SEQ_FEATURES,
        "static_features": GRU_LONG_STATIC_FEATURES,
        "seq_len": GRU_SEQ_LEN,
        "embed_dim": GRU_EMBED_DIM,
        "hidden_size": GRU_HIDDEN_SIZE,
        "num_layers": GRU_NUM_LAYERS,
        "dropout": GRU_DROPOUT,
        "item_to_idx": item_to_idx,
        "abc_map": deploy_meta["abc_map"],
        "promo_profile_df": deploy_meta["promo_profile_df"],
        "clip_caps": deploy_meta["clip_caps"],
        "itemcode_categories": deploy_meta["itemcode_categories"]
    }

    return artifacts, scalers, deploy_df

# 17) SAVE LONG GRU DEPLOY ARTIFACTS
def save_long_gru_deploy_artifacts(artifacts, scalers):
    os.makedirs(GRU_DEPLOY_DIR, exist_ok=True)

    torch.save(
        {
            "model_state_dict": artifacts["model"].state_dict(),
            "model_type": artifacts["model_type"],
            "segment": artifacts["segment"],
            "seq_features": artifacts["seq_features"],
            "static_features": artifacts["static_features"],
            "seq_len": artifacts["seq_len"],
            "embed_dim": artifacts["embed_dim"],
            "hidden_size": artifacts["hidden_size"],
            "num_layers": artifacts["num_layers"],
            "dropout": artifacts["dropout"],
            "item_to_idx": artifacts["item_to_idx"],
            "abc_map": artifacts["abc_map"],
            "clip_caps": artifacts["clip_caps"]
        },
        os.path.join(GRU_DEPLOY_DIR, "gru_long_deploy_model.pt")
    )

    joblib.dump(scalers.seq_scaler, os.path.join(GRU_DEPLOY_DIR, "gru_long_seq_scaler.pkl"))
    joblib.dump(scalers.static_scaler, os.path.join(GRU_DEPLOY_DIR, "gru_long_static_scaler.pkl"))
    joblib.dump(artifacts["promo_profile_df"], os.path.join(GRU_DEPLOY_DIR, "gru_long_promo_profile_df.pkl"))

    deploy_meta = {
        "model_type": artifacts["model_type"],
        "segment": artifacts["segment"],
        "seq_features": artifacts["seq_features"],
        "static_features": artifacts["static_features"],
        "seq_len": artifacts["seq_len"],
        "embed_dim": artifacts["embed_dim"],
        "hidden_size": artifacts["hidden_size"],
        "num_layers": artifacts["num_layers"],
        "dropout": artifacts["dropout"],
    }

    with open(os.path.join(GRU_DEPLOY_DIR, "gru_long_deploy_meta.json"), "w") as f:
        json.dump(deploy_meta, f, indent=2)

    print("\nSaved LONG GRU deploy artifacts:")
    print(f" - {GRU_DEPLOY_DIR}/gru_long_deploy_model.pt")
    print(f" - {GRU_DEPLOY_DIR}/gru_long_seq_scaler.pkl")
    print(f" - {GRU_DEPLOY_DIR}/gru_long_static_scaler.pkl")
    print(f" - {GRU_DEPLOY_DIR}/gru_long_promo_profile_df.pkl")
    print(f" - {GRU_DEPLOY_DIR}/gru_long_deploy_meta.json")


# ============================================================
# 21) RUN LONG GRU EVAL
# ============================================================
gru_seed_everything(GRU_SEED)

gru_long_eval_artifacts, gru_long_valid_df, gru_long_test_df, gru_long_test_std, gru_long_test_metrics, gru_long_eval_scalers = run_long_gru_eval_pipeline(
    full_data=Data
)

print("\n===== LONG GRU TEST METRICS =====")
print(gru_long_test_metrics)

save_long_gru_eval_artifacts(
    artifacts=gru_long_eval_artifacts,
    scalers=gru_long_eval_scalers,
    valid_result_df=gru_long_valid_df,
    test_result_df=gru_long_test_df,
    test_std_df=gru_long_test_std
)


# ============================================================
# 22) RUN LONG GRU DEPLOY
# ============================================================
gru_seed_everything(GRU_SEED)

gru_long_deploy_artifacts, gru_long_deploy_scalers, gru_long_deploy_df = train_long_gru_deployment_model(
    full_data=Data
)

save_long_gru_deploy_artifacts(
    artifacts=gru_long_deploy_artifacts,
    scalers=gru_long_deploy_scalers
)

print("\nLONG GRU deployment rows:", len(gru_long_deploy_df))



========== LONG GRU EVALUATION PIPELINE ==========
train_seq_source rows: 0 skus: 0
valid_seq_source rows: 0 skus: 0
final_train_seq_source rows: 25595 skus: 578
test_seq_source rows: 31953 skus: 578
train_all meta rows: 0
valid_all meta rows: 0
final_train_all meta rows: 15769
test_all meta rows: 22127
test_all years:
Year
2022    2731
2023    6758
2024    6280
2025    6358
Name: count, dtype: int64
LONG GRU monitor-train sequences: 0
LONG GRU monitor-valid sequences: 0
LONG GRU final-train sequences: 15769
LONG GRU test sequences: 6358


#### COMPARISON

	•	per-SKU comparison table
	•	champion map: ItemCode -> Best_Model
	•	deployment artifacts per model
	•	inference routing:
	•	lookup SKU in champion map
	•	load chosen model
	•	forecast with that model
	•	save Used_Model

In [ ]:
xgb_long_test_std = standardize_long_model_output(
    df=long_test_2025,
    actual_col=ACTUAL_TARGET_COL,
    pred_col="Pred",
    item_col="ItemCode_Original",
    year_col="Year",
    month_col="Month_Number",
    model_name="XGBOOST",
    segment="LONG"
)

catboost_long_test_std = standardize_long_model_output(
    df=catboost_long_test_2025,
    actual_col=ACTUAL_TARGET_COL,
    pred_col="Pred",
    item_col="ItemCode_Original",
    year_col="Year",
    month_col="Month_Number",
    model_name="CATBOOST",
    segment="LONG"
)

lgbm_long_test_std = standardize_long_model_output(
    df=lgbm_long_test_2025,
    actual_col=ACTUAL_TARGET_COL,
    pred_col="Pred",
    item_col="ItemCode_Original",
    year_col="Year",
    month_col="Month_Number",
    model_name="LIGHTGBM",
    segment="LONG"
)

gru_long_test_std = convert_gru_long_output_for_comparison(gru_long_test_df)


In [ ]:
# ============================================================
# LONG MODEL COMPARISON + CHAMPION MAP
# with FULL-YEAR + RECENT-MONTH scoring
# ============================================================

RECENT_MONTHS = 4   # change to 3 if you want last 3 months
FULL_WEIGHT = 0.70
RECENT_WEIGHT = 0.30

# 1) COMBINE ALL LONG MODEL TEST OUTPUTS
long_model_compare_df = pd.concat(
    [
        xgb_long_test_std.copy(),
        catboost_long_test_std.copy(),
        lgbm_long_test_std.copy(),
        gru_long_test_std.copy()
    ],
    ignore_index=True
)

# safety cleanup
long_model_compare_df["ItemCode"] = long_model_compare_df["ItemCode"].astype(str)
long_model_compare_df["ItemCode_Original"] = long_model_compare_df["ItemCode_Original"].astype(str)
long_model_compare_df["Model_Name"] = long_model_compare_df["Model_Name"].astype(str)
long_model_compare_df["Actual"] = pd.to_numeric(long_model_compare_df["Actual"], errors="coerce")
long_model_compare_df["Pred"] = pd.to_numeric(long_model_compare_df["Pred"], errors="coerce")
long_model_compare_df["Abs_Error"] = pd.to_numeric(long_model_compare_df["Abs_Error"], errors="coerce")
long_model_compare_df["Year"] = pd.to_numeric(long_model_compare_df["Year"], errors="coerce")
long_model_compare_df["Month_Number"] = pd.to_numeric(long_model_compare_df["Month_Number"], errors="coerce")

long_model_compare_df = long_model_compare_df.dropna(
    subset=["ItemCode", "Model_Name", "Actual", "Pred", "Abs_Error", "Year", "Month_Number"]
).copy()

print("Combined LONG comparison rows:", len(long_model_compare_df))
print("Models in comparison:", long_model_compare_df["Model_Name"].unique())

# ------------------------------------------------------------
# 2) IDENTIFY RECENT MONTH WINDOW
# ------------------------------------------------------------
recent_periods = (
    long_model_compare_df[["Year", "Month_Number"]]
    .drop_duplicates()
    .sort_values(["Year", "Month_Number"])
    .tail(RECENT_MONTHS)
    .copy()
)

recent_period_keys = set(
    zip(recent_periods["Year"].astype(int), recent_periods["Month_Number"].astype(int))
)

long_model_compare_df["Is_Recent_Period"] = long_model_compare_df.apply(
    lambda r: (int(r["Year"]), int(r["Month_Number"])) in recent_period_keys,
    axis=1
)

print("\nRecent periods used:")
print(recent_periods)

# ------------------------------------------------------------
# 3) FULL-YEAR SKU-MODEL SUMMARY
# ------------------------------------------------------------
long_sku_model_full_summary = (
    long_model_compare_df
    .groupby(["ItemCode", "Model_Name"], as_index=False)
    .agg(
        Full_Months=("Month_Number", "count"),
        Full_Actual_Sum=("Actual", "sum"),
        Full_Pred_Sum=("Pred", "sum"),
        Full_MAE=("Abs_Error", "mean"),
        Full_Total_Abs_Error=("Abs_Error", "sum")
    )
)

long_sku_model_full_summary["Full_WMAPE"] = np.where(
    long_sku_model_full_summary["Full_Actual_Sum"] > 0,
    long_sku_model_full_summary["Full_Total_Abs_Error"] / long_sku_model_full_summary["Full_Actual_Sum"] * 100,
    np.nan
)

long_sku_model_full_summary["Full_Bias"] = np.where(
    long_sku_model_full_summary["Full_Actual_Sum"] > 0,
    (long_sku_model_full_summary["Full_Pred_Sum"] - long_sku_model_full_summary["Full_Actual_Sum"])
    / long_sku_model_full_summary["Full_Actual_Sum"] * 100,
    np.nan
)

# ------------------------------------------------------------
# 4) RECENT-MONTH SKU-MODEL SUMMARY
# ------------------------------------------------------------
recent_df = long_model_compare_df[long_model_compare_df["Is_Recent_Period"] == True].copy()

long_sku_model_recent_summary = (
    recent_df
    .groupby(["ItemCode", "Model_Name"], as_index=False)
    .agg(
        Recent_Months=("Month_Number", "count"),
        Recent_Actual_Sum=("Actual", "sum"),
        Recent_Pred_Sum=("Pred", "sum"),
        Recent_MAE=("Abs_Error", "mean"),
        Recent_Total_Abs_Error=("Abs_Error", "sum")
    )
)

long_sku_model_recent_summary["Recent_WMAPE"] = np.where(
    long_sku_model_recent_summary["Recent_Actual_Sum"] > 0,
    long_sku_model_recent_summary["Recent_Total_Abs_Error"] / long_sku_model_recent_summary["Recent_Actual_Sum"] * 100,
    np.nan
)

long_sku_model_recent_summary["Recent_Bias"] = np.where(
    long_sku_model_recent_summary["Recent_Actual_Sum"] > 0,
    (long_sku_model_recent_summary["Recent_Pred_Sum"] - long_sku_model_recent_summary["Recent_Actual_Sum"])
    / long_sku_model_recent_summary["Recent_Actual_Sum"] * 100,
    np.nan
)

# ------------------------------------------------------------
# 5) MERGE FULL + RECENT SUMMARY
# ------------------------------------------------------------
long_sku_model_summary = long_sku_model_full_summary.merge(
    long_sku_model_recent_summary,
    on=["ItemCode", "Model_Name"],
    how="left"
)

# fallback: if recent summary missing, use full summary
long_sku_model_summary["Recent_Months"] = long_sku_model_summary["Recent_Months"].fillna(0)
long_sku_model_summary["Recent_Actual_Sum"] = long_sku_model_summary["Recent_Actual_Sum"].fillna(0)
long_sku_model_summary["Recent_Pred_Sum"] = long_sku_model_summary["Recent_Pred_Sum"].fillna(0)
long_sku_model_summary["Recent_MAE"] = long_sku_model_summary["Recent_MAE"].fillna(long_sku_model_summary["Full_MAE"])
long_sku_model_summary["Recent_Total_Abs_Error"] = long_sku_model_summary["Recent_Total_Abs_Error"].fillna(0)
long_sku_model_summary["Recent_WMAPE"] = long_sku_model_summary["Recent_WMAPE"].fillna(long_sku_model_summary["Full_WMAPE"])
long_sku_model_summary["Recent_Bias"] = long_sku_model_summary["Recent_Bias"].fillna(long_sku_model_summary["Full_Bias"])

# combined score for champion selection
long_sku_model_summary["Champion_Score"] = (
    FULL_WEIGHT * long_sku_model_summary["Full_WMAPE"] +
    RECENT_WEIGHT * long_sku_model_summary["Recent_WMAPE"]
)

print("\nLONG SKU MODEL SUMMARY")
print(long_sku_model_summary.head())

# ------------------------------------------------------------
# 6) CHAMPION MODEL PER SKU
# primary rule: lowest Champion_Score
# tie-break 1: lower Recent_WMAPE
# tie-break 2: lower Full_WMAPE
# tie-break 3: lower Full_MAE
# tie-break 4: model priority
# ------------------------------------------------------------
model_priority_map = {
    "XGBOOST": 1,
    "CATBOOST": 2,
    "LIGHTGBM": 3,
    "GRU": 4
}

long_sku_model_summary["Model_Priority"] = (
    long_sku_model_summary["Model_Name"]
    .map(model_priority_map)
    .fillna(999)
)

champion_long_map_df = (
    long_sku_model_summary
    .sort_values(
        by=[
            "ItemCode",
            "Champion_Score",
            "Recent_WMAPE",
            "Full_WMAPE",
            "Full_MAE",
            "Model_Priority"
        ],
        ascending=[True, True, True, True, True, True]
    )
    .groupby("ItemCode", as_index=False)
    .first()
)

champion_long_map_df = champion_long_map_df.rename(columns={
    "Model_Name": "Best_Model",
    "Champion_Score": "Best_Model_Score",
    "Full_WMAPE": "Best_Model_Full_WMAPE",
    "Recent_WMAPE": "Best_Model_Recent_WMAPE",
    "Full_MAE": "Best_Model_Full_MAE",
    "Recent_MAE": "Best_Model_Recent_MAE",
    "Full_Bias": "Best_Model_Full_Bias",
    "Recent_Bias": "Best_Model_Recent_Bias",
    "Full_Months": "Evaluation_Months_Full",
    "Recent_Months": "Evaluation_Months_Recent"
})

champion_long_map_df["Segment"] = "LONG"

champion_long_map_df = champion_long_map_df[
    [
        "ItemCode",
        "Segment",
        "Best_Model",
        "Best_Model_Score",
        "Best_Model_Full_WMAPE",
        "Best_Model_Recent_WMAPE",
        "Best_Model_Full_MAE",
        "Best_Model_Recent_MAE",
        "Best_Model_Full_Bias",
        "Best_Model_Recent_Bias",
        "Evaluation_Months_Full",
        "Evaluation_Months_Recent",
        "Full_Actual_Sum",
        "Full_Pred_Sum",
        "Recent_Actual_Sum",
        "Recent_Pred_Sum"
    ]
].copy()

print("\nLONG CHAMPION MAP")
print(champion_long_map_df.head())

# ------------------------------------------------------------
# 7) MODEL WIN COUNTS
# ------------------------------------------------------------
long_model_win_counts = (
    champion_long_map_df["Best_Model"]
    .value_counts(dropna=False)
    .reset_index()
)
long_model_win_counts.columns = ["Best_Model", "SKU_Count"]

print("\nLONG MODEL WIN COUNTS")
print(long_model_win_counts)

# ------------------------------------------------------------
# 8) MERGE CHAMPION BACK TO ROW LEVEL
# ------------------------------------------------------------
long_model_compare_with_champion = long_model_compare_df.merge(
    champion_long_map_df[["ItemCode", "Best_Model", "Best_Model_Score"]],
    on="ItemCode",
    how="left"
)

long_model_compare_with_champion["Is_Champion_Model"] = (
    long_model_compare_with_champion["Model_Name"] == long_model_compare_with_champion["Best_Model"]
).astype(int)

# ------------------------------------------------------------
# 9) OPTIONAL: OVERALL MODEL SUMMARY
# ------------------------------------------------------------
overall_model_summary = (
    long_model_compare_df
    .groupby("Model_Name", as_index=False)
    .agg(
        Actual_Sum=("Actual", "sum"),
        Pred_Sum=("Pred", "sum"),
        Total_Abs_Error=("Abs_Error", "sum"),
        MAE=("Abs_Error", "mean")
    )
)

overall_model_summary["WMAPE"] = np.where(
    overall_model_summary["Actual_Sum"] > 0,
    overall_model_summary["Total_Abs_Error"] / overall_model_summary["Actual_Sum"] * 100,
    np.nan
)

overall_recent_summary = (
    recent_df
    .groupby("Model_Name", as_index=False)
    .agg(
        Recent_Actual_Sum=("Actual", "sum"),
        Recent_Pred_Sum=("Pred", "sum"),
        Recent_Total_Abs_Error=("Abs_Error", "sum"),
        Recent_MAE=("Abs_Error", "mean")
    )
)

overall_recent_summary["Recent_WMAPE"] = np.where(
    overall_recent_summary["Recent_Actual_Sum"] > 0,
    overall_recent_summary["Recent_Total_Abs_Error"] / overall_recent_summary["Recent_Actual_Sum"] * 100,
    np.nan
)

overall_model_summary = overall_model_summary.merge(
    overall_recent_summary,
    on="Model_Name",
    how="left"
)

overall_model_summary["Champion_Score"] = (
    FULL_WEIGHT * overall_model_summary["WMAPE"] +
    RECENT_WEIGHT * overall_model_summary["Recent_WMAPE"]
)

print("\nOVERALL MODEL SUMMARY")
print(overall_model_summary)

# ------------------------------------------------------------
# 10) SAVE RESULTS
# ------------------------------------------------------------
with pd.ExcelWriter("long_model_comparison_and_champion_map.xlsx", engine="openpyxl") as writer:
    long_model_compare_df.to_excel(writer, sheet_name="All_Model_Row_Level", index=False)
    recent_df.to_excel(writer, sheet_name="Recent_Row_Level", index=False)
    long_sku_model_summary.to_excel(writer, sheet_name="SKU_Model_Summary", index=False)
    champion_long_map_df.to_excel(writer, sheet_name="Champion_Map", index=False)
    long_model_win_counts.to_excel(writer, sheet_name="Model_Win_Counts", index=False)
    overall_model_summary.to_excel(writer, sheet_name="Overall_Model_Summary", index=False)
    long_model_compare_with_champion.to_excel(writer, sheet_name="Row_Level_With_Champion", index=False)

print("\nSaved: long_model_comparison_and_champion_map.xlsx")

### Medium History Segment

#### XGBoost for MEDIUM HISTOR SKUs

In [ ]:
def get_last_nonzero(series):
    s = pd.Series(series).dropna()
    s = s[s > 0]
    if len(s) == 0:
        return 0.0
    return float(s.iloc[-1])


In [ ]:
def infer_expected_bonus_flag_residual(sku_code, prepared_df):
    g = prepared_df[prepared_df["ItemCode_Original"] == str(sku_code)].copy()
    g = g.sort_values(["Year", "Month_Number"])

    if g.empty:
        return 0

    recent_bonus_rate = g["Bonus_Flag"].tail(12).mean() if "Bonus_Flag" in g.columns else 0
    recurring = g["Recurring_Bonus_SKU"].iloc[-1] if "Recurring_Bonus_SKU" in g.columns else 0
    cycle_len = g["Bonus_Cycle_Length"].iloc[-1] if "Bonus_Cycle_Length" in g.columns else 0
    months_since = g["Months_Since_Last_Bonus"].iloc[-1] if "Months_Since_Last_Bonus" in g.columns else 999

    if recurring == 1 and cycle_len > 0:
        if abs((months_since + 1) - cycle_len) <= 1:
            return 1

    if recurring == 1 and recent_bonus_rate >= 0.25:
        return 1

    return 0


In [ ]:
# ============================================================
# MEDIUM SUBGROUP V1 PIPELINE
# Two internal groups inside MEDIUM:
#   1) PROMO_HEAVY
#   2) STABLE  (fallback for all non-promo-heavy medium SKUs)
# ============================================================

print("\n================ MEDIUM SUBGROUP V1 PIPELINE ================\n")

# 0) SHARED FEATURE SET
MEDIUM_SUBGROUP_FEATURE_COLS = [
    "ItemCode", "ABC_Class",

    "Lag1", "Lag2", "Lag3",
    "Rolling3M_Mean", "Rolling3M_Std",
    "Momentum",
    "Month_Sin", "Month_Cos",

    "Bonus_Flag",
    "Bonus_Flag_Lag1",
    "Free_Qty",
    "Free_Qty_Lag1",
    "Free_Qty_Rolling3",
    "Free_Ratio",
    "Promo_Intensity_History",
    "Bonus_Frequency_12M",
    "Expected_Bonus_Month",
    "Expected_Bonus_NextMonth",

    "Recurring_Bonus_SKU",
    "Bonus_Cycle_Length",
    "Months_Since_Last_Bonus",
    "Avg_Bonus_Uplift",
    "Bonus_Uplift_Ratio_Profile",
    "Bonus_Month_Count",

    "Supply_Constraint_Flag",
    "Supply_Constraint_Lag1",
    "Supply_Constraint_Lag2",

    "Available_Primary_Inventory_Qty",
    "Distributor_Inventory_Qty",
    "Net_Available_Stock",
    "Distributor_Stock_Cover",
    "Distributor_Stock_Cover_Lag1",
    "Supply_Shock",

    "ZeroRate_6M",
    "SKU_Mean_Demand",
    "SKU_ZeroRate",
    "SKU_CV",

    "Bonus_Corr",
    "Bonus_Demand_Share",
    "Promo_Uplift_Lag2",
    "Promo_Uplift_6M",
    "Last_Bonus_Demand",

    # subgroup profile features
    "Medium_SKU_Type_Encoded",
    "Medium_Bonus_Frequency",
    "Medium_Bonus_Demand_Share",
    "Medium_CV",
    "Medium_ZeroRate",
    "Medium_Supply_Rate",
    "Medium_Trend_Slope"
]

# 1) MEDIUM PROFILE BUILDING
def build_medium_sku_profile(train_df):
    train_df = train_df.copy().sort_values(["ItemCode", "Year", "Month_Number"])
    out = []

    for item, g in train_df.groupby("ItemCode"):
        g = g.copy()

        mean_demand = float(g["Clean_Demand"].mean()) if len(g) > 0 else 0.0
        std_demand = float(g["Clean_Demand"].std()) if len(g) > 1 else 0.0
        cv = 0.0 if mean_demand <= 0 else std_demand / (mean_demand + 1)

        zero_rate = float((g["Clean_Demand"] == 0).mean()) if len(g) > 0 else 0.0
        bonus_freq = float(g["Bonus_Flag"].mean()) if "Bonus_Flag" in g.columns else 0.0
        supply_rate = float(g["Supply_Constraint_Flag"].mean()) if "Supply_Constraint_Flag" in g.columns else 0.0

        total_demand = float(g["Clean_Demand"].sum())
        bonus_demand = float(g.loc[g["Bonus_Flag"] == 1, "Clean_Demand"].sum()) if total_demand > 0 else 0.0
        bonus_share = 0.0 if total_demand <= 0 else bonus_demand / total_demand

        if len(g) >= 2:
            x = np.arange(len(g))
            y = g["Clean_Demand"].values
            trend_slope = float(np.polyfit(x, y, 1)[0])
        else:
            trend_slope = 0.0

        if zero_rate > 0.5:
            sku_type = "INTERMITTENT"
        elif bonus_freq > 0.3 or bonus_share > 0.4:
            sku_type = "PROMO_HEAVY"
        elif supply_rate > 0.3:
            sku_type = "SUPPLY_AFFECTED"
        elif abs(trend_slope) > mean_demand * 0.2:
            sku_type = "TRENDING"
        else:
            sku_type = "STABLE"

        # V1 subgroup mapping:
        # PROMO_HEAVY stays PROMO_HEAVY
        # all others collapse into STABLE
        subgroup = "PROMO_HEAVY" if sku_type == "PROMO_HEAVY" else "STABLE"

        out.append({
            "ItemCode": item,
            "Medium_SKU_Type": sku_type,
            "Medium_Subgroup": subgroup,
            "Medium_Bonus_Frequency": bonus_freq,
            "Medium_Bonus_Demand_Share": bonus_share,
            "Medium_CV": cv,
            "Medium_ZeroRate": zero_rate,
            "Medium_Supply_Rate": supply_rate,
            "Medium_Trend_Slope": trend_slope
        })

    return pd.DataFrame(out)


def merge_medium_sku_profile(df, profile_df):
    df = df.copy()

    keep_cols = [
        "ItemCode",
        "Medium_SKU_Type",
        "Medium_Subgroup",
        "Medium_Bonus_Frequency",
        "Medium_Bonus_Demand_Share",
        "Medium_CV",
        "Medium_ZeroRate",
        "Medium_Supply_Rate",
        "Medium_Trend_Slope"
    ]

    df = df.drop(columns=[c for c in keep_cols if c != "ItemCode"], errors="ignore")
    df = df.merge(profile_df[keep_cols], on="ItemCode", how="left")

    df["Medium_SKU_Type"] = df["Medium_SKU_Type"].fillna("STABLE")
    df["Medium_Subgroup"] = df["Medium_Subgroup"].fillna("STABLE")

    for c in [
        "Medium_Bonus_Frequency",
        "Medium_Bonus_Demand_Share",
        "Medium_CV",
        "Medium_ZeroRate",
        "Medium_Supply_Rate",
        "Medium_Trend_Slope"
    ]:
        df[c] = df[c].fillna(0)

    type_map = {
        "STABLE": 0,
        "PROMO_HEAVY": 1,
        "TRENDING": 2,
        "SUPPLY_AFFECTED": 3,
        "INTERMITTENT": 4
    }
    df["Medium_SKU_Type_Encoded"] = df["Medium_SKU_Type"].map(type_map).fillna(0).astype(int)

    return df


# 2) FOLD-SAFE MEDIUM PREP
def prepare_medium_subgroup_frame_foldsafe(train_df, valid_df):
    train_df = train_df.copy().sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)
    valid_df = valid_df.copy().sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)

    train_df, valid_df, _ = apply_recurring_bonus_features(train_df, valid_df)
    train_df, valid_df, abc_map = apply_fold_adjustments(train_df, valid_df)

    promo_profile_df = build_promo_profile(train_df)
    train_df = merge_promo_profile(train_df, promo_profile_df)
    valid_df = merge_promo_profile(valid_df, promo_profile_df)

    medium_profile_df = build_medium_sku_profile(train_df)
    train_df = merge_medium_sku_profile(train_df, medium_profile_df)
    valid_df = merge_medium_sku_profile(valid_df, medium_profile_df)

    combined = pd.concat(
        [train_df.assign(_is_train=1), valid_df.assign(_is_train=0)],
        ignore_index=True
    )
    combined = combined.sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)

    combined = rebuild_time_features(combined)

    train_out = combined[combined["_is_train"] == 1].drop(columns=["_is_train"]).copy()
    valid_out = combined[combined["_is_train"] == 0].drop(columns=["_is_train"]).copy()

    return train_out, valid_out, abc_map, promo_profile_df, medium_profile_df


# 3) SUBGROUP FILTER
def filter_medium_subgroup(df, subgroup_name):
    df = df.copy()

    if subgroup_name == "PROMO_HEAVY":
        return df[df["Medium_Subgroup"] == "PROMO_HEAVY"].copy()

    if subgroup_name == "STABLE":
        return df[df["Medium_Subgroup"] == "STABLE"].copy()

    raise ValueError(f"Unknown subgroup_name: {subgroup_name}")


# 4) SUBGROUP TUNING
def tune_residual_xgb_medium_subgroup(
    full_data,
    feature_cols,
    subgroup_name,
    n_trials=30,
    study_name=None
):
    full_data = full_data.copy()

    if study_name is None:
        study_name = f"residual_xgb_medium_{subgroup_name.lower()}"

    def objective(trial):
        if subgroup_name == "PROMO_HEAVY":
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 600, 1400),
                "learning_rate": trial.suggest_float("learning_rate", 0.012, 0.035),
                "max_depth": trial.suggest_int("max_depth", 5, 7),
                "max_leaves": 64,
                "grow_policy": "lossguide",
                "min_child_weight": trial.suggest_int("min_child_weight", 3, 8),
                "subsample": trial.suggest_float("subsample", 0.70, 0.88),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.70, 0.88),
                "gamma": trial.suggest_float("gamma", 0.10, 0.45),
                "reg_lambda": trial.suggest_float("reg_lambda", 12, 22),
                "reg_alpha": trial.suggest_float("reg_alpha", 1.0, 6.0),
            }
        elif subgroup_name == "STABLE":
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 400, 1000),
                "learning_rate": trial.suggest_float("learning_rate", 0.012, 0.030),
                "max_depth": trial.suggest_int("max_depth", 3, 5),
                "max_leaves": 64,
                "grow_policy": "lossguide",
                "min_child_weight": trial.suggest_int("min_child_weight", 5, 12),
                "subsample": trial.suggest_float("subsample", 0.72, 0.90),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 0.85),
                "gamma": trial.suggest_float("gamma", 0.15, 0.55),
                "reg_lambda": trial.suggest_float("reg_lambda", 12, 24),
                "reg_alpha": trial.suggest_float("reg_alpha", 1.0, 6.0),
            }
        else:
            raise ValueError(f"Unknown subgroup_name: {subgroup_name}")

        scores = []

        for year in [2023, 2024]:
            train_df = full_data[full_data["Year"] < year].copy()
            valid_df = full_data[full_data["Year"] == year].copy()

            if train_df.empty or valid_df.empty:
                continue

            train_df = add_history_length_from_subset(train_df, train_df)
            valid_df = add_history_length_from_subset(train_df, valid_df)

            train_df = train_df[train_df["History_Segment"] == "MEDIUM"].copy()
            valid_df = valid_df[valid_df["History_Segment"] == "MEDIUM"].copy()

            if train_df.empty or valid_df.empty:
                continue

            train_df, valid_df, _, _, _ = prepare_medium_subgroup_frame_foldsafe(train_df, valid_df)

            train_df = filter_medium_subgroup(train_df, subgroup_name)
            valid_df = filter_medium_subgroup(valid_df, subgroup_name)

            if train_df.empty or valid_df.empty:
                continue

            caps = compute_clip_caps(train_df, cols=["Inventory_Pressure", "Stock_Cover_Months"], q=0.99)
            train_df = apply_clip_caps(train_df, caps)
            valid_df = apply_clip_caps(valid_df, caps)

            train_df = recompute_target(train_df)
            valid_df = recompute_target(valid_df)

            train_df = add_residual_target(train_df)
            valid_df = add_residual_target(valid_df)

            train_df = train_df.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()
            valid_df = valid_df.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()

            if train_df.empty or valid_df.empty:
                continue

            train_df, valid_df, _ = encode_itemcode(train_df, valid_df)

            assert_features_exist(train_df, feature_cols, where=f"{subgroup_name}_TUNE_TRAIN")
            assert_features_exist(valid_df, feature_cols, where=f"{subgroup_name}_TUNE_VALID")

            model = xgb.XGBRegressor(
                objective="reg:squarederror",
                eval_metric="rmse",
                random_state=42,
                tree_method="hist",
                n_jobs=-1,
                **params
            )

            w_train = recency_weights(train_df, yearly_boost=0.25).astype(float)
            w_train *= np.where(train_df["ABC_Class"] == 0, 2.5,
                       np.where(train_df["ABC_Class"] == 1, 1.2, 1.0))

            model.fit(
                sanitize(train_df[feature_cols]),
                train_df[MODEL_TARGET_COL],
                sample_weight=w_train,
                verbose=False
            )

            pred_residual = model.predict(sanitize(valid_df[feature_cols]))
            pred_final = np.clip(valid_df[BASELINE_COL].values + pred_residual, 0, None)

            scores.append(wmape(valid_df[ACTUAL_TARGET_COL].values, pred_final))

        return 999999.0 if len(scores) == 0 else np.mean(scores)

    study = optuna.create_study(direction="minimize", study_name=study_name)
    study.optimize(objective, n_trials=n_trials)

    return study.best_params, study


# 6) SUBGROUP FIXED-SPLIT EVALUATION
def train_eval_fixed_split_residual_medium_subgroup(
    full_data,
    feature_cols,
    best_params,
    subgroup_name,
    yearly_boost=0.25
):
    train_df = full_data[full_data["Year"] < 2025].copy()
    test_df = full_data[full_data["Year"] == 2025].copy()

    if train_df.empty or test_df.empty:
        raise ValueError("Need both train (<2025) and test (2025) data.")

    train_df = add_history_length_from_subset(train_df, train_df)
    test_df = add_history_length_from_subset(train_df, test_df)

    train_df = train_df[train_df["History_Segment"] == "MEDIUM"].copy()
    test_df = test_df[test_df["History_Segment"] == "MEDIUM"].copy()

    if train_df.empty or test_df.empty:
        raise ValueError("No MEDIUM rows available.")

    train_df, test_df, abc_map, promo_profile_df, medium_profile_df = prepare_medium_subgroup_frame_foldsafe(
        train_df, test_df
    )

    train_df = filter_medium_subgroup(train_df, subgroup_name)
    test_df = filter_medium_subgroup(test_df, subgroup_name)

    if train_df.empty or test_df.empty:
        raise ValueError(f"No usable rows for subgroup {subgroup_name}.")

    caps = compute_clip_caps(train_df, cols=["Inventory_Pressure", "Stock_Cover_Months"], q=0.99)
    train_df = apply_clip_caps(train_df, caps)
    test_df = apply_clip_caps(test_df, caps)

    train_df = recompute_target(train_df)
    test_df = recompute_target(test_df)

    train_df = add_residual_target(train_df)
    test_df = add_residual_target(test_df)

    train_df = train_df.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()
    test_df = test_df.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()

    if train_df.empty or test_df.empty:
        raise ValueError(f"No usable rows after target creation for subgroup {subgroup_name}.")

    train_df["ItemCode_Original"] = train_df["ItemCode"]
    test_df["ItemCode_Original"] = test_df["ItemCode"]

    train_df, test_df, itemcode_categories = encode_itemcode(train_df, test_df)

    assert_features_exist(train_df, feature_cols, where=f"{subgroup_name}_TRAIN")
    assert_features_exist(test_df, feature_cols, where=f"{subgroup_name}_TEST")

    model = xgb.XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        random_state=42,
        tree_method="hist",
        n_jobs=-1,
        **best_params
    )

    w_train = recency_weights(train_df, yearly_boost=yearly_boost).astype(float)
    w_train *= np.where(train_df["ABC_Class"] == 0, 2.5,
               np.where(train_df["ABC_Class"] == 1, 1.2, 1.0))

    model.fit(
        sanitize(train_df[feature_cols]),
        train_df[MODEL_TARGET_COL],
        sample_weight=w_train,
        verbose=False
    )

    test_df["Pred_Residual"] = model.predict(sanitize(test_df[feature_cols]))
    test_df["Pred"] = np.clip(test_df[BASELINE_COL] + test_df["Pred_Residual"], 0, None)

    metrics = evaluate_all_metrics(test_df[ACTUAL_TARGET_COL].values, test_df["Pred"].values)

    aux = {
        "abc_map": abc_map,
        "promo_profile_df": promo_profile_df,
        "medium_profile_df": medium_profile_df,
        "itemcode_categories": itemcode_categories,
        "clip_caps": caps,
        "subgroup_name": subgroup_name
    }

    return model, test_df, metrics, aux


# 7) SUBGROUP FEATURE PRUNING
def iterative_feature_prune_residual_medium_subgroup(
    full_data,
    start_features,
    best_params,
    subgroup_name,
    drop_k=1,
    min_features=15,
    max_rounds=10,
    tolerance=0.10,
    n_repeats=5
):
    history = []
    features = start_features.copy()

    model, test_df, m, aux = train_eval_fixed_split_residual_medium_subgroup(
        full_data=full_data,
        feature_cols=features,
        best_params=best_params,
        subgroup_name=subgroup_name
    )

    best_wmape = m["WMAPE"]
    last_accepted_state = (features.copy(), model, test_df.copy(), m.copy(), aux.copy())

    for r in range(1, max_rounds + 1):
        if len(features) <= min_features:
            break

        imp = permutation_rank(
            model=model,
            test_df=test_df,
            feature_cols=features,
            target_col=MODEL_TARGET_COL,
            n_repeats=n_repeats
        )

        protected = {
            "ItemCode",
            "ABC_Class",
            "Lag1",
            "Lag2",
            "Lag3",
            "Rolling3M_Mean",
            "Rolling3M_Std",
            "Month_Sin",
            "Month_Cos",
            "Bonus_Flag",
            "Expected_Bonus_Month",
            "Supply_Constraint_Flag",
            "Medium_SKU_Type_Encoded"
        }

        drop_candidates = [
            f for f in imp.sort_values("perm_importance").feature.tolist()
            if f not in protected
        ]

        to_drop = drop_candidates[:drop_k]

        if not to_drop:
            break

        new_features = [f for f in features if f not in to_drop]

        new_model, new_test_df, new_m, new_aux = train_eval_fixed_split_residual_medium_subgroup(
            full_data=full_data,
            feature_cols=new_features,
            best_params=best_params,
            subgroup_name=subgroup_name
        )

        history.append({
            "round": r,
            "subgroup": subgroup_name,
            "dropped": to_drop,
            "n_features": len(new_features),
            **new_m
        })

        if new_m["WMAPE"] <= best_wmape + tolerance:
            features = new_features
            model, test_df = new_model, new_test_df
            best_wmape = min(best_wmape, new_m["WMAPE"])
            last_accepted_state = (features.copy(), model, test_df.copy(), new_m.copy(), new_aux.copy())
        else:
            break

    results_df = pd.DataFrame(history)

    return (
        last_accepted_state[0],
        last_accepted_state[1],
        last_accepted_state[2],
        last_accepted_state[3],
        last_accepted_state[4],
        results_df
    )


# 8) EVALUATION MODEL
def train_single_evaluation_2025_medium_subgroup_residual(
    full_data,
    feature_cols,
    best_params,
    subgroup_name
):
    print(f"\n========== RESIDUAL XGBOOST EVALUATION MODEL → TEST ON 2025 ({subgroup_name}) ==========")

    model, test_df, metrics, aux = train_eval_fixed_split_residual_medium_subgroup(
        full_data=full_data,
        feature_cols=feature_cols,
        best_params=best_params,
        subgroup_name=subgroup_name
    )

    artifacts = {
        "model": model,
        "feature_cols": feature_cols,
        "best_params": best_params,
        "itemcode_categories": aux["itemcode_categories"],
        "abc_map": aux["abc_map"],
        "clip_caps": aux["clip_caps"],
        "promo_profile_df": aux["promo_profile_df"],
        "medium_profile_df": aux["medium_profile_df"],
        "target_mode": "residual",
        "baseline_col": BASELINE_COL,
        "actual_target_col": ACTUAL_TARGET_COL,
        "model_target_col": MODEL_TARGET_COL,
        "segment": "MEDIUM",
        "subgroup_name": subgroup_name
    }

    return artifacts, test_df, metrics


# 9) DEPLOYMENT MODEL
def train_single_deployment_model_medium_subgroup_residual(
    full_data,
    feature_cols,
    best_params,
    subgroup_name
):
    print(f"\n========== RESIDUAL XGBOOST DEPLOYMENT MODEL → TRAIN ON ALL COMPLETE DATA ({subgroup_name}) ==========")

    deploy_df = full_data.copy().sort_values(["ItemCode", "Year", "Month_Number"])
    deploy_df = add_history_length_from_subset(deploy_df, deploy_df)
    deploy_df = deploy_df[deploy_df["History_Segment"] == "MEDIUM"].copy()

    if deploy_df.empty:
        raise ValueError("No MEDIUM rows available for deployment training.")

    bonus_pattern_df = detect_recurring_bonus_skus(deploy_df)[[
        "ItemCode",
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ]].copy()

    deploy_df = deploy_df.drop(columns=[
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ], errors="ignore")

    deploy_df = deploy_df.merge(bonus_pattern_df, on="ItemCode", how="left")

    for c in ["Recurring_Bonus_SKU", "Bonus_Cycle_Length", "Avg_Bonus_Gap", "Bonus_Frequency_All"]:
        deploy_df[c] = deploy_df[c].fillna(0)
    deploy_df["Avg_Bonus_Uplift"] = deploy_df["Avg_Bonus_Uplift"].fillna(1.0)

    deploy_df, _ = apply_sku_cap(deploy_df.copy(), deploy_df.copy())

    sku_total = deploy_df.groupby("ItemCode")["Clean_Demand"].sum().sort_values(ascending=False)
    cum_pct = sku_total.cumsum() / sku_total.sum()
    abc_series = pd.cut(cum_pct, bins=[0, 0.7, 0.9, 1.0], labels=[0, 1, 2])
    abc_map = abc_series.to_dict()
    deploy_df["ABC_Class"] = deploy_df["ItemCode"].map(abc_map).fillna(2)

    promo_profile_df = build_promo_profile(deploy_df)
    deploy_df = merge_promo_profile(deploy_df, promo_profile_df)

    medium_profile_df = build_medium_sku_profile(deploy_df)
    deploy_df = merge_medium_sku_profile(deploy_df, medium_profile_df)

    deploy_df = filter_medium_subgroup(deploy_df, subgroup_name)
    if deploy_df.empty:
        raise ValueError(f"No rows available for deployment subgroup {subgroup_name}.")

    deploy_df = add_bonus_cycle_features(deploy_df)
    deploy_df = rebuild_time_features(deploy_df)

    caps = compute_clip_caps(deploy_df, cols=["Inventory_Pressure", "Stock_Cover_Months"], q=0.99)
    deploy_df = apply_clip_caps(deploy_df, caps)

    deploy_df = recompute_target(deploy_df)
    deploy_df = add_residual_target(deploy_df)
    deploy_df = deploy_df.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()

    if deploy_df.empty:
        raise ValueError(f"No usable deployment rows after target creation for {subgroup_name}.")

    deploy_df["ItemCode_Original"] = deploy_df["ItemCode"]
    deploy_df, _, itemcode_categories = encode_itemcode(deploy_df, deploy_df)

    assert_features_exist(deploy_df, feature_cols, where=f"{subgroup_name}_DEPLOY_TRAIN")

    model = xgb.XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        random_state=42,
        tree_method="hist",
        n_jobs=-1,
        **best_params
    )

    w_train = recency_weights(deploy_df, yearly_boost=0.25).astype(float)
    w_train *= np.where(deploy_df["ABC_Class"] == 0, 2.5,
               np.where(deploy_df["ABC_Class"] == 1, 1.2, 1.0))

    model.fit(
        sanitize(deploy_df[feature_cols]),
        deploy_df[MODEL_TARGET_COL],
        sample_weight=w_train,
        verbose=False
    )

    artifacts = {
        "model": model,
        "feature_cols": feature_cols,
        "best_params": best_params,
        "itemcode_categories": itemcode_categories,
        "abc_map": abc_map,
        "clip_caps": caps,
        "promo_profile_df": promo_profile_df,
        "medium_profile_df": medium_profile_df,
        "target_mode": "residual",
        "baseline_col": BASELINE_COL,
        "actual_target_col": ACTUAL_TARGET_COL,
        "model_target_col": MODEL_TARGET_COL,
        "segment": "MEDIUM",
        "subgroup_name": subgroup_name
    }

    return artifacts, deploy_df


# 10) TRAIN PROMO_HEAVY SUBGROUP
promo_best_params, promo_study = tune_residual_xgb_medium_subgroup(
    full_data=Data,
    feature_cols=MEDIUM_SUBGROUP_FEATURE_COLS,
    subgroup_name="PROMO_HEAVY",
    n_trials=30,
    study_name="residual_xgb_medium_promo_heavy"
)
print("PROMO_HEAVY best params:", promo_best_params)

promo_best_feats, _, _, promo_best_metrics, promo_aux, promo_prune_log = iterative_feature_prune_residual_medium_subgroup(
    full_data=Data,
    start_features=MEDIUM_SUBGROUP_FEATURE_COLS,
    best_params=promo_best_params,
    subgroup_name="PROMO_HEAVY",
    drop_k=1,
    min_features=15,
    max_rounds=10,
    tolerance=0.10,
    n_repeats=5
)

print("\n===== PROMO_HEAVY BEST METRICS AFTER FEATURE PRUNING =====")
print(promo_best_metrics)

print("\n===== PROMO_HEAVY BEST FEATURES =====")
print(promo_best_feats)

print("\n===== PROMO_HEAVY FEATURE PRUNE LOG =====")
print(promo_prune_log)

promo_eval_artifacts, promo_test_2025, promo_eval_metrics = train_single_evaluation_2025_medium_subgroup_residual(
    full_data=Data,
    feature_cols=promo_best_feats,
    best_params=promo_best_params,
    subgroup_name="PROMO_HEAVY"
)
print("\n===== PROMO_HEAVY EVALUATION METRICS =====")
print(promo_eval_metrics)

promo_deploy_artifacts, promo_deploy_train_df = train_single_deployment_model_medium_subgroup_residual(
    full_data=Data,
    feature_cols=promo_best_feats,
    best_params=promo_best_params,
    subgroup_name="PROMO_HEAVY"
)


# 11) TRAIN STABLE SUBGROUP
stable_best_params, stable_study = tune_residual_xgb_medium_subgroup(
    full_data=Data,
    feature_cols=MEDIUM_SUBGROUP_FEATURE_COLS,
    subgroup_name="STABLE",
    n_trials=30,
    study_name="residual_xgb_medium_stable"
)
print("STABLE best params:", stable_best_params)

stable_best_feats, _, _, stable_best_metrics, stable_aux, stable_prune_log = iterative_feature_prune_residual_medium_subgroup(
    full_data=Data,
    start_features=MEDIUM_SUBGROUP_FEATURE_COLS,
    best_params=stable_best_params,
    subgroup_name="STABLE",
    drop_k=1,
    min_features=15,
    max_rounds=10,
    tolerance=0.10,
    n_repeats=5
)

print("\n===== STABLE BEST METRICS AFTER FEATURE PRUNING =====")
print(stable_best_metrics)

print("\n===== STABLE BEST FEATURES =====")
print(stable_best_feats)

print("\n===== STABLE FEATURE PRUNE LOG =====")
print(stable_prune_log)

stable_eval_artifacts, stable_test_2025, stable_eval_metrics = train_single_evaluation_2025_medium_subgroup_residual(
    full_data=Data,
    feature_cols=stable_best_feats,
    best_params=stable_best_params,
    subgroup_name="STABLE"
)
print("\n===== STABLE EVALUATION METRICS =====")
print(stable_eval_metrics)

stable_deploy_artifacts, stable_deploy_train_df = train_single_deployment_model_medium_subgroup_residual(
    full_data=Data,
    feature_cols=stable_best_feats,
    best_params=stable_best_params,
    subgroup_name="STABLE"
)


# 12) COMBINED MEDIUM V1 RESULT
medium_subgroup_test_2025 = pd.concat(
    [promo_test_2025.copy(), stable_test_2025.copy()],
    ignore_index=True
)

medium_subgroup_metrics = evaluate_all_metrics(
    medium_subgroup_test_2025[ACTUAL_TARGET_COL].values,
    medium_subgroup_test_2025["Pred"].values
)

print("\n===== COMBINED MEDIUM_SUBGROUP_V1 METRICS =====")
print(medium_subgroup_metrics)


# 13) SAVE ARTIFACTS
joblib.dump(promo_eval_artifacts, "xgb_medium_promo_heavy_eval_artifacts_residual.pkl")
joblib.dump(promo_deploy_artifacts, "xgb_medium_promo_heavy_deploy_artifacts_residual.pkl")

joblib.dump(stable_eval_artifacts, "xgb_medium_stable_eval_artifacts_residual.pkl")
joblib.dump(stable_deploy_artifacts, "xgb_medium_stable_deploy_artifacts_residual.pkl")

print("\nSaved:")
print(" - xgb_medium_promo_heavy_eval_artifacts_residual.pkl")
print(" - xgb_medium_promo_heavy_deploy_artifacts_residual.pkl")
print(" - xgb_medium_stable_eval_artifacts_residual.pkl")
print(" - xgb_medium_stable_deploy_artifacts_residual.pkl")

### Short History Segmnet

#### Model for SHORT HISTORU SKUs

In [ ]:
def build_short_sku_profile(train_df):
    train_df = train_df.copy().sort_values(["ItemCode", "Year", "Month_Number"])
    out = []

    for item, g in train_df.groupby("ItemCode"):
        g = g.copy()

        hist_len = len(g)
        mean_demand = float(g["Clean_Demand"].mean()) if hist_len > 0 else 0.0
        zero_rate = float((g["Clean_Demand"] == 0).mean()) if hist_len > 0 else 0.0
        bonus_freq = float(g["Bonus_Flag"].mean()) if "Bonus_Flag" in g.columns else 0.0
        supply_rate = float(g["Supply_Constraint_Flag"].mean()) if "Supply_Constraint_Flag" in g.columns else 0.0

        total_demand = float(g["Clean_Demand"].sum())
        bonus_demand = float(g.loc[g["Bonus_Flag"] == 1, "Clean_Demand"].sum()) if total_demand > 0 else 0.0
        bonus_share = 0.0 if total_demand <= 0 else bonus_demand / total_demand

        if bonus_freq >= 0.25 or bonus_share >= 0.40:
            short_type = "SHORT_PROMO"
        else:
            short_type = "SHORT_NORMAL"

        out.append({
            "ItemCode": item,
            "Short_SKU_Type": short_type,
            "Short_History_Length": hist_len,
            "Short_Mean_Demand": mean_demand,
            "Short_ZeroRate": zero_rate,
            "Short_Bonus_Frequency": bonus_freq,
            "Short_Bonus_Demand_Share": bonus_share,
            "Short_Supply_Rate": supply_rate
        })

    return pd.DataFrame(out)

def merge_short_sku_profile(df, profile_df):
    df = df.copy()

    keep_cols = [
        "ItemCode",
        "Short_SKU_Type",
        "Short_History_Length",
        "Short_Mean_Demand",
        "Short_ZeroRate",
        "Short_Bonus_Frequency",
        "Short_Bonus_Demand_Share",
        "Short_Supply_Rate"
    ]

    df = df.drop(columns=[c for c in keep_cols if c != "ItemCode"], errors="ignore")
    df = df.merge(profile_df[keep_cols], on="ItemCode", how="left")

    df["Short_SKU_Type"] = df["Short_SKU_Type"].fillna("SHORT_NORMAL")

    for c in [
        "Short_History_Length",
        "Short_Mean_Demand",
        "Short_ZeroRate",
        "Short_Bonus_Frequency",
        "Short_Bonus_Demand_Share",
        "Short_Supply_Rate"
    ]:
        df[c] = df[c].fillna(0)

    return df

def filter_short_subgroup(df, subgroup_name):
    df = df.copy()

    if subgroup_name == "SHORT_PROMO":
        return df[df["Short_SKU_Type"] == "SHORT_PROMO"].copy()

    if subgroup_name == "SHORT_NORMAL":
        return df[df["Short_SKU_Type"] == "SHORT_NORMAL"].copy()

    raise ValueError(f"Unknown short subgroup: {subgroup_name}")

def prepare_short_frame_foldsafe(train_df, valid_df):
    train_df = train_df.copy().sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)
    valid_df = valid_df.copy().sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)

    train_df, valid_df, _ = apply_recurring_bonus_features(train_df, valid_df)
    train_df, valid_df, abc_map = apply_fold_adjustments(train_df, valid_df)

    promo_profile_df = build_promo_profile(train_df)
    train_df = merge_promo_profile(train_df, promo_profile_df)
    valid_df = merge_promo_profile(valid_df, promo_profile_df)

    short_profile_df = build_short_sku_profile(train_df)
    train_df = merge_short_sku_profile(train_df, short_profile_df)
    valid_df = merge_short_sku_profile(valid_df, short_profile_df)

    combined = pd.concat(
        [train_df.assign(_is_train=1), valid_df.assign(_is_train=0)],
        ignore_index=True
    )
    combined = combined.sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)
    combined = rebuild_time_features(combined)

    train_out = combined[combined["_is_train"] == 1].drop(columns=["_is_train"]).copy()
    valid_out = combined[combined["_is_train"] == 0].drop(columns=["_is_train"]).copy()

    return train_out, valid_out, abc_map, promo_profile_df, short_profile_df

def short_rule_predict(row):
    lag1 = float(row.get("Lag1", 0) or 0)
    lag2 = float(row.get("Lag2", 0) or 0)
    rolling3 = float(row.get("Rolling3M_Mean", 0) or 0)
    sku_mean = float(row.get("SKU_Mean_Demand", 0) or 0)

    last_bonus_demand = float(row.get("Last_Bonus_Demand", 0) or 0)
    avg_bonus_uplift = float(row.get("Avg_Bonus_Uplift", 1.0) or 1.0)

    bonus_flag = int(row.get("Bonus_Flag", 0) or 0)
    expected_bonus = int(row.get("Expected_Bonus_NextMonth", 0) or 0)
    supply_flag = int(row.get("Supply_Constraint_Flag", 0) or 0)

    primary_stock = float(row.get("Available_Primary_Inventory_Qty", 0) or 0)
    distributor_stock = float(row.get("Distributor_Inventory_Qty", 0) or 0)

    short_type = row.get("Short_SKU_Type", "SHORT_NORMAL")
    hist_len = int(row.get("History_Length", 0) or 0)

    anchors = [x for x in [lag1, lag2, rolling3, sku_mean] if x > 0]

    if len(anchors) == 0:
        pred = 0.0
    elif hist_len <= 2:
        pred = float(np.mean(anchors))
    else:
        pred = float(np.median(anchors))

    if short_type == "SHORT_PROMO":
        if expected_bonus == 1 and last_bonus_demand > 0:
            pred = 0.60 * pred + 0.40 * last_bonus_demand
        elif bonus_flag == 1:
            pred = pred * min(max(avg_bonus_uplift, 1.0), 1.8)

    if supply_flag == 1:
        pred = min(pred, max(lag1, rolling3, 0))

    if (primary_stock + distributor_stock) <= 0:
        pred *= 0.85

    return max(pred, 0.0)

def evaluate_short_rule_model(full_data, subgroup_name=None):
    train_df = full_data[full_data["Year"] < 2025].copy()
    test_df = full_data[full_data["Year"] == 2025].copy()

    if train_df.empty or test_df.empty:
        raise ValueError("Need both train (<2025) and test (2025) data.")

    train_df = add_history_length_from_subset(train_df, train_df)
    test_df = add_history_length_from_subset(train_df, test_df)

    train_df = train_df[train_df["History_Segment"] == "SHORT"].copy()
    test_df = test_df[test_df["History_Segment"] == "SHORT"].copy()

    if train_df.empty or test_df.empty:
        raise ValueError("No SHORT rows available.")

    train_df, test_df, abc_map, promo_profile_df, short_profile_df = prepare_short_frame_foldsafe(train_df, test_df)

    if subgroup_name is not None:
        train_df = filter_short_subgroup(train_df, subgroup_name)
        test_df = filter_short_subgroup(test_df, subgroup_name)

    caps = compute_clip_caps(train_df, cols=["Inventory_Pressure", "Stock_Cover_Months"], q=0.99)
    train_df = apply_clip_caps(train_df, caps)
    test_df = apply_clip_caps(test_df, caps)

    train_df = recompute_target(train_df)
    test_df = recompute_target(test_df)

    test_df = test_df.dropna(subset=[ACTUAL_TARGET_COL]).copy()

    if test_df.empty:
        raise ValueError("No usable SHORT rows after target creation.")

    test_df["ItemCode_Original"] = test_df["ItemCode"].astype(str)
    test_df["Pred"] = test_df.apply(short_rule_predict, axis=1)

    metrics = evaluate_all_metrics(test_df[ACTUAL_TARGET_COL].values, test_df["Pred"].values)

    artifacts = {
        "model": None,
        "feature_cols": SHORT_FEATURE_COLS,
        "best_params": None,
        "itemcode_categories": None,
        "abc_map": abc_map,
        "clip_caps": caps,
        "promo_profile_df": promo_profile_df,
        "short_profile_df": short_profile_df,
        "target_mode": "rule_based",
        "baseline_col": None,
        "actual_target_col": ACTUAL_TARGET_COL,
        "model_target_col": None,
        "segment": "SHORT",
        "subgroup_name": subgroup_name if subgroup_name is not None else "ALL_SHORT",
        "rule_name": "short_rule_v1"
    }

    return artifacts, test_df, metrics

def prepare_short_rule_deployment(full_data, subgroup_name=None):
    deploy_df = full_data.copy().sort_values(["ItemCode", "Year", "Month_Number"])
    deploy_df = add_history_length_from_subset(deploy_df, deploy_df)
    deploy_df = deploy_df[deploy_df["History_Segment"] == "SHORT"].copy()

    if deploy_df.empty:
        artifacts = {
            "model": None,
            "feature_cols": SHORT_FEATURE_COLS,
            "best_params": None,
            "itemcode_categories": None,
            "abc_map": {},
            "clip_caps": {},
            "promo_profile_df": pd.DataFrame(),
            "short_profile_df": pd.DataFrame(),
            "target_mode": "rule_based",
            "baseline_col": None,
            "actual_target_col": ACTUAL_TARGET_COL,
            "model_target_col": None,
            "segment": "SHORT",
            "subgroup_name": subgroup_name if subgroup_name is not None else "ALL_SHORT",
            "rule_name": "short_rule_v1"
        }
        return artifacts, pd.DataFrame()

    bonus_pattern_df = detect_recurring_bonus_skus(deploy_df)[[
        "ItemCode",
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ]].copy()

    deploy_df = deploy_df.drop(columns=[
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ], errors="ignore")

    deploy_df = deploy_df.merge(bonus_pattern_df, on="ItemCode", how="left")

    for c in ["Recurring_Bonus_SKU", "Bonus_Cycle_Length", "Avg_Bonus_Gap", "Bonus_Frequency_All"]:
        deploy_df[c] = deploy_df[c].fillna(0)
    deploy_df["Avg_Bonus_Uplift"] = deploy_df["Avg_Bonus_Uplift"].fillna(1.0)

    deploy_df, _ = apply_sku_cap(deploy_df.copy(), deploy_df.copy())

    sku_total = deploy_df.groupby("ItemCode")["Clean_Demand"].sum().sort_values(ascending=False)
    cum_pct = sku_total.cumsum() / sku_total.sum()
    abc_series = pd.cut(cum_pct, bins=[0, 0.7, 0.9, 1.0], labels=[0, 1, 2])
    abc_map = abc_series.to_dict()
    deploy_df["ABC_Class"] = deploy_df["ItemCode"].map(abc_map).fillna(2)

    promo_profile_df = build_promo_profile(deploy_df)
    deploy_df = merge_promo_profile(deploy_df, promo_profile_df)

    short_profile_df = build_short_sku_profile(deploy_df)
    deploy_df = merge_short_sku_profile(deploy_df, short_profile_df)

    if subgroup_name is not None:
        deploy_df = filter_short_subgroup(deploy_df, subgroup_name)

    deploy_df = add_bonus_cycle_features(deploy_df)
    deploy_df = rebuild_time_features(deploy_df)

    caps = compute_clip_caps(deploy_df, cols=["Inventory_Pressure", "Stock_Cover_Months"], q=0.99)
    deploy_df = apply_clip_caps(deploy_df, caps)

    artifacts = {
        "model": None,
        "feature_cols": SHORT_FEATURE_COLS,
        "best_params": None,
        "itemcode_categories": None,
        "abc_map": abc_map,
        "clip_caps": caps,
        "promo_profile_df": promo_profile_df,
        "short_profile_df": short_profile_df,
        "target_mode": "rule_based",
        "baseline_col": None,
        "actual_target_col": ACTUAL_TARGET_COL,
        "model_target_col": None,
        "segment": "SHORT",
        "subgroup_name": subgroup_name if subgroup_name is not None else "ALL_SHORT",
        "rule_name": "short_rule_v1"
    }

    return artifacts, deploy_df

# =========================
# SHORT RULE MODEL EVALUATION
short_eval_artifacts, short_test_2025, short_eval_metrics = evaluate_short_rule_model(
    full_data=Data,
    subgroup_name=None
)

print("\n===== SHORT RULE-BASED EVALUATION METRICS =====")
print(short_eval_metrics)

# =========================
# SHORT RULE MODEL DEPLOYMENT
short_deploy_artifacts, short_deploy_train_df = prepare_short_rule_deployment(
    full_data=Data,
    subgroup_name=None
)

short_promo_eval_artifacts, short_promo_test_2025, short_promo_eval_metrics = evaluate_short_rule_model(
    full_data=Data,
    subgroup_name="SHORT_PROMO"
)
print("\n===== SHORT_PROMO RULE METRICS =====")
print(short_promo_eval_metrics)

short_normal_eval_artifacts, short_normal_test_2025, short_normal_eval_metrics = evaluate_short_rule_model(
    full_data=Data,
    subgroup_name="SHORT_NORMAL"
)
print("\n===== SHORT_NORMAL RULE METRICS =====")
print(short_normal_eval_metrics)

print("SHORT deployment rows:", len(short_deploy_train_df))

# =========================
# SAVE SHORT ARTIFACTS
joblib.dump(short_eval_artifacts, "xgb_short_eval_artifacts_rule.pkl")
joblib.dump(short_deploy_artifacts, "xgb_short_deploy_artifacts_rule.pkl")

print("\nSaved:")
print(" - xgb_short_eval_artifacts_rule.pkl")
print(" - xgb_short_deploy_artifacts_rule.pkl")

In [ ]:
# =========================
# FINAL COMBINED RESULT: LONG + MEDIUM + SHORT
# =========================

for name, df in {
    "long_test_2025": long_test_2025,
    "medium_subgroup_test_2025": medium_subgroup_test_2025,
    "short_test_2025": short_test_2025
}.items():
    if df.empty:
        raise ValueError(f"{name} is empty. Cannot compute final metrics.")

final_test_2025 = pd.concat(
    [
        long_test_2025.copy(),
        medium_subgroup_test_2025.copy(),
        short_test_2025.copy()
    ],
    ignore_index=True
)

final_metrics = evaluate_all_metrics(
    final_test_2025[ACTUAL_TARGET_COL].values,
    final_test_2025["Pred"].values
)

print("\n===== FINAL COMBINED METRICS: LONG + MEDIUM + SHORT =====")
print(final_metrics)

# Inference

In [ ]:
# ============================================================
# FINAL INFERENCE LAYER
# Supports:
#   LONG   -> XGBoost residual model
#   MEDIUM -> subgroup routing (PROMO_HEAVY / STABLE)
#   SHORT  -> rule-based forecast
# Output:
#   pharma_forecast_feb.xlsx
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1) BASIC SEGMENT ROUTING
# ============================================================
def get_sku_history_length(raw_data, sku_code):
    sku_code = str(sku_code)
    g = raw_data[raw_data["ItemCode"].astype(str) == sku_code].copy()
    if g.empty:
        return 0
    return g[["Year", "Month_Number"]].drop_duplicates().shape[0]


def choose_model_by_history(raw_data, sku_code):
    hist_len = get_sku_history_length(raw_data, sku_code)

    if hist_len >= 18:
        return "LONG"
    elif hist_len >= 6:
        return "MEDIUM"
    else:
        return "SHORT"


# ============================================================
# 2) SHARED HELPERS
# ============================================================
def get_next_period_from_history(df, sku_code):
    sku_code = str(sku_code)

    work = df.copy()
    work["ItemCode_Original"] = work["ItemCode"].astype(str)

    sku_hist = work[work["ItemCode_Original"] == sku_code].copy()
    if sku_hist.empty:
        return None, None, None

    last_row = sku_hist.sort_values(["Year", "Month_Number"]).iloc[-1:].copy()

    next_month = int(last_row["Month_Number"].iloc[0]) + 1
    next_year = int(last_row["Year"].iloc[0])

    if next_month > 12:
        next_month = 1
        next_year += 1

    return last_row, next_year, next_month


def infer_expected_bonus_flag_residual(sku_code, prepared_df):
    sku_code = str(sku_code)

    work = prepared_df.copy()
    if "ItemCode_Original" not in work.columns:
        work["ItemCode_Original"] = work["ItemCode"].astype(str)

    g = work[work["ItemCode_Original"] == sku_code].copy()
    g = g.sort_values(["Year", "Month_Number"])

    if g.empty:
        return 0

    recent_bonus_rate = g["Bonus_Flag"].tail(12).mean() if "Bonus_Flag" in g.columns else 0
    recurring = g["Recurring_Bonus_SKU"].iloc[-1] if "Recurring_Bonus_SKU" in g.columns else 0
    cycle_len = g["Bonus_Cycle_Length"].iloc[-1] if "Bonus_Cycle_Length" in g.columns else 0
    months_since = g["Months_Since_Last_Bonus"].iloc[-1] if "Months_Since_Last_Bonus" in g.columns else 999

    if recurring == 1 and cycle_len > 0 and abs((months_since + 1) - cycle_len) <= 1:
        return 1

    if recurring == 1 and recent_bonus_rate >= 0.25:
        return 1

    return 0


def get_last_nonzero(series):
    s = pd.Series(series).dropna()
    s = s[s > 0]
    if len(s) == 0:
        return 0.0
    return float(s.iloc[-1])


# ============================================================
# 3) LONG / MEDIUM SHARED XGB PREP
# ============================================================
def prepare_residual_forecast_frame(artifacts, raw_data):
    abc_map = artifacts["abc_map"]
    caps = artifacts["clip_caps"]
    promo_profile_df = artifacts["promo_profile_df"]
    itemcode_categories = artifacts["itemcode_categories"]

    df = raw_data.copy().sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)
    df["ItemCode_Original"] = df["ItemCode"].astype(str)

    bonus_pattern_df = detect_recurring_bonus_skus(df)[[
        "ItemCode",
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ]].copy()

    df = df.drop(columns=[
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ], errors="ignore")

    df = df.merge(bonus_pattern_df, on="ItemCode", how="left")

    for c in ["Recurring_Bonus_SKU", "Bonus_Cycle_Length", "Avg_Bonus_Gap", "Bonus_Frequency_All"]:
        df[c] = df[c].fillna(0)

    df["Avg_Bonus_Uplift"] = df["Avg_Bonus_Uplift"].fillna(1.0)
    df["ABC_Class"] = df["ItemCode"].map(abc_map).fillna(2)

    df = merge_promo_profile(df, promo_profile_df)

    # for MEDIUM subgroup artifacts
    if artifacts.get("segment") == "MEDIUM" and "medium_profile_df" in artifacts:
        df = merge_medium_sku_profile(df, artifacts["medium_profile_df"])

    df = add_bonus_cycle_features(df)
    df = rebuild_time_features(df)
    df = apply_clip_caps(df, caps)

    df = recompute_target(df)
    df = add_residual_target(df)

    cat_to_code = {str(k): i for i, k in enumerate(itemcode_categories)}
    unk_code = len(cat_to_code)

    df["ItemCode_Encoded"] = df["ItemCode"].astype(str).map(cat_to_code).fillna(unk_code).astype(int)

    return df


# ============================================================
# 4) LONG / MEDIUM FORECAST ADJUSTMENTS
# ============================================================
def apply_promo_aware_adjustment_residual(row, raw_final_pred):
    raw_final_pred = max(float(raw_final_pred), 0.0)

    promo_profile = row.get("Promo_Profile", "NORMAL")
    next_bonus = int(row.get("Bonus_Flag", 0))

    rolling_mean = float(row.get("Rolling3M_Mean", 0) or 0)
    lag1 = float(row.get("Lag1", 0) or 0)
    last_bonus_demand = float(row.get("Last_Bonus_Demand", 0) or 0)

    bonus_avg = float(row.get("Bonus_Avg_Demand", 0) or 0)
    non_bonus_avg = float(row.get("NonBonus_Avg_Demand", 0) or 0)
    uplift_ratio = float(row.get("Bonus_Uplift_Ratio_Profile", 1.0) or 1.0)

    normal_anchor = max(rolling_mean, lag1, non_bonus_avg, 0)
    bonus_anchor = max(bonus_avg, last_bonus_demand, rolling_mean, lag1, 0)

    if promo_profile == "NORMAL":
        cap_anchor = max(rolling_mean, lag1, non_bonus_avg, 0)
        if cap_anchor > 0:
            raw_final_pred = min(raw_final_pred, cap_anchor * 2.5)
        return raw_final_pred

    if promo_profile == "PROMO_INFLUENCED":
        if next_bonus == 1:
            bounded_uplift = min(max(uplift_ratio, 1.0), 1.60)
            uplift_pred = raw_final_pred * bounded_uplift

            if bonus_anchor > 0:
                adjusted = 0.55 * uplift_pred + 0.45 * bonus_anchor
            else:
                adjusted = uplift_pred

            floor_val = max(raw_final_pred, normal_anchor * 1.05)
            cap_val = max(floor_val, bonus_anchor * 1.35 if bonus_anchor > 0 else uplift_pred)

            return min(max(adjusted, floor_val), cap_val)
        else:
            if non_bonus_avg > 0:
                adjusted = 0.70 * raw_final_pred + 0.30 * non_bonus_avg
                return max(adjusted, 0)
            return raw_final_pred

    if promo_profile == "PURE_PROMO":
        if next_bonus == 1:
            if bonus_avg > 0 and last_bonus_demand > 0:
                adjusted = 0.60 * bonus_avg + 0.40 * last_bonus_demand
            elif bonus_avg > 0:
                adjusted = bonus_avg
            else:
                adjusted = raw_final_pred

            if bonus_anchor > 0:
                adjusted = min(adjusted, bonus_anchor * 1.20)

            return max(adjusted, raw_final_pred * 0.90)
        else:
            if non_bonus_avg > 0:
                return min(raw_final_pred, max(non_bonus_avg, normal_anchor))
            return min(raw_final_pred, max(rolling_mean * 0.60, lag1 * 0.60, 0))

    return raw_final_pred


def apply_final_forecast_guardrails_residual(row, pred):
    pred = max(float(pred), 0.0)

    rolling_mean = float(row.get("Rolling3M_Mean", 0) or 0)
    lag1 = float(row.get("Lag1", 0) or 0)
    sku_mean = float(row.get("SKU_Mean_Demand", 0) or 0)
    non_bonus_avg = float(row.get("NonBonus_Avg_Demand", 0) or 0)
    promo_profile = row.get("Promo_Profile", "NORMAL")
    next_bonus = int(row.get("Bonus_Flag", 0) or 0)

    base_anchor = max(rolling_mean, lag1, sku_mean, non_bonus_avg, 0)

    if base_anchor < 100:
        pred = min(pred, max(base_anchor * 1.8, 30))
    elif base_anchor < 500:
        pred = min(pred, base_anchor * 2.2)
    elif promo_profile == "NORMAL" or next_bonus == 0:
        pred = min(pred, base_anchor * 2.5)
    else:
        pred = min(pred, base_anchor * 3.0)

    return max(pred, 0.0)


# ============================================================
# 4B) PRODUCTION CALIBRATION
# learned from historical backtest behavior, not current actuals
# ============================================================
def apply_production_calibration(row, pred):
    pred = max(float(pred), 0.0)

    segment = row.get("Segment_For_Calibration", "")
    subgroup = row.get("Medium_Subgroup", "")
    abc_class = str(row.get("ABC_Class", "2"))

    rolling_mean = float(row.get("Rolling3M_Mean", 0) or 0)
    lag1 = float(row.get("Lag1", 0) or 0)
    sku_mean = float(row.get("SKU_Mean_Demand", 0) or 0)
    promo_profile = row.get("Promo_Profile", "NORMAL")
    expected_bonus = int(row.get("Bonus_Flag", 0) or 0)

    anchor = max(rolling_mean, lag1, sku_mean, 0.0)

    # --- base segment calibration ---
    # tune these later using backtest summaries
    if segment == "LONG":
        factor = 1.03
    elif segment == "MEDIUM":
        factor = 1.05
    elif segment == "SHORT":
        factor = 1.08
    else:
        factor = 1.00

    # --- subgroup adjustments ---
    if subgroup == "PROMO_HEAVY":
        factor *= 1.06
    elif subgroup == "STABLE":
        factor *= 1.01

    # --- ABC adjustments ---
    if abc_class == "0":      # A
        factor *= 1.02
    elif abc_class == "1":    # B
        factor *= 1.01
    else:                     # C
        factor *= 0.99

    # --- promo-aware slight uplift ---
    if promo_profile in ["PROMO_INFLUENCED", "PURE_PROMO"] and expected_bonus == 1:
        factor *= 1.04

    calibrated = pred * factor

    # --- final safety bounds after calibration ---
    if anchor > 0:
        if segment == "LONG":
            calibrated = min(calibrated, anchor * 3.0)
        elif segment == "MEDIUM":
            calibrated = min(calibrated, anchor * 2.8)
        else:
            calibrated = min(calibrated, anchor * 2.5)

    return max(calibrated, 0.0)

# ============================================================
# 5) MEDIUM SUBGROUP ROUTING
# ============================================================
def choose_medium_subgroup_from_prepared(prepared_df, sku_code):
    sku_code = str(sku_code)

    work = prepared_df.copy()
    if "ItemCode_Original" not in work.columns:
        work["ItemCode_Original"] = work["ItemCode"].astype(str)

    g = work[work["ItemCode_Original"] == sku_code].copy()
    g = g.sort_values(["Year", "Month_Number"])

    if g.empty:
        return "STABLE"

    subgroup = g["Medium_Subgroup"].iloc[-1] if "Medium_Subgroup" in g.columns else "STABLE"
    if subgroup not in ["PROMO_HEAVY", "STABLE"]:
        subgroup = "STABLE"

    return subgroup


def get_medium_artifacts_by_subgroup(subgroup_name):
    if subgroup_name == "PROMO_HEAVY":
        return promo_deploy_artifacts
    return stable_deploy_artifacts


# ============================================================
# 6) SHORT RULE MODEL
# ============================================================
def short_rule_predict(row):
    lag1 = float(row.get("Lag1", 0) or 0)
    lag2 = float(row.get("Lag2", 0) or 0)
    rolling3 = float(row.get("Rolling3M_Mean", 0) or 0)
    sku_mean = float(row.get("SKU_Mean_Demand", 0) or 0)

    last_bonus_demand = float(row.get("Last_Bonus_Demand", 0) or 0)
    avg_bonus_uplift = float(row.get("Avg_Bonus_Uplift", 1.0) or 1.0)

    bonus_flag = int(row.get("Bonus_Flag", 0) or 0)
    expected_bonus = int(row.get("Expected_Bonus_NextMonth", 0) or 0)
    supply_flag = int(row.get("Supply_Constraint_Flag", 0) or 0)

    primary_stock = float(row.get("Available_Primary_Inventory_Qty", 0) or 0)
    distributor_stock = float(row.get("Distributor_Inventory_Qty", 0) or 0)

    short_type = row.get("Short_SKU_Type", "SHORT_NORMAL")
    hist_len = int(row.get("History_Length", 0) or 0)

    anchors = [x for x in [lag1, lag2, rolling3, sku_mean] if x > 0]

    if len(anchors) == 0:
        pred = 0.0
    elif hist_len <= 2:
        pred = float(np.mean(anchors))
    else:
        pred = float(np.median(anchors))

    if short_type == "SHORT_PROMO":
        if expected_bonus == 1 and last_bonus_demand > 0:
            pred = 0.60 * pred + 0.40 * last_bonus_demand
        elif bonus_flag == 1:
            pred = pred * min(max(avg_bonus_uplift, 1.0), 1.8)

    if supply_flag == 1:
        pred = min(pred, max(lag1, rolling3, 0))

    if (primary_stock + distributor_stock) <= 0:
        pred *= 0.85

    return max(pred, 0.0)


def prepare_short_inference_frame(raw_data):
    df = raw_data.copy().sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)
    df = add_history_length_from_subset(df, df)
    df = df[df["History_Segment"] == "SHORT"].copy()

    if df.empty:
        return df

    bonus_pattern_df = detect_recurring_bonus_skus(df)[[
        "ItemCode",
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ]].copy()

    df = df.drop(columns=[
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ], errors="ignore")

    df = df.merge(bonus_pattern_df, on="ItemCode", how="left")

    for c in ["Recurring_Bonus_SKU", "Bonus_Cycle_Length", "Avg_Bonus_Gap", "Bonus_Frequency_All"]:
        df[c] = df[c].fillna(0)

    df["Avg_Bonus_Uplift"] = df["Avg_Bonus_Uplift"].fillna(1.0)

    df, _ = apply_sku_cap(df.copy(), df.copy())

    sku_total = df.groupby("ItemCode")["Clean_Demand"].sum().sort_values(ascending=False)
    if sku_total.sum() > 0:
        cum_pct = sku_total.cumsum() / sku_total.sum()
        abc_series = pd.cut(cum_pct, bins=[0, 0.7, 0.9, 1.0], labels=[0, 1, 2])
        abc_map = abc_series.to_dict()
        df["ABC_Class"] = df["ItemCode"].map(abc_map).fillna(2)
    else:
        df["ABC_Class"] = 2

    promo_profile_df = build_promo_profile(df)
    df = merge_promo_profile(df, promo_profile_df)

    short_profile_df = build_short_sku_profile(df)
    df = merge_short_sku_profile(df, short_profile_df)

    df = add_bonus_cycle_features(df)
    df = rebuild_time_features(df)

    return df


def forecast_short_rule_sku(sku_code, raw_data):
    sku_code = str(sku_code)

    prepared_df = prepare_short_inference_frame(raw_data)
    if prepared_df.empty:
        return None

    sku_hist = prepared_df[prepared_df["ItemCode"].astype(str) == sku_code].copy()
    sku_hist = sku_hist.sort_values(["Year", "Month_Number"])

    if sku_hist.empty:
        return None

    last_row = sku_hist.iloc[-1:].copy()

    next_month = int(last_row["Month_Number"].iloc[0]) + 1
    next_year = int(last_row["Year"].iloc[0])
    if next_month > 12:
        next_month = 1
        next_year += 1

    next_bonus = infer_expected_bonus_flag_residual(sku_code, prepared_df)

    new_row = last_row.copy()
    new_row["Year"] = next_year
    new_row["Month_Number"] = next_month
    new_row["Bonus_Flag"] = int(next_bonus)

    future_df = pd.concat([prepared_df.copy(), new_row], ignore_index=True)
    future_df = future_df.sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)

    future_df = add_bonus_cycle_features(future_df)
    future_df = rebuild_time_features(future_df)
    future_df = recompute_target(future_df)

    next_row = future_df[
        (future_df["ItemCode"].astype(str) == sku_code) &
        (future_df["Year"] == next_year) &
        (future_df["Month_Number"] == next_month)
    ].copy()

    if next_row.empty:
        return None
    
    next_row["Segment_For_Calibration"] = "SHORT"

    row_dict = next_row.iloc[0].to_dict()
    pred = short_rule_predict(row_dict)
    pred = apply_production_calibration(row_dict, pred)

    short_type = next_row["Short_SKU_Type"].iloc[0] if "Short_SKU_Type" in next_row.columns else "SHORT_NORMAL"

    return {
        "ItemCode": int(float(sku_code)),
        "Forecast_Year": next_year,
        "Forecast_Month": next_month,
        "Expected_Bonus": int(next_bonus),
        "Residual_Baseline": np.nan,
        "Predicted_Residual": np.nan,
        "Forecast_Prediction": pred,
        "Segment": "SHORT",
        "Model_Type": "RULE_BASED",
        "Subgroup": short_type,
        "Status": "Success"
    }


# ============================================================
# 7) SINGLE LONG/MEDIUM XGB FORECAST
# ============================================================
def single_xgb_forecast(
    sku_code,
    next_month_bonus,
    artifacts,
    raw_data
):
    model = artifacts["model"]
    feature_cols = artifacts["feature_cols"]

    sku_code = str(sku_code)

    prepared_df = prepare_residual_forecast_frame(artifacts, raw_data)

    sku_hist = prepared_df[prepared_df["ItemCode_Original"] == sku_code].copy()
    sku_hist = sku_hist.sort_values(["Year", "Month_Number"]).reset_index(drop=True)

    if sku_hist.empty:
        return None

    last_row, next_year, next_month = get_next_period_from_history(prepared_df, sku_code)
    if last_row is None:
        return None

    new_row = last_row.copy()
    new_row["Year"] = next_year
    new_row["Month_Number"] = next_month
    new_row["Bonus_Flag"] = int(next_month_bonus)

    work_df = prepared_df.drop(columns=["ItemCode_Encoded"], errors="ignore").copy()
    work_df = pd.concat([work_df, new_row], ignore_index=True)

    bonus_pattern_df_f = detect_recurring_bonus_skus(work_df)[[
        "ItemCode",
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ]].copy()

    work_df = work_df.drop(columns=[
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ], errors="ignore")

    work_df = work_df.merge(bonus_pattern_df_f, on="ItemCode", how="left")

    for c in ["Recurring_Bonus_SKU", "Bonus_Cycle_Length", "Avg_Bonus_Gap", "Bonus_Frequency_All"]:
        work_df[c] = work_df[c].fillna(0)

    work_df["Avg_Bonus_Uplift"] = work_df["Avg_Bonus_Uplift"].fillna(1.0)
    work_df["ABC_Class"] = work_df["ItemCode"].map(artifacts["abc_map"]).fillna(2)

    work_df = merge_promo_profile(work_df, artifacts["promo_profile_df"])

    if artifacts.get("segment") == "MEDIUM" and "medium_profile_df" in artifacts:
        work_df = merge_medium_sku_profile(work_df, artifacts["medium_profile_df"])

    work_df = add_bonus_cycle_features(work_df)
    work_df = rebuild_time_features(work_df)
    work_df = apply_clip_caps(work_df, artifacts["clip_caps"])
    work_df = recompute_target(work_df)
    work_df = add_residual_target(work_df)

    cat_to_code = {str(k): i for i, k in enumerate(artifacts["itemcode_categories"])}
    unk_code = len(cat_to_code)

    work_df["ItemCode_Original"] = work_df["ItemCode"].astype(str)
    work_df["ItemCode_Encoded"] = work_df["ItemCode"].astype(str).map(cat_to_code).fillna(unk_code).astype(int)

    work_df_model = work_df.copy()
    work_df_model["ItemCode"] = work_df_model["ItemCode_Encoded"]

    next_row = work_df_model[
        (work_df_model["ItemCode_Original"] == sku_code) &
        (work_df_model["Year"] == next_year) &
        (work_df_model["Month_Number"] == next_month)
    ].copy()

    if next_row.empty:
        return None
    
    next_row["Segment_For_Calibration"] = artifacts.get("segment", "")


    assert_features_exist(next_row, feature_cols, where="NEXT_ROW_FORECAST")

    pred_residual = float(model.predict(sanitize(next_row[feature_cols]))[0])
    baseline = float(next_row[BASELINE_COL].iloc[0])
    raw_forecast = max(baseline + pred_residual, 0.0)

    row_dict = next_row.iloc[0].to_dict()

    forecast = apply_promo_aware_adjustment_residual(row_dict, raw_forecast)
    forecast = apply_final_forecast_guardrails_residual(row_dict, forecast)
    forecast = apply_production_calibration(row_dict, forecast)
    
    subgroup = None
    if artifacts.get("segment") == "MEDIUM":
        subgroup = next_row["Medium_Subgroup"].iloc[0] if "Medium_Subgroup" in next_row.columns else "STABLE"

    return {
        "ItemCode": int(float(sku_code)),
        "Forecast_Year": next_year,
        "Forecast_Month": next_month,
        "Expected_Bonus": int(next_month_bonus),
        "Residual_Baseline": baseline,
        "Predicted_Residual": pred_residual,
        "Forecast_Prediction": forecast,
        "Segment": artifacts.get("segment", "UNKNOWN"),
        "Model_Type": "XGBOOST",
        "Subgroup": subgroup if subgroup is not None else "",
        "Status": "Success"
    }


# ============================================================
# 8) UNIFIED SKU FORECAST ROUTER
# ============================================================
def forecast_one_sku(sku_code, raw_data):
    sku_code = str(sku_code)
    segment = choose_model_by_history(raw_data, sku_code)

    if segment == "LONG":
        artifacts = long_deploy_artifacts
        prepared_df = prepare_residual_forecast_frame(artifacts, raw_data)
        next_bonus = infer_expected_bonus_flag_residual(sku_code, prepared_df)
        return single_xgb_forecast(
            sku_code=sku_code,
            next_month_bonus=next_bonus,
            artifacts=artifacts,
            raw_data=raw_data
        )

    if segment == "MEDIUM":
        # decide subgroup first using stable artifact prep
        stable_prepared = prepare_residual_forecast_frame(stable_deploy_artifacts, raw_data)
        subgroup = choose_medium_subgroup_from_prepared(stable_prepared, sku_code)
        artifacts = get_medium_artifacts_by_subgroup(subgroup)

        prepared_df = prepare_residual_forecast_frame(artifacts, raw_data)
        next_bonus = infer_expected_bonus_flag_residual(sku_code, prepared_df)

        result = single_xgb_forecast(
            sku_code=sku_code,
            next_month_bonus=next_bonus,
            artifacts=artifacts,
            raw_data=raw_data
        )
        if result is not None:
            result["Subgroup"] = subgroup
        return result

    # SHORT
    return forecast_short_rule_sku(sku_code, raw_data)


# ============================================================
# 9) BULK FORECAST
# ============================================================
def bulk_segmented_forecast(raw_data, sku_list):
    results = []
    failed = []

    for sku in sku_list:
        try:
            result = forecast_one_sku(sku, raw_data)

            if result is None:
                failed.append({
                    "ItemCode": int(sku),
                    "Segment": choose_model_by_history(raw_data, sku),
                    "Model_Type": "UNKNOWN",
                    "Status": "Forecast failed"
                })
            else:
                results.append(result)

        except Exception as e:
            failed.append({
                "ItemCode": int(sku),
                "Segment": choose_model_by_history(raw_data, sku),
                "Model_Type": "UNKNOWN",
                "Status": f"Error: {str(e)}"
            })

    results_df = pd.DataFrame(results)
    failed_df = pd.DataFrame(failed)

    if not results_df.empty:
        results_df = results_df.sort_values(
            ["Forecast_Year", "Forecast_Month", "ItemCode"]
        ).reset_index(drop=True)

    if not failed_df.empty:
        failed_df = failed_df.sort_values(["ItemCode"]).reset_index(drop=True)

    return results_df, failed_df


# ============================================================
# 10) FINAL RUN FOR FEB FORECAST
# ============================================================
# Assumes PHARMA_SKUS and Cleaned_Base_Data already exist
# Assumes all training artifacts already exist in memory:
#   long_deploy_artifacts
#   promo_deploy_artifacts
#   stable_deploy_artifacts

sku_list = sorted(set(int(x) for x in PHARMA_SKUS))

segmented_predictions, segmented_failed_df = bulk_segmented_forecast(
    raw_data=Cleaned_Base_Data.copy(),
    sku_list=sku_list
)

if segmented_predictions.empty:
    raise ValueError("No predictions were generated.")

latest_forecast_year = segmented_predictions["Forecast_Year"].max()
latest_forecast_month = segmented_predictions.loc[
    segmented_predictions["Forecast_Year"] == latest_forecast_year,
    "Forecast_Month"
].max()

target_predictions = segmented_predictions[
    (segmented_predictions["Forecast_Year"] == latest_forecast_year) &
    (segmented_predictions["Forecast_Month"] == latest_forecast_month)
].copy()

output_file = "pharma_forecast_feb.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    target_predictions.to_excel(writer, sheet_name="Forecasts_Feb", index=False)

    if not segmented_failed_df.empty:
        segmented_failed_df.to_excel(writer, sheet_name="Failed_SKUs", index=False)

print(f"Saved file: {output_file}")
print(target_predictions.head())

if not segmented_failed_df.empty:
    print(segmented_failed_df.head())
else:
    print("No failed SKUs")

In [ ]:
# 18) LOAD LONG GRU DEPLOY ARTIFACTS
# ============================================================
def load_long_gru_deploy_artifacts():
    artifact = torch.load(os.path.join(GRU_DEPLOY_DIR, "gru_long_deploy_model.pt"), map_location=GRU_DEVICE)

    seq_scaler = joblib.load(os.path.join(GRU_DEPLOY_DIR, "gru_long_seq_scaler.pkl"))
    static_scaler = joblib.load(os.path.join(GRU_DEPLOY_DIR, "gru_long_static_scaler.pkl"))
    promo_profile_df = joblib.load(os.path.join(GRU_DEPLOY_DIR, "gru_long_promo_profile_df.pkl"))

    model = LongGRUResidualForecaster(
        num_items=len(artifact["item_to_idx"]),
        seq_input_dim=len(artifact["seq_features"]),
        static_input_dim=len(artifact["static_features"]),
        embed_dim=artifact["embed_dim"],
        hidden_size=artifact["hidden_size"],
        num_layers=artifact["num_layers"],
        dropout=artifact["dropout"],
    ).to(GRU_DEVICE)

    model.load_state_dict(artifact["model_state_dict"])
    model.eval()

    loaded = {
        "model": model,
        "model_type": artifact["model_type"],
        "segment": artifact["segment"],
        "seq_features": artifact["seq_features"],
        "static_features": artifact["static_features"],
        "seq_len": artifact["seq_len"],
        "embed_dim": artifact["embed_dim"],
        "hidden_size": artifact["hidden_size"],
        "num_layers": artifact["num_layers"],
        "dropout": artifact["dropout"],
        "item_to_idx": artifact["item_to_idx"],
        "abc_map": artifact["abc_map"],
        "clip_caps": artifact["clip_caps"],
        "promo_profile_df": promo_profile_df
    }

    scalers = LongGRUScalerBundle(
        seq_scaler=seq_scaler,
        static_scaler=static_scaler
    )

    return loaded, scalers


# ============================================================
# 19) LONG GRU SINGLE SKU INFERENCE
# ============================================================
def forecast_next_month_for_long_gru_sku(sku_code, raw_data, loaded_artifacts=None, loaded_scalers=None):
    sku_code = int(sku_code)

    if loaded_artifacts is None or loaded_scalers is None:
        loaded_artifacts, loaded_scalers = load_long_gru_deploy_artifacts()

    model = loaded_artifacts["model"]
    item_to_idx = loaded_artifacts["item_to_idx"]

    deploy_df, _, _, _ = prepare_long_frame_for_gru(raw_data, valid_df=None)
    if deploy_df.empty:
        return None

    sku_hist = deploy_df[deploy_df["ItemCode"] == sku_code].copy().sort_values(["Year", "Month_Number"]).reset_index(drop=True)
    if sku_hist.empty:
        return None

    last_row = sku_hist.iloc[-1].copy()
    next_year, next_month = gru_infer_next_year_month(last_row["Year"], last_row["Month_Number"])

    next_bonus = infer_expected_bonus_flag_residual(str(sku_code), deploy_df)

    new_row = last_row.copy()
    new_row["Year"] = next_year
    new_row["Month_Number"] = next_month
    new_row["Month"] = f"{next_year}-{str(next_month).zfill(2)}"
    new_row["Bonus_Flag"] = int(next_bonus)

    for c in ["Secondary_Sales_Qty", "Primary_Sales_Qty", "Free_Qty", "Clean_Demand"]:
        if c in new_row.index:
            new_row[c] = 0

    future_df = pd.concat([deploy_df.copy(), pd.DataFrame([new_row])], ignore_index=True)
    future_df = future_df.sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)

    future_df = add_bonus_cycle_features(future_df)
    future_df = rebuild_time_features(future_df)
    future_df = recompute_target(future_df)
    future_df = add_residual_target(future_df)

    future_df["Residual_Target_Log"] = gru_signed_log_transform(future_df[MODEL_TARGET_COL].fillna(0))

    g = future_df[future_df["ItemCode"] == sku_code].copy().sort_values(["Year", "Month_Number"]).reset_index(drop=True)

    target_idx = g[(g["Year"] == next_year) & (g["Month_Number"] == next_month)].index
    if len(target_idx) == 0:
        return None

    idx = target_idx[0]
    if idx < loaded_artifacts["seq_len"] - 1:
        return None

    seq_slice = g.iloc[idx - loaded_artifacts["seq_len"] + 1: idx + 1].copy()
    if len(seq_slice) != loaded_artifacts["seq_len"]:
        return None

    if sku_code not in item_to_idx:
        return None

    seq_vals = loaded_scalers.seq_scaler.transform(
        seq_slice[loaded_artifacts["seq_features"]].fillna(0).values
    )
    static_vals = loaded_scalers.static_scaler.transform(
        seq_slice.iloc[-1][loaded_artifacts["static_features"]].fillna(0).values.reshape(1, -1)
    )[0]

    x_seq = torch.tensor(seq_vals[np.newaxis, :, :], dtype=torch.float32).to(GRU_DEVICE)
    x_static = torch.tensor(static_vals[np.newaxis, :], dtype=torch.float32).to(GRU_DEVICE)
    x_item = torch.tensor([item_to_idx[sku_code]], dtype=torch.long).to(GRU_DEVICE)

    with torch.no_grad():
        pred_res_log = model(x_seq, x_static, x_item).cpu().numpy()[0]

    pred_residual = gru_signed_log_inverse(pred_res_log)
    target_row = seq_slice.iloc[-1]
    baseline = float(target_row[BASELINE_COL])
    pred = max(baseline + pred_residual, 0.0)

    return {
        "ItemCode": int(sku_code),
        "Forecast_Year": int(next_year),
        "Forecast_Month": int(next_month),
        "Expected_Bonus": int(target_row["Bonus_Flag"]),
        "Residual_Baseline": baseline,
        "Predicted_Residual": float(pred_residual),
        "Forecast_Prediction": float(pred),
        "Segment": "LONG",
        "Model_Type": "GRU",
        "Subgroup": "",
        "Status": "Success"
    }


# ============================================================
# 20) LONG GRU BULK INFERENCE
# ============================================================
def forecast_next_month_bulk_long_gru(raw_data, sku_list=None):
    loaded_artifacts, loaded_scalers = load_long_gru_deploy_artifacts()

    if sku_list is None:
        sku_list = sorted(raw_data["ItemCode"].astype(int).unique().tolist())

    rows = []
    failed = []

    for sku in sku_list:
        try:
            out = forecast_next_month_for_long_gru_sku(
                sku_code=int(sku),
                raw_data=raw_data,
                loaded_artifacts=loaded_artifacts,
                loaded_scalers=loaded_scalers
            )
            if out is not None:
                rows.append(out)
            else:
                failed.append({"ItemCode": int(sku), "Status": "Forecast failed"})
        except Exception as e:
            failed.append({"ItemCode": int(sku), "Status": str(e)})

    forecast_df = pd.DataFrame(rows)
    failed_df = pd.DataFrame(failed)

    if not forecast_df.empty:
        forecast_df = forecast_df.sort_values(
            ["Forecast_Year", "Forecast_Month", "ItemCode"]
        ).reset_index(drop=True)

    if not failed_df.empty:
        failed_df = failed_df.sort_values(["ItemCode"]).reset_index(drop=True)

    return forecast_df, failed_df


#

In [ ]:
import pandas as pd
import numpy as np

# === file paths ===
actual_file = "/Users/dhanujiamanda/Documents/Projects/Agentic AI /Doc/Company_Data.xlsx"
forecast_file = "/Users/dhanujiamanda/Documents/Projects/Agentic AI /Doc/pharma_forecast_feb.xlsx"
output_file = "/Users/dhanujiamanda/Documents/Projects/Agentic AI /Doc/forecast_vs_actual_analysis.xlsx"

# === load files ===
actual = pd.read_excel(actual_file, sheet_name="Sheet1")
forecast = pd.read_excel(forecast_file)

print("ACTUAL columns:", actual.columns.tolist())
print("FORECAST columns:", forecast.columns.tolist())

In [ ]:
# -------------------------------------------------
# Adjust these column names if needed
# -------------------------------------------------
actual_sku_col = "ItemCode"
actual_sales_col = "Secondary_Sales_Qty"

forecast_sku_col = "ItemCode"
forecast_fc_col = "Forecast_Prediction"

# optional month filter for actuals
# set these only if actual workbook contains multiple months
target_year = 2026
target_month = 2   # Feb actuals, if forecast file is pharma_forecast_feb

if "Year" in actual.columns and "Month_Number" in actual.columns:
    actual = actual[(actual["Year"] == target_year) & (actual["Month_Number"] == target_month)].copy()

# aggregate in case there are duplicate rows
actual_agg = (
    actual.groupby(actual_sku_col, as_index=False)[actual_sales_col]
    .sum()
    .rename(columns={
        actual_sku_col: "Code",
        actual_sales_col: "Sales"
    })
)

forecast_agg = (
    forecast.groupby(forecast_sku_col, as_index=False)[forecast_fc_col]
    .sum()
    .rename(columns={
        forecast_sku_col: "Code",
        forecast_fc_col: "FC"
    })
)

# merge
df = forecast_agg.merge(actual_agg, on="Code", how="outer")

df["FC"] = df["FC"].fillna(0)
df["Sales"] = df["Sales"].fillna(0)

# business-style ratio and accuracy
df["Ratio"] = np.where(df["Sales"] > 0, df["FC"] / df["Sales"], np.nan)
df["Accuracy"] = np.where(
    (df["Ratio"] > 0) & (df["Ratio"] < 2),
    np.maximum(1 - np.abs(1 - df["Ratio"]), 0),
    np.nan
)

# error diagnostics
df["Abs_Error"] = np.abs(df["FC"] - df["Sales"])
df["Error"] = df["FC"] - df["Sales"]

df["Band_80"] = np.where((df["Ratio"] >= 0.8) & (df["Ratio"] <= 1.2), 1, 0)
df["Band_70"] = np.where((df["Ratio"] >= 0.7) & (df["Ratio"] <= 1.3), 1, 0)

df["Error_Type"] = np.select(
    [
        df["Ratio"].isna(),
        df["Ratio"] > 1.3,
        df["Ratio"] < 0.7
    ],
    [
        "NO_ACTUAL",
        "OVERFORECAST",
        "UNDERFORECAST"
    ],
    default="GOOD"
)

df["Problem_Flag"] = np.where(
    (df["Ratio"] > 1.5) | (df["Ratio"] < 0.5) | (df["Sales"] == 0),
    "YES",
    "NO"
)

# sort worst first
analysis = df.sort_values(["Problem_Flag", "Abs_Error"], ascending=[False, False]).copy()

# summary
valid_acc = analysis["Accuracy"].dropna()
summary = pd.DataFrame({
    "Metric": [
        "SKU Count",
        "Forecast Sum",
        "Actual Sum",
        "Overall Accuracy",
        "In 80% Band",
        "In 70% Band",
        "Problematic SKU Count"
    ],
    "Value": [
        len(analysis),
        analysis["FC"].sum(),
        analysis["Sales"].sum(),
        valid_acc.mean() if len(valid_acc) > 0 else np.nan,
        analysis["Band_80"].mean(),
        analysis["Band_70"].mean(),
        (analysis["Problem_Flag"] == "YES").sum()
    ]
})

top_over = analysis[analysis["Error_Type"] == "OVERFORECAST"].head(30)
top_under = analysis[analysis["Error_Type"] == "UNDERFORECAST"].head(30)

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="Summary", index=False)
    analysis.to_excel(writer, sheet_name="Forecast_vs_Actual", index=False)
    top_over.to_excel(writer, sheet_name="Top_Overforecast", index=False)
    top_under.to_excel(writer, sheet_name="Top_Underforecast", index=False)

print(f"Saved: {output_file}")
print(summary)

In [ ]:
df = pd.read_excel(
    "/Users/dhanujiamanda/Documents/Projects/Agentic AI /Doc/forecast_vs_actual_analysis.xlsx",
    sheet_name="Forecast_vs_Actual"
)
# Contribution of each SKU
df["Contribution"] = df["Sales"] / df["Sales"].sum()

# Sort by importance
df = df.sort_values("Sales", ascending=False)

# cumulative contribution
df["Cum_Contribution"] = df["Contribution"].cumsum()

# classify
df["ABC_Class"] = pd.cut(
    df["Cum_Contribution"],
    bins=[0, 0.7, 0.9, 1.0],
    labels=["A", "B", "C"]
)

In [ ]:
df.groupby("ABC_Class")["Accuracy"].mean()